# События мониторинга ПКБ в софт-коллекшн: влияние и информационная ценность (IV)

**Вопрос.** Какие правила мониторинга ПКБ (Первое кредитное бюро) влияют на риск в софт-коллекшн (ранняя стадия
взыскания, просрочка 1–30 дней): в какую сторону, насколько сильно и можно ли это вообще оценить на наших данных?

**Главный ответ — в разделе 10 «Влияние правил ПКБ на риск: вывод по каждому правилу».** Там одна строка на каждое
правило ПКБ, найденное в данных: как часто бывает событие, во сколько раз риск выше или ниже при событии, сила влияния,
устойчивость во времени, качество данных и короткий «Вывод» (сильное / среднее / слабое влияние, нет влияния,
влияние не подтверждено, мало данных). Сила влияния учитывает и то, во сколько раз меняется риск клиента с событием,
и то, сколько правило даёт всему портфелю (IV). Там же — график и «Итог» простыми словами. В Excel это первый лист «Итог по правилам ПКБ».

Переменные договора (продукт, срок, сумма, прошлые просрочки клиента у нас и т. п., раздел 6) — это данные банка, а не
ПКБ. С признаками ПКБ они не сравниваются и в рейтинги признаков ПКБ не входят; они нужны только в дополнительном
разделе 15 (добавляют ли правила ПКБ точности к переменным договора), а в Excel лежат на последнем листе
«Переменные договора (справочно)».

**Входные данные** (те же объекты, что и в рабочем ноутбуке)

| объект | гранулярность | используемые столбцы |
|---|---|---|
| `ovrd_contr` | одна строка на договор и `act_date` с просрочкой (`ovd_days_cnt` > 0), объединённая с атрибутами договора | `contract_number`, `iin`, `act_date`, `ovd_days_cnt`, `process_ref`, `open_date`, `close_date`, `end_date`, `status_name`, `contract_amount`, `partner_name`, `product_id` |
| `hes` | одна строка на событие мониторинга ПКБ | `bin_iin`, `rule_name`, `old_value`, `new_value` (текст JSON), `created_date`, `event_date`, `internal_event_id` |
| `contracts` *(необязательно)* | одна строка на договор, выданный 2025-01-01 … 2026-10-15 | атрибуты договора; при наличии таблицы выполняется раздел раннего предупреждения (часть B) |

**Как определяется, чей договор.** В каждом событии ПКБ с договором есть номер этого договора (поле `contract_code`
в JSON). Ноутбук сравнивает его с номерами наших договоров (`contract_number` и `process_ref`):
* номер совпал → событие по **нашему** договору, группа «ПКБ: наши договоры»;
* номер есть, но не совпал ни с одним нашим → договор **другого банка или МФО**, группа «ПКБ: другие кредиторы»;
* номера в событии нет (онлайн-заявки, запросы кредитного отчёта, статусы «Лудоман», «Стоп-Кредит» и т. п.) → это
  событие о клиенте в целом; оно считается вместе с другими кредиторами («ПКБ: другие кредиторы»);
* два правила банк настроил в ПКБ сам: «частичная просрочка» (просрочка у нас, а другим кредиторам клиент платит) и
  «у кого до погашения не менее 80% от ОД займа — появился кредит в другом банке» (направление 80% стоит уточнить
  у владельца правила). Они по определению сравнивают нас с другими кредиторами, поэтому вынесены в отдельную группу
  «ПКБ: правила, настроенные банком».

**Внимание:** событие без номера договора может быть вызвано и нашим банком (например, наш запрос кредитного отчёта при взыскании или заявка клиента к нам). ПКБ не передаёт, кто запросил, поэтому такие события отнесены к другим кредиторам. Если событие «Запрос кредитного отчёта» сильно повышает риск, это стоит сверить с датами наших запросов в ПКБ.

Зачем это нужно: события по нашим договорам повторяют то, что банк и так знает (наш DPD, наши платежи). Поэтому
в вывод по правилам (раздел 10) и в модель «Переменные договора + ПКБ» (раздел 15) они не входят и используются только
для проверок и справки (сверка номеров договоров в разделе 7, справочная колонка в разделе 10, модель «Переменные
договора + ПКБ + события по нашим договорам» в разделе 15).

**Ограничение.** «Нашими» считаются только договоры из `ovrd_contr` и из `contracts` (если он загружен). В `ovrd_contr`
есть только договоры, которые выходили на просрочку, поэтому другие продукты банка, которых нет в этих таблицах, и (без
`contracts`) договоры без просрочки попадут в «ПКБ: другие кредиторы». Чтобы этого не было, загрузите в ядро таблицу со **всеми** номерами договоров банка по этим
клиентам (колонки `contract_number` и/или `process_ref`, любые продукты; номера лучше загружать как текст) и укажите
в `OWN_CODES_VAR` (первая ячейка кода) её имя в кавычках или саму таблицу. Раздел 3 добавит эти номера и напечатает,
сколько добавлено.

**Методика (на момент скоринга, без заглядывания в будущее)**
1. **Эпизоды просрочки** восстанавливаются по строкам с днями просрочки (DPD): `dlq_start = act_date − (ovd_days_cnt − 1)`.
   Это работает независимо от того, есть ли в `ovrd_contr` строка за каждый день просрочки или только за некоторые дни месяца.
2. **Популяция (выборка), часть A**: каждый выход договора в софт-коллекшн (скоринг выполняется в день, когда эпизод
   достигает `OBS_DPD`, по умолчанию DPD 1), если дата скоринга попадает в период покрытия мониторингом, а исход
   эпизода полностью наблюдаем.
3. **Целевая переменная (таргет)**: `bad = 1` («плохой»), если эпизод достигает DPD ≥ 31 (перекат из софт- в хард-коллекшн).
   Дополнительный таргет: договор не вышел из просрочки (нет самоизлечения) к DPD 16.
4. **Признаки**: события ПКБ, поступившие в банк **до** даты скоринга (`created_date` < даты скоринга),
   агрегированные за 7 / 14 / 30 дней по каждому правилу, плюс суммы и изменения, извлечённые из JSON. События по
   собственным договорам банка (`contract_code` совпадает с нашим `contract_number` или `process_ref`) вынесены в
   отдельную группу, потому что они повторяют информацию, которая у банка уже есть (см. «Как определяется, чей
   договор» выше).
5. **Анализ**: главный — вывод по каждому правилу ПКБ (раздел 10: риск при событии с доверительным интервалом,
   значимость с поправкой на число правил, IV лучшего признака правила, устойчивость, Джини правила и его вклад
   сверх остальных правил ПКБ). Дальше — подробности: лифт правил (раздел 11), WoE (вес свидетельства) / IV
   признаков ПКБ с монотонным разбиением на бины (интервалы), Джини (Gini) и KS (раздел 12), понедельная стабильность
   и PSI (раздел 13), фильтр по корреляции (раздел 14) и, дополнительно, прирост Джини к модели только на переменных
   договора на выборке вне времени (out-of-time, OOT) с доверительным интервалом (ДИ), полученным бутстрэпом (раздел 15).
6. **Часть B (необязательно)**: те же признаки для договоров, которые пока обслуживаются без просрочки; прогнозируется
   выход на просрочку в течение 30 дней — раннее предупреждение (EWS) до начала софт-коллекшн.

**Подписи на русском языке.** Все таблицы показываются и выгружаются в Excel с русскими названиями колонок и
значений (словари `RU_COLUMNS` и `RU_VALUES` и функция `ru_labels` в первой ячейке кода). Внутри кода имена колонок
и коды (например, `pkb_other`, `iv`, `bad_rate`) остаются английскими, поэтому логика расчёта не меняется.
Признаки и правила ПКБ везде называются по-русски. На графиках — только короткие русские названия (например,
«Макс. DPD, др. кредиторы, 30 дн.»), а под каждым графиком с признаками или правилами — таблица «Расшифровка
обозначений на графике»: название на графике, полное описание и код в данных. В таблицах и в Excel первой идёт короткая
русская подпись («Признак» / «Правило»), за ней «Описание признака» / «Описание правила», а внутренний код
(например, `dpd_oth_max_30d`) — в последней колонке «Код в данных»; по нему признак находится в коде и в данных
(функции `code_name`, `feature_desc` и `ru_labels` в первой ячейке кода). Как устроены названия и коды — в разделе 8
(«Как читать названия признаков»). Числа везде записаны одинаково: разряды — пробелом, дробная часть — точкой
(«6 624», «0.2155»); в таблицах на экране «да» / «нет» вместо True / False и «—» вместо пропуска, номера ИИН и
договоров в предпросмотре скрыты (видны последние 4 знака).

**Как запустить на реальных данных**: выполните эти ячейки в ядре, где уже загружены `ovrd_contr` и `hes` (и `contracts`,
если он у вас есть), или вставьте эти ячейки в рабочий ноутбук. По умолчанию `DEMO_MODE = False`; имена ваших датафреймов
задаются в `OVRD_VAR`, `HES_VAR`, `CONTRACTS_VAR` (и, по желанию, `OWN_CODES_VAR` — все номера договоров банка). Раздел 2b проверяет датафреймы до любого тяжёлого шага и
останавливается с понятным сообщением, если что-то не подходит.
При `DEMO_MODE = True` ноутбук строит синтетический набор данных с теми же столбцами и структурой JSON, чтобы каждую
ячейку можно было проверить от начала до конца. **Результаты на синтетических демо-данных ничего не говорят о реальном портфеле.**

In [ ]:
import os, re, gc, json, math, time, uuid, warnings, textwrap
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

try:                                   # если установлен orjson, разбор JSON в 3–5 раз быстрее
    import orjson
    _json_loads = orjson.loads
except ImportError:
    _json_loads = json.loads

try:
    display                            # noqa: F821  (функция display есть внутри Jupyter)
except NameError:
    display = print                    # вне Jupyter таблицы просто печатаются

# ------------------------------------------------------------------ русские подписи таблиц (слой отображения)
# Внутренние имена колонок и коды значений в коде остаются английскими: на них опирается логика расчёта.
# Переводятся только подписи при показе таблиц (display) и при выгрузке в Excel/CSV: ru_labels() возвращает
# переименованную КОПИЮ таблицы, числа и исходные объекты не меняются. В таблицах с кодами признаков или правил первой
# идёт короткая русская подпись («Признак» / «Правило», функция code_name ниже), за ней полное описание
# («Описание признака» / «Описание правила»), а сам код переносится в последнюю колонку «Код в данных».
# На графиках — только короткие русские подписи; под каждым графиком с признаками или правилами показывается таблица
# «Расшифровка обозначений на графике» (show_chart_legend): подпись, полное описание и код в данных.

# имена колонок и индексов -> русские подписи
RU_COLUMNS = {
    # исходные таблицы (просмотр демо-данных)
    'rep_month': 'Отчётный месяц', 'act_date': 'Дата среза', 'contract_number': 'Номер договора',
    'ovd_days_cnt': 'Дни просрочки (DPD)', 'process_ref': 'Референс процесса', 'iin': 'ИИН',
    'open_date': 'Дата открытия', 'close_date': 'Дата закрытия', 'end_date': 'Плановая дата окончания',
    'status_name': 'Статус договора', 'contract_amount': 'Сумма договора', 'partner_name': 'Партнёр',
    'product_id': 'Код продукта', 'date_value': 'Дата значения', 'date_change': 'Дата изменения',
    'internal_event_id': 'ID события', 'bin_iin': 'БИН/ИИН', 'rule_name': 'Название правила',
    'old_value': 'Старое значение (JSON)', 'new_value': 'Новое значение (JSON)', 'event_date': 'Дата события',
    'created_date': 'Дата поступления в банк',
    # предварительная проверка
    'object': 'Объект', 'check': 'Проверка', 'status': 'Статус', 'detail': 'Подробности',
    # атрибуты договоров
    'contract': 'Договор', 'amount': 'Сумма', 'product': 'Продукт', 'channel': 'Канал', 'closed': 'Закрыт',
    'from_contracts': 'Из таблицы contracts',
    # качество данных, объёмы событий, когорты, сопоставление с нашими договорами
    # 'description' вне таблиц признаков встречается только в таблицах правил (описание правила ПКБ)
    'value': 'Значение', 'description': 'Описание правила', 'rule': 'Правило', 'count': 'Число событий в выборке',
    '50%': 'Медиана, ч', '90%': '90-й перцентиль, ч', '99%': '99-й перцентиль, ч',
    'week': 'Неделя скоринга (начало)', 'episodes': 'Число эпизодов', 'bads': 'Число «плохих»',
    'bad_rate': 'Доля «плохих»', 'events': 'Число событий',
    'share_with_contract_code': 'Доля событий с кодом договора', 'share_own_contract': 'Доля событий по нашим договорам',
    # каталог признаков, бины, IV
    'feature': 'Признак', 'feature_description': 'Описание признака', 'group': 'Группа признаков', 'family': 'Семейство правил',
    'window': 'Окно наблюдения, дн.', 'bin': 'Бин (интервал)', 'bin_chart': 'Бин на графике', 'n': 'Число наблюдений',
    'share': 'Доля бина',
    'woe': 'WoE', 'iv': 'IV', 'strength': 'Сила признака по IV', 'gini': 'Джини', 'ks': 'KS',
    'n_bins': 'Число бинов', 'share_nonzero': 'Доля ненулевых значений', 'share_missing': 'Доля пропусков',
    'bads_outside_main_bin': 'Число «плохих» вне основного бина', 'reliable': 'Надёжен',
    'direction': 'Направление зависимости',
    # влияние правил (лифт)
    'rule_key': 'Код правила', 'share_flagged': 'Доля наблюдений с событием', 'n_flagged': 'Число наблюдений с событием',
    'bads_flagged': 'Число «плохих» с событием', 'bad_rate_flagged': 'Доля «плохих» с событием',
    'bad_rate_not': 'Доля «плохих» без события', 'lift': 'Лифт', 'iv_flag': 'IV флага события',
    # понедельная стабильность
    'iv_min': 'IV мин. по неделям', 'iv_max': 'IV макс. по неделям', 'iv_cv': 'Коэф. вариации IV',
    'gini_min_week': 'Джини мин. по неделям', 'gini_max_week': 'Джини макс. по неделям', 'psi_max': 'PSI макс.',
    'stable': 'Стабилен', 'iv_all_episodes': 'IV (все эпизоды)',
    'iv_full_30d_history': 'IV (окно наблюдения заполнено полностью)',
    # прирост качества
    'model': 'Модель', 'n_features': 'Число признаков', 'gini_train': 'Джини, обучающая выборка',
    'gini_test': 'Джини, тестовая выборка (OOT)', 'capture_top10%': 'Доля выявленных «плохих» в топ-10%',
    'capture_top20%': 'Доля выявленных «плохих» в топ-20%', 'capture_top30%': 'Доля выявленных «плохих» в топ-30%',
    'delta_gini_vs_A': 'Прирост Джини к модели «Только переменные договора»', 'delta_CI95': '95% ДИ прироста Джини',
    'gini_test_boosting': 'Джини бустинга, тестовая выборка', 'delta_boosting_CI95': '95% ДИ прироста Джини (бустинг)',
    'coef': 'Коэффициент', 'sign_ok': 'Знак ожидаемый (отрицательный)',
}

# коды значений (группы, семейства, сила IV, подписи бинов, статусы, названия моделей) -> русские подписи
RU_VALUES = {
    # группы признаков
    # как определяется группа — раздел 7 («Как определяется, чей договор»)
    'pkb_other': 'ПКБ: другие кредиторы', 'pkb_bank_rule': 'ПКБ: правила, настроенные банком',
    'pkb_own': 'ПКБ: наши договоры', 'baseline': 'Переменные договора',
    # семейства правил и служебные значения каталога признаков
    'credit_hunger': 'кредитный голод', 'relief': 'облегчение условий (пролонгация, реструктуризация)',
    'delinquency': 'просрочка', 'positive': 'позитивные события', 'status': 'особые статусы клиента',
    'technical': 'технические события', 'debt_burden': 'долговая нагрузка',
    'bank_rule': 'правила, настроенные банком', 'other': 'прочие', 'unknown': 'неизвестные правила',
    'all': 'все', 'family': 'все правила семейства',
    # сила признака по IV и направление зависимости риска от значения
    'not useful': 'бесполезный', 'weak': 'слабый', 'medium': 'средний', 'strong': 'сильный',
    'suspicious: check leakage': 'подозрительно высокая IV: проверить утечку данных',
    'risk up with value': 'риск растёт при увеличении значения', 'risk down with value': 'риск снижается при увеличении значения',
    # подписи бинов и значения канала продаж
    'MISSING': 'пропуск', 'OTHER': 'прочие значения',
    'direct': 'прямой', 'merchant': 'через партнёра (мерчант)', 'marketplace': 'маркетплейс', 'NA': 'нет данных',
    # статусы предварительной проверки (OK не переводится)
    'WARN': 'ПРЕДУПРЕЖДЕНИЕ', 'FAIL': 'ОШИБКА',
    # модели в таблице прироста качества
    'A: baseline': 'Только переменные договора',
    'B: + PKB other lenders': 'Переменные договора + ПКБ (другие кредиторы и правила, настроенные банком)',
    'C: + PKB own contracts': 'Переменные договора + ПКБ + события по нашим договорам',
}
# значения, перевод которых зависит от колонки (имя колонки или оси -> {код: подпись}); остальные — по RU_VALUES
RU_VALUES_BY_COL = {
    'window': {'all': 'вся история'},                                  # признаки days_since_*: за всю историю
    'family': {'all': 'все правила (сводные счётчики)'},               # all_oth_n, all_oth_nrules, days_since_any_oth
    'rule': {'all': 'все правила', 'family': 'все правила семейства', 'status': 'статусные правила'},
}

# ------------------------------------------------------------------ русские описания признаков (только для показа)
# Переменные договора bl_* из раздела 6 (build_baseline; в коде — группа 'baseline'): данные банка, не ПКБ. «Договор» (bl_c_*) — этот договор, «Клиент» (bl_i_*) —
# все наши договоры клиента по ИИН. «Прошлые эпизоды» — эпизоды просрочки, начавшиеся раньше даты скоринга, кроме
# текущего; их DPD берётся на день до даты скоринга. Для части B дата скоринга — дата среза договора.
BASELINE_DESC = {
    'bl_product': 'Продукт договора (код продукта product_id; NA или nan — нет данных)',
    'bl_channel': 'Канал продаж договора: прямой, мерчант (партнёр с БИН) или имя партнёра; NA — нет данных',
    'bl_amount': 'Сумма договора (contract_amount), тенге',
    'bl_mob_days': 'Дней от открытия договора до даты скоринга',
    'bl_term_days': 'Плановый срок договора: дней от открытия до плановой даты окончания (end_date)',
    'bl_term_elapsed': 'Доля прошедшего срока договора на дату скоринга: дней с открытия / плановый срок договора',
    'bl_c_n_prev_ep': 'Договор: число прошлых эпизодов просрочки, шт.',
    'bl_c_n_prev_ep_180d': 'Договор: число прошлых эпизодов просрочки, начавшихся за последние 180 дней до даты скоринга, шт.',
    'bl_c_max_prev_dpd': 'Договор: худший DPD прошлых эпизодов на день до даты скоринга, дни (0 — не было)',
    'bl_c_days_since_prev': 'Договор: дней с последнего дня просрочки прошлых эпизодов (пусто — не было)',
    'bl_i_n_prev_ep': 'Клиент: число прошлых эпизодов просрочки по всем нашим договорам клиента, шт.',
    'bl_i_n_prev_ep_180d': 'Клиент: число прошлых эпизодов просрочки, начавшихся за последние 180 дней до даты скоринга, шт.',
    'bl_i_max_prev_dpd': 'Клиент: худший DPD прошлых эпизодов на день до даты скоринга, дни (0 — не было)',
    'bl_i_days_since_prev': 'Клиент: дней с последнего дня просрочки прошлых эпизодов (1 — ещё идёт; пусто — не было)',
    'bl_i_n_dlq_now': 'Клиент: число других эпизодов просрочки (кроме текущего), ещё продолжавшихся накануне даты скоринга, шт.',
    'bl_i_n_prev_roll': 'Клиент: число прошлых эпизодов просрочки, дошедших до DPD 31+ к дню накануне даты скоринга, шт.',
    'bl_i_n_open_contracts': 'Клиент: число наших договоров, открытых на дату скоринга, включая текущий '
                             '(без таблицы contracts — только с просрочкой до этой даты), шт.',
    'bl_i_open_amount': 'Клиент: сумма (contract_amount) наших договоров, открытых на дату скоринга '
                        '(те же договоры, что в признаке «Клиент: число наших открытых договоров»), тенге',
}

# Признаки ПКБ из раздела 8 (build_pkb_features): те же формулировки, что в каталоге, без окна «за последние W дн.».
# Нужны, только если признака нет в каталоге (catalog, catalog_B); к ним добавляется окно из суффикса _<W>d.
_PKB_BODY_DESC = {
    'all_oth_n': 'Число событий у других кредиторов по всем правилам, кроме правил, настроенных банком',
    'all_oth_nrules': 'Число различных сработавших правил у других кредиторов (без правил, настроенных банком)',
    'dpd_oth_max': 'Худшее (максимальное) новое значение DPD у других кредиторов',
    'dpd_oth_n_up': 'Число увеличений DPD у других кредиторов',
    'dpd_oth_n_ge30': 'Число событий изменения DPD у других кредиторов с новым значением DPD ≥ 30',
    'ovdsum_oth_max': 'Максимальная сумма просрочки у других кредиторов',
    'ovdsum_oth_n_up': 'Число увеличений суммы просрочки у других кредиторов',
    'ovdsum_oth_up_sum': 'Рост суммы просрочки у других кредиторов: сумма всех увеличений (уменьшения не вычитаются)',
    'debt_oth_n_up': 'Число увеличений основного долга у других кредиторов',
    'debt_oth_n_down': 'Число уменьшений основного долга у других кредиторов',
    'debt_oth_up_sum': 'Рост основного долга у других кредиторов: сумма всех увеличений (уменьшения не вычитаются)',
    'debt_oth_net': 'Чистое изменение основного долга у других кредиторов (увеличения минус уменьшения)',
    'inst_net': 'Чистое изменение общего ежемесячного платежа у других кредиторов (увеличения минус уменьшения)',
    'inst_n_up': 'Число увеличений общего ежемесячного платежа у других кредиторов',
    'app_amt_sum': 'Общая сумма онлайн-заявок на кредит',
    'app_amt_max': 'Максимальная сумма онлайн-заявки на кредит',
    'app_n_mfo': 'Число онлайн-заявок в МФО',
    'inq_n_mfo': 'Число запросов кредитного отчёта со стороны МФО',
    'newloan_n_mfo': 'Число событий о новых займах в МФО (правила «новый заём», «новый договор», «заём на рефинансирование»)',
    'newloan_amt_sum': 'Сумма новых займов у других кредиторов (только события, где ПКБ передаёт сумму займа summa_zaima)',
    'prolong_days_sum': 'Суммарное число дней пролонгации у других кредиторов',
    'status_on_n': 'Число событий установки статуса «Лудоман», «Призывник», «Стоп-Кредит» или «Банкротство» (новое значение флага = 1)',
    'dpd_own_max': 'Худшее (максимальное) значение DPD по всем нашим договорам клиента, переданное в ПКБ',
}
# Части имени для признаков, которых нет ни в каталоге, ни в _PKB_BODY_DESC: <объект>[_oth|_own]_<мера>_<W>d
_RU_OBJECT = {'dpd': 'DPD (дни просрочки)', 'ovdsum': 'сумма просрочки', 'debt': 'основной долг',
              'inst': 'общий ежемесячный платёж', 'app': 'онлайн-заявки на кредит', 'inq': 'запросы кредитного отчёта',
              'newloan': 'новые займы', 'prolong': 'пролонгации', 'status_on': 'установленные статусы'}
_RU_MEASURE = {'max': 'Максимальное значение', 'n_up': 'Число увеличений', 'n_down': 'Число уменьшений',
               'up_sum': 'Сумма всех увеличений (уменьшения не вычитаются)', 'net': 'Чистое изменение (увеличения минус уменьшения)', 'sum': 'Сумма',
               'n_ge30': 'Число событий с новым значением ≥ 30', 'n_mfo': 'Число событий от МФО', 'n': 'Число событий',
               'amt_sum': 'Общая сумма', 'amt_max': 'Максимальная сумма', 'days_sum': 'Суммарное число дней'}
_RU_SCOPE = {'_oth': ' у других кредиторов', '_own': ' по нашим договорам', '': ''}


def _rule_key_desc(rk):
    """Описание кода правила с необязательным суффиксом _oth/_own (как rkey_desc в разделе 8), по RULE_DESC из раздела 1.
    Код other_<...> (правило, которого нет в RULES) без записи в RULE_DESC описывается по самому коду."""
    rule_desc = globals().get('RULE_DESC') or {}
    base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
    tail = ' (другие кредиторы)' if rk.endswith('_oth') else ' (наши договоры)' if rk.endswith('_own') else ''
    if rule_desc.get(base):
        return f'{rule_desc[base]}{tail}'
    if base.startswith('other_') and len(base) > 6:
        return f'правило ПКБ «{base[6:].replace("_", " ")}» (нет в справочнике RULES){tail}'
    return f'правило «{base}»{tail}' if base else ''


def _parse_feature_code(code):
    """Русское описание по грамматике имён признаков (раздел 8 и BASELINE_DESC); '' — код не подходит под грамматику."""
    if not isinstance(code, str):
        return ''
    if code.startswith('bl_') and len(code) > 3:
        return f'Переменная договора «{code[3:]}»: описание в BASELINE_DESC не задано'
    if code == 'days_since_any_oth':
        return ('Дней с последнего события у других кредиторов по любому правилу, кроме правил, настроенных банком, '
                'за всю историю (пусто — не было)')
    if code.startswith('days_since_') and len(code) > 11:
        return f'Дней с последнего события: {_rule_key_desc(code[11:])}, за всю историю (пусто — не было)'
    m = re.fullmatch(r'(.+)_(\d+)d', code)
    if not m:
        return ''
    body, window = m.group(1), f', за последние {m.group(2)} дн.'
    if body in _PKB_BODY_DESC:
        return _PKB_BODY_DESC[body] + window
    m = re.fullmatch(r'fam_(.+)_(oth|own)_n', body)
    if m:
        who = 'кроме событий по нашим договорам' if m.group(2) == 'oth' else 'наши договоры'
        return f'Число событий семейства «{RU_VALUES.get(m.group(1), m.group(1))}» ({who}){window}'
    if body.startswith('n_') and len(body) > 2:
        return f'Число событий: {_rule_key_desc(body[2:])}{window}'
    m = re.fullmatch(r'(dpd|ovdsum|debt|inst|app|inq|newloan|prolong|status_on)(_oth|_own)?_(.+)', body)
    if m and m.group(3) in _RU_MEASURE:
        return f'{_RU_MEASURE[m.group(3)]}: {_RU_OBJECT[m.group(1)]}{_RU_SCOPE[m.group(2) or ""]}{window}'
    return ''


_CATALOG_NAMES = ('catalog', 'catalog_B', 'catalog_main')      # каталоги признаков ПКБ (популяции A и B)


def feature_desc(code):
    """Русское описание признака по его коду: каталоги признаков ПКБ (catalog, catalog_B), затем BASELINE_DESC,
    затем разбор имени по грамматике (_parse_feature_code). '' — если код неизвестен."""
    if not isinstance(code, str) or not code:
        return ''
    for name in _CATALOG_NAMES:
        cat = globals().get(name)
        if isinstance(cat, pd.DataFrame) and 'description' in cat.columns and code in cat.index:
            d = cat.loc[code, 'description']
            d = d.iloc[0] if isinstance(d, pd.Series) else d
            if isinstance(d, str) and d.strip():
                return d
    return BASELINE_DESC.get(code) or _parse_feature_code(code)


def _merged_rule_names():
    """{код other_<...>: [названия правил ПКБ]} для кодов, под которыми в RULE_MAP (разделы 5 и 7) оказались несколько
    разных названий: map_rule_names обрезает код до 30 символов, и такие правила считаются как одно."""
    by = {}
    for nm, c in (globals().get('RULE_MAP') or {}).items():
        if isinstance(c, str) and c.startswith('other_'):
            by.setdefault(c, []).append(str(nm))
    return {c: v for c, v in by.items() if len(set(v)) > 1}


def _merged_note(code):
    # пометка к описанию кода, в который слились несколько правил ПКБ с похожим названием ('' — таких нет)
    if not isinstance(code, str) or 'other_' not in code:
        return ''
    for c, names in _merged_rule_names().items():
        if re.fullmatch(r'(?:n_|days_since_)?' + re.escape(c) + r'(?:_oth|_own)?(?:_\d+d)?', code):
            k = len(set(names)) - 1
            return (f' [и ещё {k} {_plural_ru(k, ("правило", "правила", "правил"))} ПКБ с похожим названием, '
                    'посчитаны вместе с этим]')
    return ''


def _plural_ru(n, forms):
    # forms = ('правило', 'правила', 'правил'): форма слова после числа n
    n = abs(int(n))
    if n % 10 == 1 and n % 100 != 11:
        return forms[0]
    if 2 <= n % 10 <= 4 and not 12 <= n % 100 <= 14:
        return forms[1]
    return forms[2]


def code_desc(code):
    # полное русское описание кода: признак (feature_desc), иначе правило (_rule_key_desc); '' — код неизвестен
    if not isinstance(code, str) or not code:
        return ''
    d = feature_desc(code) or (_rule_key_desc(code) if _is_rule_code(code) else '')
    return d + _merged_note(code) if d and _merged_note(code) not in d else d


def _are_feature_codes(values, min_share=0.8):
    # True, если все значения — текст и не меньше min_share различных значений — коды признаков
    # (признаки из каталогов и BASELINE_DESC или имена по грамматике); правила, бины, даты, модели сюда не подходят
    vals = list(pd.unique(pd.Series(list(values), dtype=object).dropna()))
    if not vals or not all(isinstance(v, str) for v in vals):
        return False
    known = set(BASELINE_DESC)
    for name in _CATALOG_NAMES:
        cat = globals().get(name)
        if isinstance(cat, pd.DataFrame):
            known.update(cat.index)
    return sum(v in known or _parse_feature_code(v) != '' for v in vals) >= min_share * len(vals)


def _is_feature_index(idx):
    # индекс из кодов признаков: один уровень, имя None или 'feature', текстовые метки
    return idx.nlevels == 1 and idx.name in (None, 'feature') and _is_text(idx.dtype) and _are_feature_codes(idx)


def _is_rule_code(v):
    # код правила (ключ rule_key, индекс таблиц по правилам): правило из RULES или other_<...> из RULE_DESC,
    # с необязательным суффиксом _oth / _own
    if not isinstance(v, str) or not v:
        return False
    base = v[:-4] if v.endswith(('_oth', '_own')) else v
    return base in RULE_SHORT or base in (globals().get('RULE_DESC') or {})


def _are_rule_codes(values, min_share=0.8):
    # True, если все значения — текст и не меньше min_share различных значений — коды правил
    vals = list(pd.unique(pd.Series(list(values), dtype=object).dropna()))
    if not vals or not all(isinstance(v, str) for v in vals):
        return False
    return sum(_is_rule_code(v) for v in vals) >= min_share * len(vals)


CODE_COL = 'Код в данных'                # последняя колонка таблиц с кодами: код признака или правила, как в данных
RULE_CODE_COL = 'Код правила в данных'   # код правила из колонки 'rule' таблиц признаков (каталог, таблицы IV)


def _code_layout(df):
    """Где в таблице коды: ('feature' или 'rule', номер колонки с кодами или None, если коды в индексе, список кодов).
    Признаки — индекс или колонка 'feature'; правила — индекс (имя None или 'rule') или колонка 'rule_key'.
    None — в таблице нет кодов признаков и правил."""
    if df.columns.nlevels != 1:
        return None
    cols, idx = list(df.columns), df.index
    if _is_feature_index(idx):
        return 'feature', None, list(idx)
    if cols.count('feature') == 1 and _are_feature_codes(df['feature']):
        return 'feature', cols.index('feature'), list(df['feature'])
    if idx.nlevels == 1 and idx.name in (None, 'rule') and _is_text(idx.dtype) and _are_rule_codes(idx):
        return 'rule', None, list(idx)
    if cols.count('rule_key') == 1 and _are_rule_codes(df['rule_key']):
        return 'rule', cols.index('rule_key'), list(df['rule_key'])
    return None


def _ru_value(v, col=None):
    # одно значение: перевод из RU_VALUES_BY_COL[col] (если для этой колонки задан свой перевод), затем из RU_VALUES;
    # числа, даты, bool и прочее не меняются
    if not isinstance(v, str):
        return v
    return RU_VALUES_BY_COL.get(col, {}).get(v, RU_VALUES.get(v, v))


def _ru_column(c):
    # имя колонки или индекса: RU_COLUMNS, затем имена с параметрами (недели, целевые переменные), затем RU_VALUES
    if not isinstance(c, str):
        return c
    if c in RU_COLUMNS:
        return RU_COLUMNS[c]
    m = re.fullmatch(r'iv_(\d\d\.\d\d)', c)                     # понедельная IV: iv_<дд.мм>
    if m:
        return f'IV, неделя с {m.group(1)}'
    if c == f'iv_roll{BAD_DPD}':                                # IV по основной целевой переменной
        return f'IV: перекат в DPD {BAD_DPD}+ (основной таргет)'
    for name, thr in EXTRA_TARGETS.items():                     # дополнительные целевые переменные
        if c == f'iv_{name}':
            return f'IV: DPD ≥ {thr} (дополнительный таргет)'
        if c == f'{name}_rate':
            return f'Доля эпизодов с DPD ≥ {thr}'
    return RU_VALUES.get(c, c)


def _is_text(dtype):
    # текстовые колонки: object, string/str и category (числа, bool и даты не трогаются)
    return (dtype == object or isinstance(dtype, pd.CategoricalDtype)
            or (pd.api.types.is_string_dtype(dtype) and not pd.api.types.is_bool_dtype(dtype)))


def _ru_index(idx, values):
    # Index: имена уровней переводятся как имена колонок; метки — как значения (values=True, с учётом имени оси) или как
    # имена колонок
    if _is_text(idx.dtype):
        idx = idx.map((lambda v: _ru_value(v, idx.name)) if values else _ru_column)
    return idx.set_names([_ru_column(nm) for nm in idx.names])


def ru_labels(obj):
    """Копия таблицы pandas с русскими подписями: имена колонок и индексов — по RU_COLUMNS, текстовые значения и
    метки индекса — по RU_VALUES. Если в индексе или в колонке ('feature', 'rule_key') коды признаков или правил:
    на месте кода — короткая русская подпись («Признак» / «Правило», code_names), сразу за ней — полное описание
    («Описание признака» / «Описание правила»; готовая колонка 'description' переносится туда же, пустые описания
    дополняются), коды — в последней колонке «Код в данных». Колонка 'rule' таблиц признаков (каталог, IV) показывает
    русское название правила, а его код — в колонке «Код правила в данных» перед «Кодом в данных».
    Числовые данные не меняются, порядок строк и исходный объект не изменяются; объекты других типов возвращаются как есть."""
    if isinstance(obj, pd.Series):
        out = obj.astype(object).map(_ru_value) if _is_text(obj.dtype) else obj.copy()
        out.index = _ru_index(obj.index, values=True)
        out.name = _ru_column(obj.name)
        return out
    if not isinstance(obj, pd.DataFrame):
        return obj
    lay = _code_layout(obj)
    out = obj.copy()
    cols, out.columns = obj.columns, range(obj.shape[1])         # временно номера колонок (имена могут повторяться)
    for j in range(obj.shape[1]):
        if _is_text(out[j].dtype):
            out[j] = out[j].astype(object).map(lambda v, c=cols[j]: _ru_value(v, c))
    # колонка 'bin' (таблицы бинов): границы в русской записи, пропуск и единый бин — по смыслу признака (bin_exact)
    if list(cols).count('bin') == 1:
        bj = list(cols).index('bin')
        fc = obj['feature'] if list(cols).count('feature') == 1 else pd.Series([None] * len(obj), index=obj.index)
        has_miss = set(fc[obj.iloc[:, bj].astype(object).eq('MISSING').values]) if len(obj) else set()
        out[bj] = [bin_exact(f, b, f in has_miss) for f, b in zip(fc, obj.iloc[:, bj])]
    # если у оси колонок есть имя (crosstab, unstack), её метки — это значения (например, сила IV), иначе — имена колонок
    heads = _ru_index(cols, values=cols.name is not None)
    out.index = _ru_index(obj.index, values=True)
    if lay is None:
        out.columns = heads
        return out
    kind, pos, codes = lay
    heads, cols = list(heads), list(cols)
    name_col = RU_COLUMNS['feature'] if kind == 'feature' else RU_COLUMNS['rule']   # «Признак» / «Правило»
    # 1) короткая русская подпись на месте кода (в индексе или в колонке кодов)
    if pos is None:
        out.index = pd.Index(code_names(codes), name=name_col)
    else:
        out[pos] = code_names(codes)
        heads[pos] = name_col
    # 2) колонка 'rule' таблицы признаков: русское название правила вместо кода
    rj = cols.index('rule') if kind == 'feature' and cols.count('rule') == 1 else None
    if rj is not None:
        out[rj] = [code_name(v) if _is_rule_code(v) else _ru_value(v) for v in obj.iloc[:, rj]]
    # 3) полное описание сразу после подписи
    old = cols.index('description') if cols.count('description') == 1 else None
    desc = [code_desc(c) for c in codes]
    if old is not None:
        desc = [(p + _merged_note(c) if _merged_note(c) not in p else p) if isinstance(p, str) and p.strip() else d
                for p, d, c in zip(obj.iloc[:, old], desc, codes)]
    keep = [k for k in range(len(cols)) if k != old]
    out = out[keep]
    out.columns = pd.Index([heads[k] for k in keep])
    at = 0 if pos is None else keep.index(pos) + 1
    out.insert(at, RU_COLUMNS['feature_description'] if kind == 'feature' else RU_COLUMNS['description'], desc,
               allow_duplicates=True)
    # 4) коды — в последних колонках
    if rj is not None:
        out.insert(out.shape[1], RULE_CODE_COL, list(obj.iloc[:, rj]), allow_duplicates=True)
    out.insert(out.shape[1], CODE_COL, codes, allow_duplicates=True)
    return out


# ------------------------------------------------------------------ числа на экране (только показ)
# Одно правило для таблиц, печати и графиков: разряды — пробел, дробная часть — точка (запятая не используется:
# «6,624» по-русски читается как 6.6). Выгрузка в Excel/CSV не меняется.
def fmt_int(x):
    """Целое число с пробелами между разрядами: 6624 -> '6 624'; NaN/None -> '—'."""
    try:
        v = float(x)
    except (TypeError, ValueError):
        return str(x)
    return f'{v:,.0f}'.replace(',', ' ') if np.isfinite(v) else '—'


def fmt_num(x, decimals=4):
    """Число для таблиц: целое значение — без дробной части ('2 683'), иначе decimals знаков после точки; NaN -> '—'."""
    try:
        v = float(x)
    except (TypeError, ValueError):
        return str(x)
    if not np.isfinite(v):
        return '—'
    return (f'{v:,.0f}' if v.is_integer() else f'{v:,.{decimals}f}').replace(',', ' ')


# Персональные данные в предпросмотре таблиц (только на экране): номера ИИН/БИН и договоров — последние 4 знака,
# JSON событий — первые 60 символов, длинные числа в нём скрыты так же.
_MASK_COLS = {'iin', 'bin_iin', 'contract_number', 'process_ref', 'contract'}
_JSON_COLS = {'old_value', 'new_value'}


def _mask_id(v):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return v
    s = str(v).strip()
    return s if len(s) <= 4 else '\u2022' * 4 + s[-4:]


def _mask_json(v):
    if not isinstance(v, str):
        return v
    s = re.sub(r'("(?:contract_code|iin|bin|bin_iin)"\s*:\s*")([^"]*)(")', lambda m: m.group(1) + _mask_id(m.group(2)) + m.group(3), v)
    s = re.sub(r'\d{8,}', lambda m: '\u2022' * 4 + m.group(0)[-4:], s)
    return s if len(s) <= 60 else s[:59] + '\u2026'


def _mask_personal(df):
    # колонки с номерами (по имени) и JSON событий; в остальных текстовых колонках — значения из 10 и более цифр
    # (ИИН/БИН, например БИН партнёра в partner_name)
    names = {C_IIN, H_IIN, C_CONTRACT, C_PROCESS_REF} | _MASK_COLS
    js = {H_OLD, H_NEW} | _JSON_COLS
    out = None
    for j, c in enumerate(df.columns):
        s = df.iloc[:, j]
        if c in names or c in js:
            v = s.astype(object).map(_mask_json if c in js else _mask_id)
        elif _is_text(s.dtype) and s.astype(object).map(lambda x: isinstance(x, str) and bool(re.fullmatch(r'\s*\d{10,}\s*', x))).any():
            v = s.astype(object).map(lambda x: _mask_id(x) if isinstance(x, str) and re.fullmatch(r'\s*\d{10,}\s*', x) else x)
        else:
            continue
        if out is None:
            out = df.copy()
        out.isetitem(j, v) if hasattr(out, 'isetitem') else out.__setitem__(c, v)
    return df if out is None else out


def _screen_cell(x):
    # одно значение текстовой колонки на экране: bool -> да/нет, пропуск -> «—», числа — fmt_num / fmt_int
    if isinstance(x, (bool, np.bool_)):
        return 'да' if x else 'нет'
    if x is None or x is pd.NaT or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return '—'
    if isinstance(x, (float, np.floating)):
        return fmt_num(x)
    if isinstance(x, (int, np.integer)):
        return fmt_int(x)
    return x


def _screen_values(df):
    """Копия русской таблицы для экрана: True/False -> да/нет, пропуски -> «—», дробные числа — fmt_num, целые — fmt_int."""
    cols = {}
    for j in range(df.shape[1]):
        s = df.iloc[:, j]
        if pd.api.types.is_bool_dtype(s.dtype):
            s = s.map({True: 'да', False: 'нет'})
        elif pd.api.types.is_float_dtype(s.dtype):
            s = s.map(fmt_num)
        elif pd.api.types.is_integer_dtype(s.dtype):
            s = s.map(fmt_int)
        elif s.dtype == object or _is_text(s.dtype):
            s = s.astype(object).map(_screen_cell)
        cols[j] = s.to_numpy()
    out = pd.DataFrame(cols, index=df.index)
    out.columns = df.columns
    return out


# display показывает каждую таблицу через ru_labels (работает и с запасным вариантом display = print);
# Series с кодами признаков в индексе показывается таблицей, чтобы рядом с подписью были описание и код.
# Только на экране: персональные номера скрыты, True/False -> да/нет, пропуски -> «—», числа — по правилу fmt_num.
if not getattr(display, '_ru_labels', False):                  # при повторном запуске ячейки не оборачиваем дважды
    _display_orig = display


def display(*objs, **kw):
    objs = [o.to_frame() if isinstance(o, pd.Series) and _is_feature_index(o.index) else o for o in objs]
    objs = [_mask_personal(o) if isinstance(o, pd.DataFrame) else o for o in objs]
    return _display_orig(*[_screen_values(r) if isinstance(r, pd.DataFrame) else r for r in map(ru_labels, objs)], **kw)


display._ru_labels = True

# ------------------------------------------------------------------ короткие русские подписи признаков и правил
# Код модуля short_labels (short_labels.py, проверен самопроверкой на всех кодах DEMO и с фотографий реального запуска)
# без изменений по смыслу: short_label(код, RULE_DESC, RU_VALUES) -> короткая русская подпись (цель <= 45, предел 55
# символов). Неизвестный код возвращается как есть. Соглашения подписей (теги — как группы признаков, раздел 7):
# «др. кредиторы» — договор другого кредитора (_oth) и сводные признаки по всем событиям, кроме событий по нашим
# договорам (в т. ч. события без номера договора: группа «ПКБ: другие кредиторы»); «наши договоры» — наш договор (_own);
# окно — в конце («, 30 дн.»); «Договор:» / «Клиент:» — переменные договора, «у нас» — по нашим договорам.
TARGET_LEN = 45      # желаемая длина подписи
MAX_LEN = 55         # жёсткий предел длины подписи

# ------------------------------------------------------------------ правила ПКБ (ячейка 3): код -> короткое название
RULE_SHORT = {
    'app_online': 'Онлайн-заявка на кредит',
    'prolong': 'Пролонгация',
    'lookup_other': 'Поиск данных о клиенте (не ПКБ)',     # «Любой поставщик (кроме ПКБ) искал информацию по субъекту»
    'ovd_sum_chg': 'Изм. суммы просрочки',
    'new_loan_refin': 'Рефинансирующий заём',               # новый заём с целью рефинансирования
    'new_loan': 'Новый заём',
    'settlement': 'Урегулирование долга',
    'st_gambler': 'Статус «Лудоман»',
    'unsubscribe': 'Отписка (нет договоров)',
    'st_recruit': 'Статус «Призывник»',
    'collateral_chg': 'Изм. по залогу',
    'report_req': 'Запрос кредитного отчёта',
    'new_contract': 'Новый договор',
    'restruct': 'Реструктуризация',                         # реструктурирован / рефинансирован / пересмотрен
    'st_stopcredit': 'Статус «Стоп-Кредит»',
    'debt_terminated': 'Закрытие с долгом',                 # прекращено без ПОЛНОГО исполнения (списание, прощение)
    'partial_ovd': 'Просрочка только у нас',                # просрочка у нас при платежах другим кредиторам
    'st_bankrupt': 'Статус «Банкротство»',
    'loan_assigned': 'Переуступка',
    'loan_closed': 'Погашение займа',                       # «закрытие займа (погашение)»; переуступка — loan_assigned
    'refin_out_80': 'Кредит в др. банке, у нас ≥80% ОД',  # «Сделать сравнение, у кого до погашения не менее 80% от ОД
                                                            # займа – появился кредит в другом банке»: по буквальному
                                                            # смыслу у нас до погашения осталось ≥80% основного долга
                                                            # (ОД); направление уточнить у владельца правила
    'instalment_chg': 'Изм. ежемес. платежа',
    'dpd_chg': 'Изм. DPD',
    'debt_chg': 'Изм. осн. долга',
}

# семейство правила (как FAMILY в ячейке 3); нужно для группировки и для code_family
_RULE_FAMILY = {
    'app_online': 'credit_hunger', 'prolong': 'relief', 'lookup_other': 'credit_hunger', 'ovd_sum_chg': 'delinquency',
    'new_loan_refin': 'credit_hunger', 'new_loan': 'credit_hunger', 'settlement': 'positive', 'st_gambler': 'status',
    'unsubscribe': 'technical', 'st_recruit': 'status', 'collateral_chg': 'other', 'report_req': 'credit_hunger',
    'new_contract': 'credit_hunger', 'restruct': 'relief', 'st_stopcredit': 'status', 'debt_terminated': 'delinquency',
    'partial_ovd': 'bank_rule', 'st_bankrupt': 'status', 'loan_assigned': 'delinquency', 'loan_closed': 'positive',
    'refin_out_80': 'bank_rule', 'instalment_chg': 'debt_burden', 'dpd_chg': 'delinquency', 'debt_chg': 'debt_burden',
}

# ------------------------------------------------------------------ семейства правил: код -> короткое название
FAMILY_SHORT = {
    'credit_hunger': 'Кредитный голод',
    'relief': 'Облегчение условий',
    'delinquency': 'Просрочка',
    'positive': 'Позитивные события',
    'status': 'Особые статусы',
    'technical': 'Технические',
    'debt_burden': 'Долговая нагрузка',
    'bank_rule': 'Правила, настроенные банком',             # правила ПКБ, настроенные банком (сравнивают нас с другими)
    'other': 'Прочие',
    'unknown': 'Неизвестные правила',
}

# группы признаков (колонка group каталога)
GROUP_SHORT = {
    'pkb_other': 'ПКБ: другие кредиторы',                   # o: не наши договоры + правила без кода договора
    'pkb_bank_rule': 'ПКБ: правила, настроенные банком',
    'pkb_own': 'ПКБ: наши договоры',
    'baseline': 'Переменные договора',
}

# тег совокупности: oth — договор другого кредитора, own — наш договор, mix — кадр o целиком (все события, кроме
# событий по нашим договорам и правил, настроенных банком, в т. ч. события правил без кода договора)
# «др. кредиторы» для oth и mix: события без номера договора в ноутбуке относятся к группе «ПКБ: другие кредиторы»
# (раздел 7), поэтому у столбцов одной группы и одного цвета один и тот же тег; own — «наши договоры», как группа
# «ПКБ: наши договоры»
_POP = {'oth': 'др. кредиторы', 'own': 'наши договоры', 'mix': 'др. кредиторы'}
# множественное число названия правила в счётчиках и «дн. с последнего» (рядом с «Онлайн-заявки в МФО: число» и т. п.);
# в единственном числе название остаётся только как имя правила (график лифта)
_RULE_PLURAL = {'app_online': 'Онлайн-заявки на кредит', 'report_req': 'Запросы кредитного отчёта',
                'new_loan': 'Новые займы', 'new_contract': 'Новые договоры'}
# правила, к которым тег _oth/_own не добавляется: правила, настроенные банком (BANK_RULES ячейки 3), и техническая отписка —
# по смыслу они не относятся к договору другого кредитора («Просрочка только у нас, др. кредиторы» противоречиво)
_NO_POP_RULES = frozenset({'partial_ovd', 'refin_out_80', 'unsubscribe'})
# тег для fam_<семейство>_oth_n_<W>d: в семействе только правила с кодом договора -> oth; есть правила без кода
# (app_online, report_req, lookup_other в кредитном голоде, settlement в позитивных) -> mix; статусы клиента — без тега
_FAM_POP = {'delinquency': 'oth', 'debt_burden': 'oth', 'relief': 'oth', 'credit_hunger': 'mix', 'positive': 'mix',
            'status': None}

# ------------------------------------------------------------------ признаки из JSON (ячейка 21), без окна
# (подпись, семейство). Окно ', W дн.' добавляется в конце.
_DERIVED = {
    # просрочка: события dpd_chg / ovd_sum_chg у других кредиторов (кадр o)
    'dpd_oth_max': ('Макс. DPD, др. кредиторы', 'delinquency'),                    # max нового значения DPD
    'dpd_oth_n_up': ('Рост DPD: число, др. кредиторы', 'delinquency'),             # v_new > v_old
    'dpd_oth_n_ge30': ('DPD ≥ 30: число событий, др. кредиторы', 'delinquency'),   # v_new >= 30 (любое изм., 45 -> 60 тоже)
    'ovdsum_oth_max': ('Макс. сумма просрочки, др. кредиторы', 'delinquency'),
    'ovdsum_oth_n_up': ('Рост суммы просрочки: число, др. кредиторы', 'delinquency'),
    'ovdsum_oth_up_sum': ('Рост суммы просрочки: итого, др. кредиторы', 'delinquency'),   # сумма положительных приростов
    # долговая нагрузка: debt_chg / instalment_chg у других кредиторов
    'debt_oth_n_up': ('Рост осн. долга: число, др. кредиторы', 'debt_burden'),
    'debt_oth_n_down': ('Снижение осн. долга: число, др. кредиторы', 'debt_burden'),
    'debt_oth_up_sum': ('Рост осн. долга: итого, др. кредиторы', 'debt_burden'),      # сумма положительных приростов
    'debt_oth_net': ('Осн. долг: чистое изм., др. кредиторы', 'debt_burden'),         # сумма (new - old)
    'inst_net': ('Ежемес. платёж: чистое изм., др. кредиторы', 'debt_burden'),
    'inst_n_up': ('Рост ежемес. платежа: число, др. кредиторы', 'debt_burden'),
    # кредитный голод: заявки и запросы без кода договора (все события, тега нет), новые займы у других кредиторов
    'app_amt_sum': ('Онлайн-заявки на кредит: общая сумма', 'credit_hunger'),
    'app_amt_max': ('Онлайн-заявки на кредит: макс. сумма', 'credit_hunger'),         # макс. сумма одной заявки
    'app_n_mfo': ('Онлайн-заявки в МФО: число', 'credit_hunger'),                    # creditor_type = 2
    'inq_n_mfo': ('Запросы кредитного отчёта от МФО: число', 'credit_hunger'),
    'newloan_n_mfo': ('Новые займы в МФО: число, др. кредиторы', 'credit_hunger'),   # new_loan / new_contract / new_loan_refin
    'newloan_amt_sum': ('Новые займы: общая сумма, др. кредиторы', 'credit_hunger'),
    # облегчение условий, статусы
    'prolong_days_sum': ('Суммарный срок пролонгаций, др. кредиторы', 'relief'),      # сумма дней пролонгации
    'status_on_n': ('Установка особого статуса: число', 'status'),                   # f_new == 1 по статусным правилам
    # наши договоры (проверка на утечку)
    'dpd_own_max': ('Макс. DPD, наши договоры', 'delinquency'),
    # все правила, кадр o (без событий по нашим договорам, в т. ч. правила без кода договора)
    'all_oth_n': ('Все события: число, др. кредиторы', 'all'),
    'all_oth_nrules': ('Число сработавших правил, др. кредиторы', 'all'),           # nunique(rule)
}

# ------------------------------------------------------------------ переменные договора (ячейка 15)
_BASELINE = {
    'bl_product': 'Договор: продукт',
    'bl_channel': 'Договор: канал продаж',
    'bl_amount': 'Договор: сумма выдачи',                          # contract_amount («Сумма договора»)
    'bl_mob_days': 'Договор: дней с выдачи',                       # score_date - open_date
    'bl_term_days': 'Договор: плановый срок, дн.',                 # end_date - open_date
    'bl_term_elapsed': 'Договор: доля истёкшего срока',            # mob / term
    'bl_i_n_dlq_now': 'Клиент: число других текущих просрочек у нас',    # другие эпизоды клиента, идущие на дату скоринга
    'bl_i_n_prev_roll': 'Клиент: число предыдущих перекатов в DPD 31+ у нас',   # dpd_asof >= 31
    'bl_i_n_open_contracts': 'Клиент: число наших открытых договоров',
    'bl_i_open_amount': 'Клиент: сумма выдачи по нашим открытым договорам',    # сумма contract_amount, не остаток долга
}
# bl_{lvl}_*: lvl 'c' — по договору (key 'contract'), lvl 'i' — по клиенту (key 'iin', все наши договоры клиента)
_LVL_PREFIX = {'c': 'Договор:', 'i': 'Клиент:'}
_LVL_OURS = {'c': '', 'i': ' у нас'}

_RE_WINDOW = re.compile(r'^(?P<base>.+)_(?P<w>\d+)d$')
_RE_BL_LVL = re.compile(r'^bl_(?P<lvl>[ci])_(?P<what>n_prev_ep|max_prev_dpd|days_since_prev)(?:_(?P<w>\d+)d)?$')
_RE_FAM = re.compile(r'^fam_(?P<fam>.+)_oth_n_(?P<w>\d+)d$')


def _win(w):
    return f', {int(w)} дн.'


def _get(mapping, key):
    # безопасное обращение к dict / pandas.Series / любому объекту с .get
    if mapping is None or isinstance(mapping, str):
        return None
    try:
        v = mapping.get(key)
    except Exception:
        return None
    return v if isinstance(v, str) and v.strip() else None


# хвостовые слова, на которых обрезанная подпись не должна заканчиваться: предлоги, союзы, прилагательные
_RE_DANGLING = re.compile(r'(?i)^(?:по|в|во|на|с|со|у|о|об|от|до|для|при|из|за|к|ко|и|или|а|без|над|под|'
                          r'\w{3,}(?:ой|ый|ая|ое|ые|ых|ого|ому|ую))$')


def _cut(s, limit):
    """Обрезка по границе слова с многоточием; подпись не заканчивается предлогом или прилагательным."""
    s = s.strip()
    if len(s) <= limit:
        return s
    if limit < 2:
        return s[:max(limit, 1)]
    cut = s[:limit - 1]
    if s[limit - 1] != ' ' and ' ' in cut[3:]:                         # не рвать слово посередине
        cut = cut[:cut.rfind(' ')]
    words = cut.rstrip(' -–—:;,.«(').split(' ')
    while len(words) > 2 and _RE_DANGLING.match(words[-1].strip(' -–—:;,.«(')):
        words.pop()
    return ' '.join(words).rstrip(' -–—:;,.«(') + '…'


def _shorten_desc(desc, limit=28):
    """Русское название правила ПКБ -> короткое существительное: без «Мониторинг ...», скобок и хвостов."""
    s = str(desc)
    s = re.sub(r'\([^)]*\)?', ' ', s)                                   # скобки (в т. ч. незакрытые после обрезки до 80)
    s = re.sub(r'(?i)^\s*мониторинг\w*\s+(факта\s+)?', '', s)
    s = re.sub(r'\s+', ' ', s).strip(' -–—:;,.')
    s = re.sub(r'(?i)^изменени[еяйю]\b\s*', 'Изм. ', s).strip()         # как в RULE_SHORT: «Изм. DPD», «Изм. по залогу»
    s2 = re.sub(r'(?i)\s+по\s+(?:займ|кредит|договор|контракт)\w*(?:\s+(?:займа|кредита|договора|контракта))?$',
                '', s)                                                  # хвост «по займу / кредиту / договору займа»
    if len(s2.split()) >= 2:
        s = s2
    if not s or s == 'Изм.':
        s = str(desc).strip() or '?'
    s = s[0].upper() + s[1:]
    return _cut(s, limit)


def _rule_subject(base, rule_desc=None, plural=False):
    """Короткое название правила по его коду без суффикса _oth/_own; None, если правило неизвестно.
    plural=True — форма для счётчиков и «дн. с последнего» (_RULE_PLURAL), если она задана."""
    if plural and base in _RULE_PLURAL:
        return _RULE_PLURAL[base]
    if base in RULE_SHORT:
        return RULE_SHORT[base]
    if isinstance(rule_desc, str) and rule_desc.strip():
        return _shorten_desc(rule_desc)
    d = _get(rule_desc, base)
    if d:
        return _shorten_desc(d)
    if base.startswith('other_') and len(base) > 6:                      # other_<slug>: slug — слова названия правила
        slug = base[6:]
        words = slug.replace('_', ' ').strip()
        if words:
            s = _shorten_desc(words)
            if len(slug) >= 30 and not s.endswith('…'):                 # slug обрезан до 30 символов в map_rule_names:
                tokens = s.split(' ')                                   # последнее слово, скорее всего, неполное —
                if len(tokens) >= 3 or (len(tokens) == 2 and len(tokens[1]) <= 3 and tokens[0] != 'Изм.'):
                    tokens = tokens[:-1]                                # убираем его, если остаётся осмысленная часть
                s = _cut(' '.join(tokens), 27)
                s = s if s.endswith('…') else s + '…'
            return s
    return None


def _split_rkey(rk):
    if rk.endswith('_oth'):
        return rk[:-4], 'oth'
    if rk.endswith('_own'):
        return rk[:-4], 'own'
    return rk, None


def _fit(subject, tail):
    """subject + tail; если длиннее MAX_LEN (длинное название правила, трёхзначное окно), сокращает subject."""
    s = subject + tail
    if len(s) > MAX_LEN:
        s = _cut(subject, max(MAX_LEN - len(tail), 6)) + tail
    return s


def _rkey_label(rk, rule_desc, tail_fn, plural=False):
    base, pop = _split_rkey(rk)
    subj = _rule_subject(base, rule_desc, plural)
    if subj is None and pop is None:
        return None
    if subj is None:                                                    # суффикс есть, но правило неизвестно
        subj = _rule_subject(rk, rule_desc)
        pop = None
        if subj is None:
            return None
    if base in _NO_POP_RULES:                                           # правила, настроенные банком, отписка — без тега
        pop = None
    return _fit(subj, tail_fn(_POP.get(pop)))


def _family_name(fam, family_ru=None):
    if fam in FAMILY_SHORT:
        return FAMILY_SHORT[fam]
    if isinstance(family_ru, str) and family_ru.strip():
        return _cut(family_ru[0].upper() + family_ru[1:], 20)
    v = _get(family_ru, fam)
    if v:
        return _cut(v[0].upper() + v[1:], 20)
    return None


def short_label(code, rule_desc=None, family_ru=None):
    """Короткая русская подпись кода признака / правила для графиков (<= 55 символов).

    code      — имя признака (n_dpd_chg_oth_30d, bl_i_max_prev_dpd ...), ключ правила (dpd_chg_oth, app_online),
                код семейства или группы;
    rule_desc — словарь RULE_DESC ноутбука (код правила -> русское описание) или строка-описание этого правила;
                нужен для нераспознанных правил other_<slug>;
    family_ru — словарь семейство -> русское название (например, RU_VALUES) или строка; нужен только для семейств,
                которых нет в FAMILY_SHORT.
    Неизвестный код возвращается как есть; функция не бросает исключений."""
    try:
        return _short_label(code, rule_desc, family_ru)
    except Exception:
        return str(code)


def _short_label(code, rule_desc, family_ru):
    if not isinstance(code, str):
        return str(code)
    c = code.strip()
    if not c:
        return code

    # ---- переменные договора
    if c.startswith('bl_'):
        if c in _BASELINE:
            return _BASELINE[c]
        m = _RE_BL_LVL.match(c)
        if m:
            lvl, what, w = m.group('lvl'), m.group('what'), m.group('w')
            p, ours = _LVL_PREFIX[lvl], _LVL_OURS[lvl]
            if what == 'n_prev_ep':
                return f'{p} число предыдущих просрочек{ours}' + (_win(w) if w else '')
            if w:
                return code
            if what == 'max_prev_dpd':
                return f'{p} макс. DPD предыдущих просрочек{ours}'
            # score_date - последний день просрочки ДРУГИХ (более ранних) эпизодов; текущий эпизод не учитывается
            return f'{p} дней с конца предыдущей просрочки{ours}'
        return code

    # ---- группы и семейства, переданные как значения
    if c in GROUP_SHORT:
        return GROUP_SHORT[c]
    if c in FAMILY_SHORT:
        return FAMILY_SHORT[c]

    # ---- дни с последнего события (вся история)
    if c == 'days_since_any_oth':
        return f'Любое событие: дн. с последнего, {_POP["mix"]}'
    if c.startswith('days_since_'):
        lab = _rkey_label(c[len('days_since_'):], rule_desc,
                          lambda pop: ': дн. с последнего' + (f', {pop}' if pop else ''), plural=True)
        return lab or code

    # ---- признаки с окном W
    m = _RE_WINDOW.match(c)
    if m:
        base, w = m.group('base'), m.group('w')
        if base in _DERIVED:
            return _DERIVED[base][0] + _win(w)
        mf = _RE_FAM.match(c)
        if mf:
            fam_code = mf.group('fam')
            fam = _family_name(fam_code, family_ru)
            if fam is None:
                return code
            pop = _POP.get(_FAM_POP.get(fam_code, 'mix'))
            return _fit(fam, ': всего' + (f', {pop}' if pop else '') + _win(w))
        if base.startswith('n_'):
            lab = _rkey_label(base[2:], rule_desc,
                              lambda pop: ': число' + (f', {pop}' if pop else '') + _win(w), plural=True)
            return lab or code
        return code

    # ---- ключ правила (таблица влияния правил, график лифта): dpd_chg_oth, app_online, other_<slug>_own ...
    lab = _rkey_label(c, rule_desc, lambda pop: f', {pop}' if pop else '')
    return lab or code


def full_label(code, rule_desc=None, family_ru=None):
    """'короткая подпись (код)'; для неизвестного кода — сам код."""
    s = short_label(code, rule_desc, family_ru)
    return s if s == str(code) else f'{s} ({code})'


def short_labels(codes, rule_desc=None, family_ru=None):
    """Список коротких подписей без повторов (при совпадении к подписи добавляется код) — для осей matplotlib,
    где одинаковые подписи категорий слились бы в один столбец."""
    codes = list(codes)
    labs = [short_label(c, rule_desc, family_ru) for c in codes]
    seen = {}
    for lab in labs:
        seen[lab] = seen.get(lab, 0) + 1
    return [lab if seen[lab] == 1 or lab == str(c) else f'{lab} ({c})' for c, lab in zip(codes, labs)]


def label_map(codes, rule_desc=None, family_ru=None):
    """{код: короткая подпись} без повторов; удобно для DataFrame.rename(index=...)."""
    codes = list(codes)
    return dict(zip(codes, short_labels(codes, rule_desc, family_ru)))


def code_family(code):
    """Семейство кода (для группировки): baseline, all, семейство правила или unknown."""
    if not isinstance(code, str):
        return 'unknown'
    c = code.strip()
    if c.startswith('bl_'):
        return 'baseline'
    if c in GROUP_SHORT:
        return 'group'
    if c in FAMILY_SHORT:
        return c
    if c == 'days_since_any_oth':
        return 'all'
    m = _RE_WINDOW.match(c)
    if m:
        base = m.group('base')
        if base in _DERIVED:
            return _DERIVED[base][1]
        mf = _RE_FAM.match(c)
        if mf:
            return mf.group('fam')
        if base.startswith('n_'):
            c = base[2:]
    elif c.startswith('days_since_'):
        c = c[len('days_since_'):]
    base, _ = _split_rkey(c)
    return _RULE_FAMILY.get(base, 'unknown')


# ------------------------------------------------------------------ подписи на графиках и в первых колонках таблиц
def code_name(code):
    """Короткая русская подпись признака или правила: short_label модуля выше; RULE_DESC (ячейка 3) и RU_VALUES
    берутся в момент вызова (по RULE_DESC подписываются правила other_<...> реальных данных). Если модулю код не
    известен, берётся начало полного русского описания (code_desc), чтобы на графике не было латиницы."""
    s = short_label(code, globals().get('RULE_DESC'), RU_VALUES)
    if s == str(code):
        d = code_desc(code)
        if d:
            s = _cut(d, MAX_LEN)
    return s


_RE_NO_POP = re.compile(r'(' + '|'.join(sorted(_NO_POP_RULES)) + r')_(?:oth|own)(?=_\d+d$|$)')


def _known_codes():
    """Все коды, которые могут встретиться на графиках этого ядра: признаки из каталогов (catalog, catalog_B) и, для
    каждого правила из RULE_DESC / RULE_SHORT, ключи с суффиксами _oth/_own, счётчики n_<ключ>_<W>d и days_since_<ключ>."""
    known = set()
    for name in _CATALOG_NAMES:
        cat = globals().get(name)
        if isinstance(cat, pd.DataFrame):
            known.update(c for c in cat.index if isinstance(c, str))
    for base in set(globals().get('RULE_DESC') or {}) | set(RULE_SHORT):
        for rk in (base, base + '_oth', base + '_own'):
            known.update([rk, f'days_since_{rk}'] + [f'n_{rk}_{w}d' for w in WINDOWS])
    return known


def code_names(codes):
    """Подписи code_name для списка кодов в том же порядке (оси графика, первая колонка таблицы). Одинаковые коды
    получают одну подпись. Если у разных кодов подписи совпадают (например, два похожих правила ПКБ other_<...>),
    к подписи добавляется номер « (1)», « (2)» — без латиницы. Номер не зависит от графика: коды с одной подписью
    нумеруются по алфавиту кода среди всех известных кодов ядра (_known_codes) и номер ставится, даже если на графике
    только один из них. Код виден в «Расшифровке обозначений на графике» и в колонке «Код в данных»."""
    codes = list(codes)
    uniq = list(dict.fromkeys(codes))
    lab = {c: code_name(c) for c in uniq}
    # варианты правил без тега (_NO_POP_RULES) с суффиксом и без него — один и тот же код по смыслу (в одном запуске
    # встречается только один из них), поэтому номер им не нужен
    canon = lambda c: _RE_NO_POP.sub(r'\1', c) if isinstance(c, str) else c         # noqa: E731
    groups = {}
    for c in _known_codes() | set(uniq):
        groups.setdefault(code_name(c), set()).add(canon(c))
    for c in uniq:
        same = sorted(groups.get(lab[c], {canon(c)}), key=str)
        if len(same) > 1:
            lab[c] = f'{lab[c]} ({same.index(canon(c)) + 1})'
    return [lab[c] for c in codes]


def chart_label(name, width=45):
    """Подпись для графика с переносом по width символов: окно («30 дн.») и «др. кредиторы» не разрываются, на последней
    строке не остаётся короткий хвост (тогда строка удлиняется на 12 символов); $ в matplotlib включает режим формул."""
    s = re.sub(r'(\d) (дн\.)|(др\.) (кредитор)|(наши) (договоры)', lambda m: '\u00a0'.join(g for g in m.groups() if g),
               str(name))
    lines = textwrap.wrap(s, width)
    if len(lines) > 1 and len(lines[-1]) < 12:
        lines = textwrap.wrap(s, width + 12)
    return '\n'.join(lines).replace('\u00a0', ' ').replace('$', r'\$')


# ------------------------------------------------------------------ подписи бинов (только показ и текст выгрузки)
# Подписи бинов fit_bins (раздел 9): '<= b', '(a, b]', '> a', 'all' (один бин на все значения), 'MISSING' (пропуск),
# 'OTHER' и категории. Для графиков — bin_names (читаемо: «0», «1–2», «≥ 7», «≤ 7 дн.», «78–150 тыс.»), для таблиц и
# Excel — bin_exact (точные границы в русской записи: «(213 000; 459 000]»). Пропуск у признаков ПКБ означает «в окне не
# было такого события» (счётчики событий пропусков не имеют: там 0).
_RE_BNUM = r'-?(?:inf|\d[\d,]*(?:\.\d+)?(?:e[+-]?\d+)?)'


def _bin_parse(label):
    """('le', b) для '<= b', ('gt', a) для '> a', ('in', a, b) для '(a, b]'; None — не интервал."""
    if not isinstance(label, str):
        return None
    num = lambda t: float(t.replace(',', ''))                                        # noqa: E731
    m = re.fullmatch(r'<= (' + _RE_BNUM + ')', label)
    if m:
        return 'le', num(m.group(1))
    m = re.fullmatch(r'> (' + _RE_BNUM + ')', label)
    if m:
        return 'gt', num(m.group(1))
    m = re.fullmatch(r'\((' + _RE_BNUM + r'), (' + _RE_BNUM + r')\]', label)
    if m:
        return 'in', num(m.group(1)), num(m.group(2))
    return None


def _bin_kind(code):
    # что измеряет признак: count (число событий), days (дни), dpd, money (суммы, тенге) или num (прочее)
    c = re.sub(r'_\d+d$', '', str(code))
    if (c.startswith('n_') or c.endswith(('_n', '_nrules', '_n_up', '_n_down', '_n_ge30', '_n_mfo'))
            or re.match(r'bl_[ci]_n_', c)):
        return 'count'
    if 'days_since' in c or c in ('bl_mob_days', 'bl_term_days') or c.endswith('_days_sum'):
        return 'days'
    if 'dpd' in c:
        return 'dpd'
    if re.search(r'amt|amount|^ovdsum_|^debt_|^inst_', c):
        return 'money'
    return 'num'


def _bin_is_pkb(code):
    return isinstance(code, str) and not code.startswith('bl_') and feature_desc(code) != ''


def _bin_missing_name(code):
    # подпись бина пропусков по смыслу признака
    if isinstance(code, str) and code.startswith('bl_'):
        return 'не было прошлых просрочек' if 'days_since_prev' in code else 'нет данных'
    return 'не было события' if _bin_is_pkb(code) else 'пропуск (нет значения)'


def _bin_all_name(code, has_missing):
    # подпись единственного бина всех непустых значений ('all')
    if has_missing and isinstance(code, str) and code.startswith('bl_') and 'days_since_prev' in code:
        return 'были прошлые просрочки'
    return 'было событие' if has_missing and _bin_is_pkb(code) else 'все значения'


def bin_exact(code, label, has_missing=False):
    """Подпись бина для таблиц и Excel: точные границы в русской записи (разряды — пробел, точка с запятой между
    границами): '(213,000, 459,000]' -> '(213 000; 459 000]', '<= 0' -> '≤ 0'; пропуск и единый бин — по смыслу признака
    code (None — код неизвестен); категории — по RU_VALUES."""
    if not isinstance(label, str):
        return label
    if label == 'MISSING':
        return _bin_missing_name(code)
    if label == 'all':
        return _bin_all_name(code, has_missing)
    t = lambda x: x.replace(',', ' ')                                                  # noqa: E731
    m = re.fullmatch(r'\((' + _RE_BNUM + r'), (' + _RE_BNUM + r')\]', label)
    if m:
        return f'({t(m.group(1))}; {t(m.group(2))}]'
    m = re.fullmatch(r'(<=|>) (' + _RE_BNUM + ')', label)
    if m:
        return f'{"≤" if m.group(1) == "<=" else ">"} {t(m.group(2))}'
    return RU_VALUES.get(label, label)


def _bin_money(v):
    # граница-сумма: (число, единица) — '78', 'тыс.' / '1.2', 'млн' / '950', ''
    a = abs(v)
    if a >= 1e6:
        x, u = f'{v / 1e6:.1f}'.rstrip('0').rstrip('.'), 'млн'
    elif a >= 1e3:
        x, u = f'{v / 1e3:.0f}', 'тыс.'
    else:
        x, u = fmt_num(v, 2), ''
    return x.replace('-', '−'), u


def _bin_g(v):
    # граница общего вида: целое — с пробелами между разрядами, иначе 3 значащие цифры, точка
    return (fmt_int(v) if float(v).is_integer() else f'{v:.3g}').replace('-', '−')


def bin_names(code, labels, x=None):
    """Подписи бинов признака code для оси графика (читаемые, без латиницы): счётчики и DPD — целыми значениями
    ('<= 0' -> '0', '(0, 2]' -> '1–2', '> 6' -> '≥ 7'), дни — целыми днями, суммы — в тыс. / млн, пропуск и единый
    бин — по смыслу признака. x — значения признака (по ним видно, целые ли они). Если округление сделало подписи
    одинаковыми, возвращаются точные подписи bin_exact."""
    labels = list(labels)
    has_miss = 'MISSING' in labels
    kind = _bin_kind(code)
    v = pd.to_numeric(pd.Series(x), errors='coerce').to_numpy(dtype=float) if x is not None else np.array([])
    v = v[np.isfinite(v)]
    is_int = kind in ('count', 'dpd') or (len(v) > 0 and bool(np.all(v == np.round(v))))
    nonneg = kind in ('count', 'dpd', 'days') or (len(v) > 0 and v.min() >= 0)
    rnd = lambda z: int(np.floor(z + 0.5))                                             # noqa: E731
    out = []
    for lab in labels:
        p = _bin_parse(lab)
        if p is None:
            out.append(bin_exact(code, lab, has_miss))
            continue
        unit = ' дн.' if kind == 'days' else ''
        if kind == 'money':
            if p[0] == 'in':
                (xa, ua), (xb, ub) = _bin_money(p[1]), _bin_money(p[2])
                out.append(f'{xa}–{xb} {ub}'.strip() if ua == ub else f'{xa} {ua} – {xb} {ub}'.strip())
            else:
                xb, ub = _bin_money(p[1])
                out.append(f'{"≤" if p[0] == "le" else ">"} {xb} {ub}'.strip())
        elif is_int and kind != 'money':
            if p[0] == 'le':
                hi = int(np.floor(p[1]))
                out.append(('0' if hi <= 0 else f'0–{hi}') + unit if nonneg and hi >= 0 else f'≤ {_bin_g(hi)}{unit}')
            elif p[0] == 'gt':
                out.append(f'≥ {_bin_g(int(np.floor(p[1])) + 1)}{unit}')
            else:
                lo, hi = int(np.floor(p[1])) + 1, int(np.floor(p[2]))
                out.append((f'{_bin_g(lo)}' if lo >= hi else f'{_bin_g(lo)}–{_bin_g(hi)}') + unit)
        elif kind == 'days':
            if p[0] == 'in':
                a, b = rnd(p[1]), rnd(p[2])
                out.append(f'{a}–{b} дн.' if a != b else f'{p[1]:.1f}–{p[2]:.1f} дн.')
            else:
                out.append(f'{"≤" if p[0] == "le" else ">"} {rnd(p[1])} дн.')
        else:
            if p[0] == 'in':
                out.append(f'{_bin_g(p[1])}–{_bin_g(p[2])}')
            else:
                out.append(f'{"≤" if p[0] == "le" else ">"} {_bin_g(p[1])}')
    if len(set(out)) < len(out):
        out = [bin_exact(code, lab, has_miss) for lab in labels]
    return out


def iv_class_order(t):
    """Таблица с колонками-классами силы IV (crosstab): колонки от сильных к бесполезным, отсутствующие классы — нулями."""
    order = ['suspicious: check leakage', 'strong', 'medium', 'weak', 'not useful']
    cols = [c for c in order if c in t.columns or c != order[0]] + [c for c in t.columns if c not in order]
    out = t.reindex(columns=cols, fill_value=0)
    out.columns.name = t.columns.name
    return out


def show_chart_legend(codes, names=None):
    """Под графиком: таблица «Расшифровка обозначений на графике» — только элементы этого графика, в его порядке
    (сверху вниз, для сетки — по строкам): подпись на графике, полное русское описание и код в данных."""
    codes = list(codes)
    if not codes:
        return
    names = code_names(codes) if names is None else list(names)
    t = pd.DataFrame({'Название на графике': [str(n).replace('\n', ' ') for n in names],
                      'Что означает': [code_desc(c) for c in codes], CODE_COL: codes},
                     index=pd.RangeIndex(1, len(codes) + 1, name='№'))
    print('Расшифровка обозначений на графике:')
    _display_orig(t)                                             # таблица уже на русском: ru_labels не нужен

# скрываем технические предупреждения pandas/numpy и задаём формат вывода таблиц
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 170)                    # описание признака (до 160 символов) показывается целиком
pd.set_option('display.float_format', fmt_num)              # '2 683', '0.2155': разряды пробелом, точка

# ------------------------------------------------------------------ режим запуска
DEMO_MODE = False           # False: ваши датафреймы уже загружены в этом ядре. True: синтетические демо-данные.
SEED = 42                   # зерно генератора случайных чисел (воспроизводимость демо-данных, выборок и бутстрэпа)

# ------------------------------------------------------------------ имена ВАШИХ датафреймов в этом ядре
OVRD_VAR = 'ovrd_contr'     # строки с просрочкой, объединённые с атрибутами договоров
HES_VAR = 'hes'             # события мониторинга ПКБ (Первое кредитное бюро)
CONTRACTS_VAR = 'contracts' # необязательно: все выданные договоры (нужны для части B); если такого объекта нет, часть B просто пропускается
# необязательно: ВСЕ номера договоров банка по этим клиентам (любые продукты, в том числе без просрочки) — имя объекта
# в ядре в кавычках (например 'all_contracts') или сам объект: DataFrame с колонками contract_number и/или process_ref,
# Series или список. Номера лучше загружать как текст (dtype=str): длинные номера, прочитанные как числа (Excel, float),
# теряют последние цифры. Без него «нашими» считаются только договоры из ovrd_contr (и contracts), а остальные договоры
# банка попадают в «ПКБ: другие кредиторы» (раздел 7).
OWN_CODES_VAR = None        # например 'all_contracts' или all_contracts; None — не использовать

# ------------------------------------------------------------------ имена столбцов (измените, если у вас они другие)
# столбцы наших данных (ovrd_contr / contracts): договор, ИИН, дата среза, дни просрочки (DPD)
C_CONTRACT, C_IIN, C_ACT_DATE, C_DPD = 'contract_number', 'iin', 'act_date', 'ovd_days_cnt'
# process_ref (ещё один номер договора, тоже сверяется с contract_code из ПКБ), даты открытия, закрытия и окончания договора
C_PROCESS_REF, C_OPEN, C_CLOSE, C_END = 'process_ref', 'open_date', 'close_date', 'end_date'
# статус договора, сумма договора, партнёр, продукт
C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT = 'status_name', 'contract_amount', 'partner_name', 'product_id'
# столбцы таблицы событий ПКБ (hes): ИИН/БИН клиента, название правила, старое и новое значение (JSON)
H_IIN, H_RULE, H_OLD, H_NEW = 'bin_iin', 'rule_name', 'old_value', 'new_value'
# идентификатор события, дата события в ПКБ, дата поступления события в банк
H_ID, H_EVENT_DATE, H_CREATED = 'internal_event_id', 'event_date', 'created_date'

# ------------------------------------------------------------------ логика «на момент скоринга» (без заглядывания в будущее)
EVENT_TIME_COL = H_CREATED  # когда событие поступило в банк (безопасный вариант). event_date = когда оно произошло в ПКБ.
EVENT_LAG_DAYS = 0          # сколько дней ещё выждать, прежде чем событие можно использовать (запас на задержку в промышленной среде)
OBS_DPD = 1                 # точка скоринга: день, когда эпизод достигает этого DPD (1 = первый день софт-коллекшн)
BAD_DPD = 31                # основная целевая переменная (таргет): эпизод достигает DPD >= 31
EXTRA_TARGETS = {'nocure_16': 16}   # дополнительные таргеты: имя -> порог DPD (добавьте 'roll_61': 61, когда истории станет достаточно)
WINDOWS = (7, 14, 30)       # окна наблюдения в днях
IMPACT_WINDOW = 14          # окно для таблицы лифта по правилам (раздел 11)
MIN_HISTORY_DAYS = 14       # первая дата скоринга = начало мониторинга + столько дней
HES_START = None            # начало мониторинга; None = определить по дневным объёмам, либо дата, например '2026-09-07'
DATA_END = None             # последний день с полными данными по DPD; None = max(act_date)
EPISODE_START_TOL_DAYS = 1  # допустимый разброс dlq_start (в днях) внутри одного эпизода
BANK_RULES = ('partial_ovd', 'refin_out_80')   # правила ПКБ, настроенные банком: сравнивают наш договор с другими кредиторами

# ------------------------------------------------------------------ биннинг, отбор признаков, модель
MAX_BINS = 6                # максимальное число бинов (интервалов) на признак
MIN_BIN_SHARE = 0.05        # минимальная доля наблюдений в бине; более мелкие бины объединяются с соседними
MIN_BADS_RELIABLE = 100     # IV помечается как ненадёжная, если вне основного бина меньше стольких «плохих»
IV_SELECT = 0.02            # минимальная IV, чтобы признак ПКБ стал кандидатом в модель
CORR_MAX = 0.75             # максимальная |корреляция Спирмена| с уже отобранным признаком
MAX_MODEL_FEATURES = 15     # максимальное число отбираемых признаков ПКБ в модели
TRAIN_SHARE = 0.70          # доля обучающей выборки; разбиение вне времени (OOT) по дате скоринга
N_BOOT = 200                # число итераций бутстрэпа для доверительного интервала (ДИ) прироста Джини
RUN_HGB = True              # контрольная проверка прироста качества на градиентном бустинге
RUN_HES_DQ = True           # проверка качества данных по всей таблице hes (понедельные количества по правилам, задержки)
DQ_SAMPLE = 2_000_000       # сколько строк hes брать в выборку для статистики задержек и предварительных проверок
DQ_GAP_SHARE = 0.35         # провал загрузки: за день событий правила меньше этой доли от его обычного уровня ...
DQ_GAP_MIN_DAYS = 2         # ... не меньше стольких дней подряд (обычный уровень — медиана предыдущих нормальных дней)
DQ_RECENT_DAYS = 14         # последние полные дни, поток событий которых сравнивается с периодом выборки A
DQ_REGIME_RATIO = 3         # резкое изменение потока: стало больше чем в 3 раза больше или меньше, чем в периоде выборки A
DQ_DUP_MAX_ROWS = 300_000_000   # повторы internal_event_id по неделям считаются, если строк в hes не больше
                                # (хеш считается частями по 5 млн строк: дополнительно ~0.7 ГБ на 85 млн строк + ~0.6 ГБ)
# праздничные (нерабочие) дни Казахстана, включая переносы выходных по Трудовому кодексу: в такие дни поток событий
# как в выходной, поэтому они не считаются провалом загрузки (раздел 5). Курбан айт — по ожидаемой дате (уточните);
# добавьте переносы рабочих дней по постановлению правительства, если они были в вашем периоде.
DQ_HOLIDAYS = ['2025-01-01', '2025-01-02', '2025-01-07', '2025-03-10', '2025-03-21', '2025-03-24', '2025-03-25',
               '2025-05-01', '2025-05-07', '2025-05-09', '2025-06-06', '2025-07-07', '2025-09-01', '2025-10-27',
               '2025-12-16',
               '2026-01-01', '2026-01-02', '2026-01-07', '2026-03-09', '2026-03-23', '2026-03-24', '2026-03-25',
               '2026-05-01', '2026-05-07', '2026-05-11', '2026-05-27', '2026-07-06', '2026-08-31', '2026-10-26',
               '2026-12-16',
               '2027-01-01', '2027-01-04', '2027-01-07', '2027-03-08', '2027-03-22', '2027-03-23', '2027-03-24',
               '2027-05-03', '2027-05-07', '2027-05-10', '2027-07-06', '2027-08-30', '2027-10-25', '2027-12-16']
STOP_ON_PREFLIGHT_FAIL = True   # True: остановить ноутбук, если предварительная проверка дала статус ОШИБКА (FAIL)

# ------------------------------------------------------------------ главный раздел 10: влияние каждого правила ПКБ на риск
RULE_WINDOW = 30            # было ли событие правила за последние столько дней до даты скоринга (основное окно)
RULE_EXTRA_WINDOWS = (7, 14)   # те же расчёты для коротких окон — справочно
MIN_EVENT_EPISODES = 200    # «мало данных — оценить нельзя», если событие было меньше чем у стольких эпизодов ...
MIN_EVENT_BADS = 30         # ... или «плохих» среди них меньше стольких — и наблюдаемых, и ожидаемых без влияния (эпизоды
                            # с событием x общая доля «плохих»): так правило, сильно снижающее риск, не прячется в «мало данных»
RULE_ALPHA = 0.05           # порог значимости для q-значения (p-значение с поправкой на число правил)
# сила влияния — большая из двух оценок: по IV лучшего признака правила (вклад во весь портфель) и по тому, во сколько
# раз меняется риск у эпизодов с событием (для клиента с событием)
RULE_IV_STRONG = 0.10       # по IV: от 0.10 — сильное,
RULE_IV_MEDIUM = 0.02       # от 0.02 до 0.10 — среднее, ниже 0.02 — слабое
RULE_RR_STRONG = 2.0        # по риску: в 2 раза выше (ниже) и больше — сильное, если граница 95% ДИ не ближе к 1, чем 1.4,
RULE_RR_MEDIUM = 1.4        # в 1.4 раза и больше — среднее, если граница 95% ДИ не ближе к 1, чем RULE_RR_NEGLIGIBLE
RULE_IV_NEGLIGIBLE = 0.01   # «практически нет влияния»: разница значима, но IV лучшего признака ниже этого ...
RULE_RR_NEGLIGIBLE = 1.15   # ... и риск при событии отличается меньше чем в 1.15 раза. «Нет влияния»: разница незначима и
                            # весь 95% ДИ риска при событии — от 1/1.15 до 1.15 раза; иначе «влияние не подтверждено»
RULE_RARE_SHARE = 0.02      # пометка «редкое событие» в выводе: событие меньше чем у 2% эпизодов
RULE_STAB_KEEP = 1 / 3      # «неустойчиво»: на тесте эффект ослаб не случайно (разница с обучением значима) — сменил
                            # направление или стал меньше трети эффекта на обучении
RULE_BOOT = 400             # число повторов бутстрэпа по клиентам для 95% доверительных интервалов (ДИ)

# ------------------------------------------------------------------ часть B: раннее предупреждение (EWS) (нужен `contracts`)
RUN_EWS = True              # выполнять часть B (только если загружена таблица contracts)
EWS_HORIZON = 30            # договор без просрочки выходит на просрочку в течение стольких дней ...
EWS_MIN_DPD = 1             # ... и этот эпизод достигает как минимум такого DPD
EWS_SNAPSHOT_FREQ_DAYS = 7  # шаг (в днях) между датами среза для популяции B
EWS_MAX_ROWS = 400_000      # максимум строк в популяции B; при превышении берётся случайная подвыборка

# необязательно: названия продуктов для таблицы «IV в разрезе продуктов» (раздел 13), код продукта -> название,
# например {'16007': 'Кредит наличными'}; без названия показывается код
PRODUCT_NAMES = {}

OUTPUT_DIR = 'pkb_iv_output'   # папка для сохранения результатов (Excel или CSV)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# стиль графиков: тонкие линии, тонкая сетка, фиксированный порядок категорий
# PAL: палитра цветов (ключи - внутренние имена цветов, не менять)
PAL = {'blue': '#2a78d6', 'orange': '#eb6834', 'aqua': '#1baf7a', 'yellow': '#eda100',
       'magenta': '#e87ba4', 'green': '#008300', 'violet': '#6250d6', 'red': '#e34948',
       'gray': '#8c8b85', 'ink': '#0b0b0b', 'muted': '#52514e'}
SERIES = [PAL[k] for k in ('blue', 'orange', 'aqua', 'yellow', 'magenta', 'green', 'violet', 'red')]   # порядок цветов линий
# цвет группы признаков: pkb_other - другие кредиторы, pkb_bank_rule - правила, настроенные банком,
# pkb_own - наши договоры, baseline - переменные договора
GROUP_COLOR = {'pkb_other': PAL['blue'], 'pkb_bank_rule': PAL['orange'], 'pkb_own': PAL['aqua'], 'baseline': PAL['yellow']}
plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae', 'axes.linewidth': 0.6,
    'axes.titlesize': 10.5, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'axes.labelcolor': PAL['muted'],
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'xtick.color': PAL['muted'], 'ytick.color': PAL['muted'],
    'legend.fontsize': 8, 'legend.frameon': False, 'lines.linewidth': 2})
# версии библиотек и используемый парсер JSON
print('pandas', pd.__version__, '| numpy', np.__version__, '| парсер JSON:', _json_loads.__module__)

## 1. Вспомогательные функции
Нормализация ключей (ИИН с ведущими нулями, коды договоров), справочник правил ПКБ (Первое кредитное бюро), таймеры.

In [ ]:
class Timer:
    # Таймер шага: печатает название шага при входе и время выполнения при выходе
    def __init__(self, msg):
        self.msg = msg
    def __enter__(self):
        self.t0 = time.time()
        print(f'> {self.msg} ...', flush=True)
        return self
    def __exit__(self, *exc):
        print(f'  готово за {time.time() - self.t0:.1f} с', flush=True)


# текстовые значения, которые считаются пропусками (пустыми ключами)
_NULLS = ['', 'None', 'none', 'nan', 'NaN', 'NAN', 'NONE', 'NULL', 'null', '<NA>', 'NaT']


def norm_iin(s):
    """ИИН/БИН в виде строки из 12 символов; восстанавливает ведущие нули, потерянные при преобразовании в int/float."""
    s = pd.Series(s, copy=False).astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    short_digits = s.str.fullmatch(r'\d{1,11}').fillna(False).astype(bool)
    s = s.where(~short_digits, s.str.zfill(12))
    return s.where(~s.isin(_NULLS), np.nan).astype(object)


def norm_code(s):
    """Код договора как строка в верхнем регистре без пробелов по краям (одно правило для наших договоров и для contract_code из ПКБ)."""
    s = pd.Series(s, copy=False).astype(str).str.strip().str.upper()
    return s.where(~s.isin([x.upper() for x in _NULLS]), np.nan).astype(object)


_PD2 = int(pd.__version__.split('.')[0]) >= 2   # True, если версия pandas 2 или выше


def to_dt(s):
    """Надёжный разбор дат. pandas >= 2 определяет ОДИН формат по первому значению и превращает значения в другом формате в NaT,
    поэтому ISO8601 запрашивается явно (охватывает 'YYYY-MM-DD', дату со временем, с долями секунды, объекты date/datetime)."""
    s = pd.Series(s, copy=False) if not isinstance(s, pd.Series) else s
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    if not _PD2:
        return pd.to_datetime(s, errors='coerce')
    out = pd.to_datetime(s, errors='coerce', format='ISO8601')
    lost = out.isna() & s.notna()
    if lost.mean() > 0.01:                       # формат не ISO: разбираем каждое значение отдельно
        out = pd.to_datetime(s, errors='coerce', format='mixed')
    return out


def days(n):
    # n дней в виде pd.Timedelta
    return pd.Timedelta(days=int(n))


# Правила мониторинга ПКБ: (фрагмент rule_name в нижнем регистре, код, семейство, описание).
# Порядок важен: срабатывает первый фрагмент, найденный в названии правила.
# Фрагменты на русском должны совпадать с реальными названиями правил ПКБ - их не менять.
# Семейства: credit_hunger - кредитный голод (заявки, запросы, новые займы), relief - облегчение условий
# (пролонгация, реструктуризация), delinquency - просрочка, positive - позитивные события, status - особые статусы
# клиента, technical - технические, debt_burden - долговая нагрузка, bank_rule - правила, настроенные банком (сравнивают нас с другими кредиторами),
# other - прочие.
RULES = [
    ('кредитной заявки', 'app_online', 'credit_hunger', 'Онлайн-заявка на кредит'),
    ('пролонгация', 'prolong', 'relief', 'Пролонгация займа'),
    ('любой поставщик', 'lookup_other', 'credit_hunger', 'Другой поставщик (не ПКБ) запросил данные по клиенту'),
    ('сумма просрочки', 'ovd_sum_chg', 'delinquency', 'Общая сумма просрочки выросла/снизилась'),
    ('с целью рефинанс', 'new_loan_refin', 'credit_hunger', 'Новый заём с целью рефинансирования'),
    ('появление нового займа', 'new_loan', 'credit_hunger', 'Появился новый заём'),
    ('урегулирование', 'settlement', 'positive', 'Урегулирование задолженности'),
    ('лудоман', 'st_gambler', 'status', 'Статус «Лудоман» (самозапрет на азартные игры) установлен или снят'),
    ('отписка', 'unsubscribe', 'technical', 'Отписка: нет действующих договоров (техническое событие)'),
    ('призывник', 'st_recruit', 'status', 'Статус «Призывник» (призыв на воинскую службу) установлен или снят'),
    ('залогу', 'collateral_chg', 'other', 'Изменение по залогу'),
    ('запросов на кредитный отчет', 'report_req', 'credit_hunger', 'Запрос кредитного отчёта'),
    ('новый контракт', 'new_contract', 'credit_hunger', 'Появился новый договор'),
    ('реструктур', 'restruct', 'relief', 'Заём реструктурирован / рефинансирован / пересмотрен'),
    ('стоп-кредит', 'st_stopcredit', 'status', 'Статус «Стоп-Кредит» (добровольный запрет на займы) установлен или снят'),
    ('статуса по кредиту изменился', 'debt_terminated', 'delinquency', 'Обязательство прекращено без полного погашения'),
    ('частичная просрочка', 'partial_ovd', 'bank_rule', 'Частичная просрочка (клиент просрочил платёж у нас, но платит другим кредиторам)'),
    ('банкрот', 'st_bankrupt', 'status', 'Статус «Банкротство» установлен или снят'),
    ('переуступка', 'loan_assigned', 'delinquency', 'Заём закрыт переуступкой (продан)'),
    ('закрытие займа', 'loan_closed', 'positive', 'Заём закрыт (погашен)'),
    ('сделать сравнение', 'refin_out_80', 'bank_rule', 'Кредит в другом банке, когда у нас до погашения оставалось ≥80% основного долга (по формулировке правила банка в ПКБ; уточнить у владельца правила)'),
    ('ежемесячного платежа', 'instalment_chg', 'debt_burden', 'Изменился общий ежемесячный платёж'),
    ('дней просрочки', 'dpd_chg', 'delinquency', 'Изменилось количество дней просрочки'),
    ('основного долга', 'debt_chg', 'debt_burden', 'Общий основной долг вырос/снизился'),
]
FAMILY = {code: fam for _, code, fam, _ in RULES}          # код правила -> семейство
RULE_DESC = {code: desc for _, code, _, desc in RULES}     # код правила -> описание
STATUS_RULES = [code for _, code, fam, _ in RULES if fam == 'status']   # коды статусных правил


def map_rule_names(names):
    """{rule_name: код}. Неизвестные названия правил получают автоматический код, чтобы по ним всё равно строились признаки-счётчики."""
    out = {}
    for nm in names:
        low = str(nm).lower()
        for frag, code, _, _ in RULES:
            if frag in low:
                out[nm] = code
                break
        else:
            out[nm] = 'other_' + re.sub(r'\W+', '_', low).strip('_')[:30]
            FAMILY.setdefault(out[nm], 'unknown')
            RULE_DESC.setdefault(out[nm], str(nm)[:80])
    return out


def iv_strength(iv):
    # Сила признака по IV: < 0.02 бесполезный, 0.02-0.1 слабый, 0.1-0.3 средний, 0.3-0.5 сильный,
    # >= 0.5 подозрительно высокая IV: проверьте утечку данных (leakage).
    # Возвращаемые значения на английском - это коды категорий столбца strength в таблицах.
    if not np.isfinite(iv):
        return ''
    return ('not useful' if iv < 0.02 else 'weak' if iv < 0.1 else 'medium' if iv < 0.3
            else 'strong' if iv < 0.5 else 'suspicious: check leakage')

# Вспомогательные функции для JSON (используются в предварительных проверках и в парсере событий в разделе 7)
NUM_KEYS = ('overdue_sum', 'debt_sum', 'instalment', 'overdue_instalment_count')   # числовые поля JSON в порядке приоритета
NAN = float('nan')
_EMPTY_NEW = (None, NAN, NAN, NAN, NAN, NAN, NAN, NAN)   # результат разбора пустого/некорректного new_value
_EMPTY_OLD = (NAN, NAN)                                  # результат разбора пустого/некорректного old_value


def _num(v):
    # значение из JSON -> число: bool и 'true'/'false' -> 1/0, строки с пробелами и десятичной запятой -> float, иначе NaN
    if v is None:
        return NAN
    if isinstance(v, bool):
        return 1.0 if v else 0.0
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        t = v.strip().lower()
        if t in ('true', 'false'):
            return 1.0 if t == 'true' else 0.0
        try:
            return float(t.replace(' ', '').replace(',', '.'))
        except ValueError:
            return NAN
    return NAN


def _id(v):
    # числовой идентификатор: из словаря берётся поле 'id', иначе само значение
    return _num(v.get('id')) if isinstance(v, dict) else _num(v)


def _loads(s):
    # текст JSON -> словарь; None, если текст пустой, некорректный или это не объект JSON
    if isinstance(s, dict):
        return s
    if not isinstance(s, (str, bytes)) or len(s) < 2:
        return None
    try:
        d = _json_loads(s)
    except Exception:
        return None
    return d if isinstance(d, dict) else None


def _flag(d):
    # первый флаг вида is_* / has_* (или stopcredit) как 1/0; NaN, если флага нет
    for k, v in d.items():
        if k.startswith(('is_', 'has_')) or k == 'stopcredit':
            x = _num(v)
            if x == x:                           # x == x ложно только для NaN
                return x
    return NAN


def _first_num(d):
    # первое заполненное числовое поле из NUM_KEYS
    for k in NUM_KEYS:
        v = d.get(k)
        if v is not None:
            return _num(v)
    return NAN


def _parse_new(s):
    # разбор new_value: (код договора, тип кредитора, первое числовое поле, флаг, сумма займа,
    #                    задолженность, остаток долга, длительность пролонгации в днях)
    d = _loads(s)
    if d is None:
        return _EMPTY_NEW
    code = d.get('contract_code')
    code = str(code).strip().upper() if code not in (None, '') else None
    pdays = NAN
    a, b = d.get('prolongation_start_date'), d.get('prolongation_end_date')
    if a and b:
        try:
            pdays = float((date.fromisoformat(str(b)[:10]) - date.fromisoformat(str(a)[:10])).days)
        except ValueError:
            pass
    return (code, _id(d.get('creditor_type')), _first_num(d), _flag(d), _num(d.get('summa_zaima')),
            _num(d.get('zadolzhenost')), _num(d.get('outstanding_debt')), pdays)


def _parse_old(s):
    # разбор old_value: (первое числовое поле, флаг)
    d = _loads(s)
    if d is None:
        return _EMPTY_OLD
    return (_first_num(d), _flag(d))

## 2. Данные
`DEMO_MODE = True` генерирует синтетический портфель с теми же столбцами, типами данных (dtypes) и структурой JSON, что и
`ovrd_contr`, `hes` и `contracts` (около 60 тыс. клиентов, 9 недель мониторинга). Генератор закладывает в данные заранее
известный сигнал, чтобы можно было проверить каждый шаг анализа.

При `DEMO_MODE = False` ноутбук использует объекты, уже загруженные в память. Альтернативные способы загрузки показаны в комментариях в ячейке.

In [ ]:
CT = {1: {'id': '1', 'name': 'Банк второго уровня'}, 2: {'id': '2', 'name': 'Микрофинансовая организация'}}
SYN_RULE_NAMES = {   # тексты rule_name, которые использует генератор (реальные названия правил сопоставляются по фрагментам, см. RULES)
    'app_online': 'Мониторинг кредитной заявки (онлайн заявки)',
    'prolong': 'Пролонгация по займу',
    'lookup_other': 'Любой поставщик (кроме ПКБ) искал информацию по данному субъекту',
    'ovd_sum_chg': 'Совокупная сумма просрочки увеличилась/уменьшилась',
    'new_loan': 'Мониторинг появление нового займа',
    'settlement': 'Мониторинг факта урегулирование задолженности',
    'st_gambler': 'Мониторинг статуса «Лудоман»',
    'unsubscribe': 'Отписка субъектов без активных контрактов',
    'st_recruit': 'Мониторинг статуса «Призывник»',
    'collateral_chg': 'Изменения по залогу займа',
    'report_req': 'Мониторинг запросов на кредитный отчет',
    'new_contract': 'У субъекта появился новый контракт',
    'restruct': 'Мониторинг изменения по кредиту - (Реструктурирован/Рефинансирован/Пересмотрен)',
    'new_loan_refin': 'Появление нового займа с целью Рефинансирование, где получатель единственный кредитор',
    'st_stopcredit': 'Мониторинг статуса «Стоп-Кредит»',
    'debt_terminated': 'Мониторинг статуса по кредиту изменился – Ссудная задолженность прекращена без полного исполнения обязательства',
    'partial_ovd': 'Частичная просрочка по кредиторам (заемщик допустил просрочку в организации получателя, но продолжает платить в других)',
    'st_bankrupt': 'Мониторинг статуса «Банкротства»',
    'loan_closed': 'Мониторинг закрытие займа (погашение)',
    'loan_assigned': 'Мониторинг закрытие займа – переуступка',
    'refin_out_80': 'Сделать сравнение, у кого до погашения не менее 80% от ОД займа – появился кредит в другом банке',
    'instalment_chg': 'Совокупная сумма ежемесячного платежа изменилась',
    'dpd_chg': 'Мониторинг изменения дней просрочки займа',
    'debt_chg': 'Совокупная сумма основного долга увеличилась/уменьшилась',
}


def _j(d):
    return json.dumps(d, ensure_ascii=False)


def _ext_codes(r, n):
    """Коды договоров других кредиторов в нескольких форматах."""
    k = r.integers(0, 5, n)
    x = r.integers(10**6, 10**9, n)
    fmt = ['KZ{:010d}', 'CO2026{:012d}', 'Z{:08d}A0219-3', 'КАРТ-{:010d}', '{:d}']
    return [fmt[a].format(b) for a, b in zip(k, x)]


def _payloads(code, sv, r, codes=None, dpd_old=None, dpd_new=None):
    """JSON-строки old_value / new_value для n событий одного правила (структура JSON скопирована из реальной таблицы)."""
    n = len(sv)
    mfo = r.random(n) < 1 / (1 + np.exp(-(-0.9 + 0.9 * sv)))
    ct = [CT[2] if m else CT[1] for m in mfo]
    cc = codes if codes is not None else _ext_codes(r, n)
    amt = np.round(np.exp(r.normal(12.2 + 0.25 * sv, 0.8)), -3)
    none = [None] * n
    if code == 'app_online':
        new = [_j({'object': {'id': None, 'name': None}, 'purpose': {'id': '04', 'name': 'Потребительские цели'},
                   'summa_zaima': str(int(a)), 'credit_app_id': int(i), 'creditor_type': c})
               for a, i, c in zip(amt, r.integers(10**8, 10**9, n), ct)]
        return none, new
    if code in ('prolong', 'new_loan_refin', 'loan_closed'):
        debt = np.round(amt * r.uniform(0.2, 0.95, n), 0)
        new = []
        for a, d_, c, k, p in zip(amt, debt, ct, cc, r.integers(0, 10, n)):
            d = {'object': {'id': '10', 'name': 'Прочие'},
                 'purpose': {'id': '16', 'name': 'Рефинансирование'} if code == 'new_loan_refin' else {'id': '09', 'name': 'Прочие'},
                 'summa_zaima': str(int(a)) if code != 'loan_closed' else float(a), 'contract_code': k, 'creditor_type': c}
            if code != 'loan_closed':
                d['zadolzhenost'] = str(int(d_))
            if code == 'prolong':
                d['prolongation_start_date'] = '2026-09-08T00:00:00+05:00'
                d['prolongation_end_date'] = f'2026-09-{14 + int(p):02d}T00:00:00+05:00'
            if code == 'new_loan_refin':
                d.update({'platezh': None, 'valuta_platezha': None})
            new.append(_j(d))
        return none, new
    if code == 'lookup_other':
        return none, [_j({'report': {'id': '6', 'name': 'Кредитный отчет – расширенный физическое лицо'}})] * n
    if code == 'report_req':
        return none, [_j({'creditor_type': c, 'credit_report_id': str(i)}) for c, i in zip(ct, r.integers(10**8, 10**9, n))]
    if code == 'ovd_sum_chg':
        o = np.round(np.exp(r.normal(10 + 0.4 * sv, 1.0)), 2) * (r.random(n) < 0.7)
        up = r.random(n) < 1 / (1 + np.exp(-0.9 * sv))
        nv = np.where(up, o + np.round(np.exp(r.normal(9.5, 0.8, n)), 2), o * r.uniform(0, 0.6, n))
        old = [_j({'overdue_sum': float(a), 'contract_code': k, 'creditor_type': c}) for a, k, c in zip(o, cc, ct)]
        new = [_j({'overdue_sum': float(round(b, 2)), 'contract_code': k, 'creditor_type': c}) for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code in ('new_loan', 'restruct', 'collateral_chg', 'debt_terminated', 'loan_assigned'):
        return none, [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': None}) for k, c in zip(cc, ct)]
    if code == 'new_contract':
        old = [_j({'end_date': None, 'start_date': None, 'subject_role': {'id': None, 'name': None},
                   'contract_code': None, 'financial_institution': {'id': 0, 'name': None}})] * n
        new = [_j({'end_date': '2028-09-01', 'start_date': '2026-09-01', 'subject_role': {'id': 1, 'name': 'Заемщик'},
                   'contract_code': k, 'financial_institution': {'id': int(f), 'name': 'ФИ'}})
               for k, f in zip(cc, r.integers(1, 400, n))]
        return old, new
    if code == 'debt_chg':
        o = np.round(np.exp(r.normal(13, 1.0, n)), 2)
        up = r.random(n) < 1 / (1 + np.exp(-(-1.6 + 0.6 * sv)))
        nv = np.where(up, o * r.uniform(1.0, 1.3, n), o * r.uniform(0.9, 1.0, n))
        old = [_j({'debt_sum': float(a), 'contract_code': k, 'creditor_type': c}) for a, k, c in zip(o, cc, ct)]
        new = [_j({'debt_sum': float(round(b, 2)), 'contract_code': k, 'creditor_type': c}) for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code == 'instalment_chg':
        o = np.round(np.exp(r.normal(11, 0.8, n)), 2)
        nv = o * np.where(r.random(n) < 1 / (1 + np.exp(-0.5 * sv)), r.uniform(1.0, 1.5, n), r.uniform(0.6, 1.0, n))
        old = [_j({'instalment': str(a)}) for a in o]
        new = [_j({'instalment': str(round(b, 2)), 'contract_code': k, 'creditor_type': c, 'outstanding_debt': str(round(b * 12, 2))})
               for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code == 'dpd_chg':
        if dpd_old is None:
            dpd_old = np.floor(r.exponential(8 * np.exp(0.5 * sv))).astype(int)
            worse = r.random(n) < 1 / (1 + np.exp(-(0.2 + 0.9 * sv)))
            dpd_new = np.where(worse, dpd_old + r.integers(1, 31, n), 0)
        old = [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': int(a)}) for k, c, a in zip(cc, ct, dpd_old)]
        new = [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': int(b)}) for k, c, b in zip(cc, ct, dpd_new)]
        return old, new
    flags = {'st_gambler': 'is_gambler', 'st_recruit': 'is_recruit', 'st_stopcredit': 'stopcredit', 'st_bankrupt': 'is_bankrupt'}
    if code in flags:
        return [_j({flags[code]: 0})] * n, [_j({flags[code]: 1})] * n
    if code == 'settlement':
        return none, [_j({'is_settlement': 1})] * n
    if code == 'partial_ovd':
        return none, [_j({'has_half_overdue_bank_and_not_overdue_non_bank': True})] * n
    if code == 'refin_out_80':
        return none, [_j({'has_refinance_more_80_percent_ood': True})] * n
    if code == 'unsubscribe':
        return ([_j({'reason': None, 'status': 'Active', 'group_id': 66, 'removed_at': None})] * n,
                [_j({'reason': 'NO_ACTIVE_CONTRACTS', 'status': 'Removed', 'group_id': 66,
                     'removed_at': '2026-10-19T23:08:40.846181+05:00'})] * n)
    return none, none


def make_synthetic_data(n_clients=60_000, seed=SEED):
    """Синтетические ovrd_contr / hes / contracts со схемой, как в промышленных данных, и заранее заложенным сигналом."""
    r = np.random.default_rng(seed)
    D = np.timedelta64(1, 'D')
    issue_from, issue_to = np.datetime64('2025-01-01'), np.datetime64('2026-10-15')
    data_end = np.datetime64('2026-11-18')
    hes_from, hes_to = np.datetime64('2026-09-07'), np.datetime64('2026-11-18')   # события создаются по 2026-11-17 включительно

    # клиенты: z — внутренний риск, s — финансовый стресс, видимый у других кредиторов (коррелирует с z)
    z = r.normal(size=n_clients)
    s = 0.5 * z + np.sqrt(0.75) * r.normal(size=n_clients)
    iin = np.array([f'{a:02d}{b:010d}' for a, b in zip(r.integers(60, 106, n_clients) % 100,
                                                       r.integers(0, 10**10, n_clients))])
    # договоры
    n_c = 1 + r.poisson(0.45, n_clients)
    ci = np.repeat(np.arange(n_clients), n_c)
    N = len(ci)
    open_d = issue_from + r.integers(0, int((issue_to - issue_from) / D) + 1, N) * D
    prod = r.choice(np.array(['16007', '16062', '16167', '16168']), N, p=[.35, .15, .25, .25])
    term_m = np.where(prod == '16167', r.choice([6, 12, 18], N),
             np.where(prod == '16168', r.choice([12, 24], N),
             np.where(prod == '16062', r.choice([12, 18, 24, 36], N), r.choice([24, 36, 48, 60], N))))
    end_d = open_d + np.round(term_m * 30.44).astype(int) * D
    mu = np.where(prod == '16167', 10.5, np.where(prod == '16168', 13.0, np.where(prod == '16062', 13.5, 14.5)))
    amount = np.round(np.exp(r.normal(mu, 0.6)), -2)
    early = r.random(N) < 0.2
    life = ((end_d - open_d) / D).astype(int)
    close_d = end_d.copy()
    close_d[early] = open_d[early] + np.maximum(30, (life[early] * r.uniform(0.15, 0.9, early.sum())).astype(int)) * D
    status = np.where(close_d <= data_end, 'Погашен', 'Актуален')
    pref = r.choice(np.array(['3199191', '1199221', '1340260', '3277770', '1677770', '2477770']), N)
    contract_number = np.array([f'{p}{i:09d}' for p, i in zip(pref, r.permutation(N) + 100_000)])
    pr = np.where(np.isin(prod, ['16007', '16062']), 'EKP', np.where(prod == '16167', 'CCC', 'CPA'))
    process_ref = np.array([f'{p}{i:09d}' if p == 'EKP' else f'{p}{i:012d}'
                            for p, i in zip(pr, r.permutation(N) + 10_000_000)])
    merchants = np.array([f'{x:012d}' for x in r.integers(10**11, 10**12, 300)])
    partner = np.where(prod == '16167', 'marketplace', np.where(prod == '16168', r.choice(merchants, N), None)).astype(object)
    prod_eff = pd.Series(prod).map({'16007': 0.0, '16062': 0.2, '16167': 0.35, '16168': 0.15}).values

    # эпизоды просрочки
    first_due = open_d + 30 * D
    active_days = np.maximum(0, ((np.minimum(close_d, data_end) - first_due) / D).astype(int))
    n_ep = r.poisson(0.0030 * np.exp(0.8 * z[ci] + prod_eff) * active_days)
    e_c = np.repeat(np.arange(N), n_ep)
    e_start = first_due[e_c] + (r.random(len(e_c)) * active_days[e_c]).astype(int) * D
    zc, sc = z[ci[e_c]], s[ci[e_c]]
    roll = r.random(len(e_c)) < 1 / (1 + np.exp(-(-2.4 + 1.0 * zc + 0.6 * sc + prod_eff[e_c])))
    e_dpd = np.where(roll, 31 + r.exponential(45, len(e_c)).astype(int),
                     1 + np.floor(29.999 * r.beta(0.7, np.clip(2.6 - 0.5 * sc, 1.2, 4.0))).astype(int))
    E = pd.DataFrame({'c': e_c, 'start': e_start, 'dpd': e_dpd, 'roll': roll}).sort_values(['c', 'start'])
    E['end'] = E['start'] + pd.to_timedelta(E['dpd'] - 1, unit='D')
    E['cm'] = E.groupby('c')['end'].cummax()
    prev = E.groupby('c')['cm'].shift()
    E = E[prev.isna() | (E['start'] > prev + pd.Timedelta(days=1))]
    E = E[E['start'] <= pd.Timestamp(data_end)].reset_index(drop=True)
    E['dpd_obs'] = np.minimum(E['dpd'], (pd.Timestamp(data_end) - E['start']).dt.days + 1)

    # строки дней просрочки (DPD): последний день эпизода, концы месяцев и ~35% остальных дней (проверка разреженной выгрузки)
    L = E['dpd_obs'].values.astype(int)
    rep = np.repeat(np.arange(len(E)), L)
    off = np.arange(L.sum()) - np.repeat(np.cumsum(L) - L, L)
    act = E['start'].values.astype('datetime64[D]')[rep] + off * D
    is_last = off == (L[rep] - 1)
    is_month_end = (act + D).astype('datetime64[M]') != act.astype('datetime64[M]')
    keep = is_last | is_month_end | (r.random(len(off)) < 0.35)
    cidx = E['c'].values[rep][keep]
    act_k = act[keep]
    ns = lambda a: a.astype('datetime64[ns]')
    ovrd = pd.DataFrame({
        'rep_month': np.char.add(act_k.astype('datetime64[M]').astype(str), '-01').astype(object),
        'act_date': act_k.astype(str).astype(object),
        'contract_number': contract_number[cidx].astype(object),
        'ovd_days_cnt': (off[keep] + 1).astype(float),
        'process_ref': process_ref[cidx].astype(object),
        'iin': iin[ci[cidx]].astype(object),
        'open_date': ns(open_d[cidx]), 'close_date': ns(close_d[cidx]), 'end_date': ns(end_d[cidx]),
        'status_name': status[cidx].astype(object),
        'contract_amount': amount[cidx].astype(float),
        'partner_name': partner[cidx],
        'product_id': prod[cidx].astype(object)})
    dup = ovrd.sample(frac=0.01, random_state=seed).assign(status_name='Погашен')   # дубликаты от слияния, как в реальных данных
    ovrd = pd.concat([ovrd, dup], ignore_index=True).sort_values(['rep_month', 'act_date'], kind='mergesort').reset_index(drop=True)

    contracts = pd.DataFrame({
        'contract_number': contract_number.astype(object), 'process_ref': process_ref.astype(object),
        'iin': iin[ci].astype(object), 'open_date': ns(open_d), 'close_date': ns(close_d), 'end_date': ns(end_d),
        'status_name': status.astype(object), 'contract_amount': amount, 'partner_name': partner,
        'product_id': prod.astype(object)})

    # события мониторинга ПКБ
    win_days = float((hes_to - hes_from) / D)
    act_client = np.zeros(n_clients, bool)
    act_client[ci[(open_d < hes_to) & (close_d > hes_from)]] = True
    rates = {   # события по другим кредиторам: (дневная интенсивность, влияние стресса s)
        'app_online': (0.010, 0.7), 'lookup_other': (0.004, 0.5), 'report_req': (0.007, 0.6),
        'new_loan': (0.003, 0.5), 'new_contract': (0.003, 0.4), 'new_loan_refin': (0.0008, 0.6),
        'ovd_sum_chg': (0.004, 1.0), 'dpd_chg': (0.006, 1.1), 'debt_chg': (0.030, 0.1),
        'instalment_chg': (0.010, 0.2), 'collateral_chg': (0.001, 0.0), 'loan_closed': (0.005, -0.3),
        'settlement': (0.0008, 0.3), 'loan_assigned': (0.0003, 1.3), 'debt_terminated': (0.0002, 1.3),
        'prolong': (0.002, 0.8), 'restruct': (0.0005, 0.8), 'st_gambler': (0.0002, 0.9),
        'st_recruit': (0.0002, 0.0), 'st_stopcredit': (0.0002, 0.5), 'st_bankrupt': (0.00005, 1.5),
        'refin_out_80': (0.0004, -0.5), 'unsubscribe': (0.0003, 0.0)}
    parts = []

    def add(code, cl, t, codes=None, dpd_old=None, dpd_new=None):
        if len(cl) == 0:
            return
        old, new = _payloads(code, s[cl], r, codes, dpd_old, dpd_new)
        parts.append(pd.DataFrame({'cl': cl, 'code': code, 't': t, 'old': old, 'new': new}))

    t_unif = lambda n: pd.Timestamp(hes_from) + pd.to_timedelta(r.random(n) * win_days * 86400, unit='s')
    for code, (rate, beta) in rates.items():
        lam = rate * np.exp(beta * s - beta ** 2 / 2) * win_days * np.where(act_client, 1.0, 0.15)
        cl = np.repeat(np.arange(n_clients), r.poisson(lam))
        add(code, cl, t_unif(len(cl)))

    # события по нашим договорам в период просрочки и всплески стресса перед началом эпизода просрочки
    E['cl'] = ci[E['c'].values]
    hes_from_ts, hes_to_ts = pd.Timestamp(hes_from), pd.Timestamp(hes_to)
    Ew = E[(E['start'] > hes_from_ts - pd.Timedelta(days=60)) & (E['start'] < hes_to_ts)]
    own_cl, own_t, own_code, d_old, d_new = [], [], [], [], []
    for row in Ew.itertuples(index=False):
        k = contract_number[row.c] if r.random() < 0.6 else process_ref[row.c]
        for step in range(1, int(row.dpd_obs), 7):
            t = row.start + pd.Timedelta(days=step + 1, hours=float(r.uniform(0, 20)))
            if hes_from_ts <= t < hes_to_ts:
                own_cl.append(row.cl); own_t.append(t); own_code.append(k)
                d_old.append(max(step - 7, 0)); d_new.append(step)
    add('dpd_chg', np.array(own_cl, int), pd.to_datetime(own_t), own_code, np.array(d_old), np.array(d_new))
    m = (Ew['start'] + pd.Timedelta(days=2) >= hes_from_ts) & (Ew['start'] + pd.Timedelta(days=2) < hes_to_ts)
    po = Ew[m & (r.random(len(Ew)) < 0.35)]
    add('partial_ovd', po['cl'].values, po['start'] + pd.Timedelta(days=2, hours=9))
    Eb = E[(E['start'] >= hes_from_ts + pd.Timedelta(days=5)) & (E['start'] < hes_to_ts)]
    nb = r.poisson(np.where(Eb['roll'], 0.7, 0.35))
    bcl = np.repeat(Eb['cl'].values, nb)
    bst = np.repeat(Eb['start'].values, nb)
    bcode = r.choice(['app_online', 'report_req', 'dpd_chg', 'ovd_sum_chg', 'new_loan', 'lookup_other'],
                     len(bcl), p=[.3, .25, .15, .1, .1, .1])
    bt = pd.to_datetime(bst) - pd.to_timedelta(r.uniform(1, 21, len(bcl)) * 86400, unit='s')
    for code in np.unique(bcode):
        mm = bcode == code
        add(code, bcl[mm], bt[mm])

    ev = pd.concat(parts, ignore_index=True)
    ev = ev[(ev['t'] >= hes_from_ts) & (ev['t'] < hes_to_ts)].reset_index(drop=True)
    n = len(ev)
    created = pd.to_datetime(ev['t'].values)
    dv = (created.normalize() + pd.to_timedelta(r.integers(0, 2, n), unit='D'))
    hes = pd.DataFrame({
        'date_value': dv.strftime('%Y-%m-%d').astype(object),
        'date_change': dv + pd.Timedelta(days=1, hours=14, minutes=13, seconds=47),
        'internal_event_id': [str(uuid.UUID(bytes=r.bytes(16))) for _ in range(n)],
        'bin_iin': iin[ev['cl'].values].astype(object),
        'rule_name': ev['code'].map(SYN_RULE_NAMES).values.astype(object),
        'old_value': ev['old'].values, 'new_value': ev['new'].values,
        'event_date': created - pd.to_timedelta(r.uniform(0, 1.5, n) * 86400, unit='s'),
        'created_date': created})
    hes = hes.sort_values(['date_value', 'internal_event_id']).reset_index(drop=True)
    return ovrd, hes, contracts


if DEMO_MODE:
    with Timer('Генерация синтетических демо-данных'):
        ovrd_contr, hes, contracts = make_synthetic_data()
else:
    # Ваши датафреймы берутся из этого ядра по имени (ничего не копируется).
    # Если они загружены в другом ноутбуке, вставьте эти ячейки туда либо сохраните их и прочитайте здесь, например:
    #   ovrd_contr = pd.read_pickle('ovrd_contr.pkl'); hes = pd.read_pickle('hes.pkl')
    # hes может быть и итератором по частям (чанкам), например pd.read_sql(sql, conn, chunksize=2_000_000);
    # тогда он фильтруется по частям в разделе 7 и никогда не хранится в памяти целиком (HES_START задайте вручную).
    _g = globals()
    _missing = [v for v in (OVRD_VAR, HES_VAR) if v not in _g]
    if _missing:
        raise NameError(f'Датафреймов {_missing} нет в этом ядре. Сначала загрузите их (или измените OVRD_VAR / HES_VAR), '
                        'либо задайте DEMO_MODE = True, чтобы запустить демо на синтетических данных.')
    ovrd_contr = _g[OVRD_VAR]
    hes = _g[HES_VAR]
    contracts = _g.get(CONTRACTS_VAR)
    if not isinstance(contracts, pd.DataFrame):
        contracts = None

print('ovrd_contr:', ovrd_contr.shape, '| contracts:', None if contracts is None else contracts.shape,
      '| hes:', hes.shape if isinstance(hes, pd.DataFrame) else 'итератор по частям')
if DEMO_MODE:
    display(ovrd_contr.head(3))
    display(hes.head(3))

## 2b. Предварительная проверка (preflight check) входных датафреймов
Быстрые проверки на выборках строк до начала тяжёлых вычислений: наличие обязательных колонок, распознавание дат и дней
просрочки (DPD), формат ИИН (IIN), пересечение клиентов между `hes` и `ovrd_contr`, доля успешно разобранных JSON по каждому
правилу, распознанные названия правил, а также совпадают ли коды договоров в ПКБ (Первое кредитное бюро, PKB) с номерами
наших договоров. Статус «ОШИБКА» (`FAIL`) останавливает ноутбук (`STOP_ON_PREFLIGHT_FAIL`), «ПРЕДУПРЕЖДЕНИЕ» (`WARN`) носит только
информационный характер.

In [ ]:
def _sample_rows(df, n, seed=SEED):
    if len(df) <= n:
        return df
    pos = np.unique(np.random.default_rng(seed).integers(0, len(df), size=n))   # быстро даже на 85 млн строк
    return df.iloc[pos]


def preflight(ovrd, hes, contracts):
    rows = []

    def add(obj, check, status, detail):
        rows.append({'object': obj, 'check': check, 'status': status, 'detail': detail})

    def cols(df, obj, need, opt):
        miss = [c for c in need if c not in df.columns]
        add(obj, 'обязательные колонки', 'FAIL' if miss else 'OK', f'отсутствуют: {miss}' if miss else ', '.join(need))
        mo = [c for c in opt if c not in df.columns]
        if mo:
            add(obj, 'необязательные колонки', 'WARN', f'отсутствуют: {mo} (признаки на их основе будут пустыми)')
        return not miss

    if not isinstance(ovrd, pd.DataFrame):
        add('ovrd_contr', 'тип', 'FAIL', f'ожидался DataFrame, получен {type(ovrd).__name__}')
    elif cols(ovrd, 'ovrd_contr', [C_CONTRACT, C_IIN, C_ACT_DATE, C_DPD],
              [C_PROCESS_REF, C_OPEN, C_CLOSE, C_END, C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT]):
        s = _sample_rows(ovrd, DQ_SAMPLE)
        ad = to_dt(s[C_ACT_DATE])
        r = ad.notna().mean()
        add('ovrd_contr', f'{C_ACT_DATE} распознаётся как дата', 'OK' if r > 0.99 else 'WARN' if r > 0.9 else 'FAIL',
            f'{r:.2%} распознано, диапазон {ad.min()} .. {ad.max()}')
        d = pd.to_numeric(s[C_DPD], errors='coerce')
        add('ovrd_contr', f'{C_DPD} числовое и > 0', 'OK' if (d > 0).mean() > 0.95 else 'WARN',
            f'{(d > 0).mean():.2%} > 0, {d.isna().mean():.2%} пропусков, максимум {d.max():.0f}')
        ii = norm_iin(s[C_IIN])
        r = ii.str.fullmatch(r'\d{12}').fillna(False).astype(bool).mean()
        add('ovrd_contr', f'{C_IIN} = 12 цифр', 'OK' if r > 0.99 else 'WARN' if r > 0.8 else 'FAIL', f'{r:.2%}')
        cn = norm_code(s[C_CONTRACT])
        add('ovrd_contr', f'{C_CONTRACT} заполнен', 'OK' if cn.notna().mean() > 0.99 else 'WARN', f'{cn.notna().mean():.2%}')
    if contracts is not None:
        cols(contracts, 'contracts', [C_CONTRACT, C_IIN, C_OPEN], [C_PROCESS_REF, C_CLOSE, C_END, C_STATUS, C_AMOUNT,
                                                                  C_PARTNER, C_PRODUCT])
    else:
        add('contracts', 'наличие', 'WARN', f'`{CONTRACTS_VAR}` не найден: часть B (раннее предупреждение, EWS) пропускается')
    if isinstance(hes, pd.DataFrame):
        if cols(hes, 'hes', [H_IIN, H_RULE, H_NEW, EVENT_TIME_COL], [H_OLD, H_ID, H_EVENT_DATE]):
            s = _sample_rows(hes, min(DQ_SAMPLE, 300_000))
            t = s[EVENT_TIME_COL] if pd.api.types.is_datetime64_any_dtype(s[EVENT_TIME_COL]) else to_dt(s[EVENT_TIME_COL])
            r = t.notna().mean()
            add('hes', f'{EVENT_TIME_COL} распознаётся как дата', 'OK' if r > 0.99 else 'WARN' if r > 0.9 else 'FAIL',
                f'{r:.2%} распознано, диапазон {t.min()} .. {t.max()}')
            hi = norm_iin(s[H_IIN])
            r = hi.str.fullmatch(r'\d{12}').fillna(False).astype(bool).mean()
            add('hes', f'{H_IIN} = 12 цифр', 'OK' if r > 0.99 else 'WARN' if r > 0.8 else 'FAIL', f'{r:.2%}')
            if isinstance(ovrd, pd.DataFrame) and C_IIN in ovrd.columns:
                known = set(norm_iin(pd.Series(pd.unique(ovrd[C_IIN].values))).dropna())
                r = hi.isin(known).mean()
                add('hes', 'клиенты есть также в ovrd_contr', 'OK' if r > 0.05 else 'FAIL',
                    f'{r:.2%} событий выборки относятся к клиентам с просрочкой (0% означает, что ключи ИИН не совпадают)')
            rm = map_rule_names(pd.unique(s[H_RULE].dropna()))
            unk = [k for k, v in rm.items() if v.startswith('other_')]
            add('hes', 'названия правил распознаны', 'OK' if not unk else 'WARN',
                f'{len(rm) - len(unk)} из {len(rm)} распознано' + (f'; неизвестные (только подсчитываются): {unk[:5]}' if unk else ''))
            parsed = s[H_NEW].map(lambda v: _loads(v) is not None)
            by_rule = parsed.groupby(s[H_RULE].map(rm)).mean()
            bad = by_rule[by_rule < 0.5]
            add('hes', 'JSON в new_value разбирается', 'OK' if parsed.mean() > 0.9 and bad.empty else 'WARN',
                f'{parsed.mean():.2%} в целом' + (f'; низкая доля для {bad.round(2).to_dict()}' if len(bad) else ''))
            if isinstance(ovrd, pd.DataFrame) and C_CONTRACT in ovrd.columns:
                dpd_rows = s[s[H_RULE].map(rm) == 'dpd_chg']
                codes = pd.Series([_parse_new(v)[0] for v in dpd_rows[H_NEW].values], dtype=object).dropna()
                own = set(norm_code(pd.Series(pd.unique(ovrd[C_CONTRACT].values))).dropna())
                if C_PROCESS_REF in ovrd.columns:
                    own |= set(norm_code(pd.Series(pd.unique(ovrd[C_PROCESS_REF].values))).dropna())
                r = codes.isin(own).mean() if len(codes) else np.nan
                add('hes', 'события изменения DPD по нашим договорам', 'OK' if r == r and r > 0.01 else 'WARN',
                    f'{r:.2%} событий «Изм. DPD» совпадают с нашими номерами договоров '
                    '(около 0% — ПКБ передаёт другой номер, и наши договоры попадут в «другие кредиторы»)')
        add('hes', 'память (поверхностная оценка)', 'OK', f'{hes.memory_usage(index=False).sum() / 1e9:.1f} ГБ без учёта строковых данных')
    else:
        add('hes', 'тип', 'WARN', 'итератор по частям: проверки выполняются на лету; задайте HES_START вручную')
    res = pd.DataFrame(rows)
    display(res)
    return res


with Timer('Предварительные проверки'):
    preflight_result = preflight(ovrd_contr, hes, contracts)
if STOP_ON_PREFLIGHT_FAIL and (preflight_result['status'] == 'FAIL').any():
    raise ValueError('Предварительные проверки не пройдены (см. строки со статусом ОШИБКА (FAIL) выше). Исправьте названия колонок в первой ячейке '
                     'или сами данные, либо задайте STOP_ON_PREFLIGHT_FAIL = False, чтобы всё равно продолжить.')

## 3. Атрибуты договоров и множество кодов наших договоров
В данных ПКБ (Первое кредитное бюро, PKB) поле `contract_code` для наших собственных кредитов приходит либо как `contract_number`
(например, `3577770000000001`), либо как `process_ref` (например, `EKP000000001`), поэтому собираются оба кода. Дублирующиеся
строки договора (после объединения таблиц один и тот же договор указан один раз как «Актуален» и один раз как «Погашен») сводятся
к одной строке, при этом предпочтение отдаётся закрытому статусу. Если задан `OWN_CODES_VAR` (имя объекта
в кавычках, например `'all_contracts'`, или сам объект: таблица или список всех номеров договоров банка по этим клиентам,
любые продукты), эти номера добавляются к кодам наших договоров. Номера, прочитанные как числа (`3577770000000001.0`),
приводятся к целому виду; если номер длиннее 15 цифр и был прочитан как число, ноутбук предупредит, что последние
цифры могли исказиться (тогда загрузите колонку как текст).

In [ ]:
def channel_of(partner):
    p = pd.Series(partner, copy=False).fillna('').astype(str).str.strip()
    out = np.where(p.isin(_NULLS), 'direct', np.where(p.str.fullmatch(r'\d{10,12}').fillna(False), 'merchant', p))
    return pd.Series(out, index=p.index)


def build_contract_info(ovrd, contracts=None):
    cols = [C_CONTRACT, C_IIN, C_PROCESS_REF, C_OPEN, C_CLOSE, C_END, C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT]
    srcs = [x for x in (contracts, ovrd) if x is not None]
    frames = []
    for i, src in enumerate(srcs):
        c = [k for k in cols if k in src.columns]
        f = src[c].drop_duplicates()
        f = f.assign(_src=i)
        frames.append(f)
    ci = pd.concat(frames, ignore_index=True)
    ci['contract'] = norm_code(ci[C_CONTRACT]).values
    ci = ci.dropna(subset=['contract'])
    closed = ci[C_STATUS].astype(str).str.contains('Погаш|Закры|Спис', case=False, regex=True) if C_STATUS in ci else False
    ci['_closed'] = np.asarray(closed, dtype=int)
    ci = ci.sort_values(['contract', '_src', '_closed'], ascending=[True, True, False])
    ci = ci.groupby('contract', sort=False).first().reset_index()   # первое непустое значение по каждой колонке: сначала из contracts, затем из закрытых строк
    out = pd.DataFrame({'contract': ci['contract'].values})
    out['iin'] = norm_iin(ci[C_IIN]).values if C_IIN in ci else np.nan
    out['process_ref'] = norm_code(ci[C_PROCESS_REF]).values if C_PROCESS_REF in ci else np.nan
    for src, dst in ((C_OPEN, 'open_date'), (C_CLOSE, 'close_date'), (C_END, 'end_date')):
        out[dst] = to_dt(ci[src]).values if src in ci else pd.NaT
    out['amount'] = pd.to_numeric(ci[C_AMOUNT], errors='coerce').values if C_AMOUNT in ci else np.nan
    out['product'] = ci[C_PRODUCT].astype(str).values if C_PRODUCT in ci else 'NA'
    out['channel'] = channel_of(ci[C_PARTNER]).values if C_PARTNER in ci else 'NA'
    out['closed'] = ci['_closed'].values
    # True, если договор взят из полной таблицы `contracts`; о договорах, которые есть только в ovrd_contr, известно
    # лишь начиная с их первой просрочки (использовать их раньше означало бы утечку данных из будущего, leakage)
    out['from_contracts'] = (ci['_src'].values == 0) if contracts is not None else np.zeros(len(ci), dtype=bool)
    return out


from decimal import Decimal


def _own_code_value(v):
    """Одно значение номера договора до norm_code: целое число, прочитанное как float / Decimal (3577770000000001.0),
    -> '3577770000000001'; бесконечность и NaN -> None; остальное — как есть."""
    if isinstance(v, (float, np.floating, Decimal)):
        f = float(v)
        if not math.isfinite(f):
            return None
        return str(int(v)) if f == math.floor(f) else v
    return v


def own_codes_from(obj, name='OWN_CODES_VAR'):
    """Нормализованные коды договоров из объекта OWN_CODES_VAR (тем же правилом norm_code, что и наши договоры, и
    contract_code из ПКБ): DataFrame — колонки contract_number и/или process_ref (C_CONTRACT, C_PROCESS_REF);
    Series, Index, массив numpy, список, кортеж или множество — сами значения. Числа, прочитанные как float
    (в том числе в текстовой колонке и в виде текста '…0001.0'), приводятся к целому без '.0'. name — имя для сообщений.
    Возвращает (множество кодов, число значений-чисел ≥ 1e15, которые могли потерять последние цифры)."""
    if isinstance(obj, pd.DataFrame):
        cols = [c for c in (C_CONTRACT, C_PROCESS_REF) if c in obj.columns]
        if not cols:
            raise ValueError(f'В таблице {name} нет колонок {C_CONTRACT} / {C_PROCESS_REF}: '
                             f'есть только {list(obj.columns)[:20]}')
        parts = [obj[c] for c in cols]
    elif isinstance(obj, (pd.Series, pd.Index, np.ndarray, list, tuple, set, frozenset)):
        parts = [pd.Series(list(obj) if isinstance(obj, (set, frozenset)) else obj)]
    else:
        raise TypeError(f'{name}: ожидается DataFrame, Series или список кодов договоров, а не {type(obj).__name__}')
    out, n_long = set(), 0
    for p in parts:
        p = pd.Series(p, copy=False).dropna()
        if isinstance(p.dtype, pd.CategoricalDtype):
            p = p.astype(object)
        n_long += int(sum(isinstance(v, (float, np.floating, Decimal)) and math.isfinite(float(v)) and abs(float(v)) >= 1e15
                          for v in p.values))
        p = pd.Series([_own_code_value(v) for v in p.values], dtype=object).dropna()
        c = norm_code(p).dropna()
        c = c.str.replace(r'^(\d+)\.0+$', r'\1', regex=True)        # текст '3577770000000001.0' -> '3577770000000001'
        out |= set(c)
    return out, n_long


with Timer('Атрибуты договоров'):
    cinfo = build_contract_info(ovrd_contr, contracts)
    OWN_CODES = set(cinfo['contract'].dropna()) | set(cinfo['process_ref'].dropna())
print(f'{fmt_int(len(cinfo))} договоров, {fmt_int(len(OWN_CODES))} кодов наших договоров')
if OWN_CODES_VAR is not None and not (isinstance(OWN_CODES_VAR, str) and not OWN_CODES_VAR.strip()):
    # все договоры банка по этим клиентам (любые продукты): события ПКБ по ним тоже относятся к «нашим договорам».
    # OWN_CODES_VAR — имя объекта в ядре (в кавычках) или сам объект
    if isinstance(OWN_CODES_VAR, str):
        if OWN_CODES_VAR not in globals():
            raise NameError(f'Объекта {OWN_CODES_VAR!r} (OWN_CODES_VAR) нет в этом ядре: загрузите его или задайте OWN_CODES_VAR = None')
        _own_obj, _own_name = globals()[OWN_CODES_VAR], OWN_CODES_VAR
    else:
        _own_obj, _own_name = OWN_CODES_VAR, f'OWN_CODES_VAR ({type(OWN_CODES_VAR).__name__})'
    _extra, _n_long = own_codes_from(_own_obj, _own_name)
    _n0 = len(OWN_CODES)
    OWN_CODES |= _extra
    print(f'Из {_own_name}: {fmt_int(len(_extra))} {_plural_ru(len(_extra), ("код", "кода", "кодов"))} договоров, '
          f'из них новых {fmt_int(len(OWN_CODES) - _n0)} '
          f'(остальные уже были в ovrd_contr / contracts); всего кодов наших договоров: {fmt_int(len(OWN_CODES))}')
    _odd = sum(bool(re.search(r'\.|E\+', c)) for c in _extra)
    if _odd:
        print(f'  среди добавленных кодов {fmt_int(_odd)} содержат «.» или «E+» — проверьте, что номера не были прочитаны как числа')
    if _n_long:
        print(f'  ВНИМАНИЕ: {fmt_int(_n_long)} {_plural_ru(_n_long, ("номер прочитан как число", "номера прочитаны как числа",
                                                          "номеров прочитаны как числа"))} '
              'и длиннее 15 цифр — последние цифры могли быть '
              'искажены при чтении (Excel, float). Загрузите колонку как текст (dtype=str) и запустите ячейку снова.')
    del _extra, _n0, _n_long, _odd, _own_obj, _own_name
display(cinfo.head(3))

## 4. Эпизоды просрочки и проверки качества данных в `ovrd_contr`
Для каждой строки `dlq_start = act_date − (ovd_days_cnt − 1)`. Строки одного договора, у которых `dlq_start` совпадает
(с допуском `EPISODE_START_TOL_DAYS` дней), образуют один эпизод просрочки; по его максимальному числу дней просрочки (DPD)
видно, перекатился ли он дальше 30 дней просрочки.
Диагностика показывает, насколько плотно идут строки (ежедневно или с пропусками) и растёт ли DPD на один день за каждый
календарный день.

In [ ]:
def build_episodes(ovrd, tol_days=EPISODE_START_TOL_DAYS, data_end=None):
    d = pd.DataFrame({'contract': norm_code(ovrd[C_CONTRACT]).values,
                      'iin': norm_iin(ovrd[C_IIN]).values,
                      'act_date': to_dt(ovrd[C_ACT_DATE]).values,
                      'dpd': pd.to_numeric(ovrd[C_DPD], errors='coerce').values})
    n0 = len(d)
    d = d.dropna(subset=['contract', 'act_date', 'dpd'])
    d = d[d['dpd'] >= 1]
    if data_end is not None:
        d = d[d['act_date'] <= pd.Timestamp(data_end)]
    d['dpd'] = d['dpd'].round().astype('int32')
    # одна строка на договор-день: объединение со статусами договоров может дублировать строки; оставляем максимальный DPD
    d = (d.groupby(['contract', 'act_date'], sort=False)
           .agg(dpd=('dpd', 'max'), iin=('iin', 'first')).reset_index())
    d['dlq_start'] = d['act_date'] - pd.to_timedelta(d['dpd'] - 1, unit='D')
    d = d.sort_values(['contract', 'dlq_start', 'act_date'], kind='mergesort')
    gap = (d['dlq_start'] - d.groupby('contract')['dlq_start'].shift()).dt.days
    d['ep_seq'] = (gap.isna() | (gap > tol_days)).astype('int32').groupby(d['contract']).cumsum()
    ep = (d.groupby(['contract', 'ep_seq'], sort=False)
            .agg(iin=('iin', 'first'), dlq_start=('dlq_start', 'min'), max_dpd=('dpd', 'max'),
                 first_seen=('act_date', 'min'), last_seen=('act_date', 'max'), n_rows=('dpd', 'size'))
            .reset_index())
    ep = ep.sort_values(['contract', 'dlq_start'], kind='mergesort').reset_index(drop=True)
    # снижение DPD после частичной оплаты даёт «новый» dlq_start внутри ещё не закончившегося эпизода: это не новый выход на просрочку
    ep['overlap'] = (ep['dlq_start'] <= ep.groupby('contract')['last_seen'].shift()).values
    ep['ep_uid'] = np.arange(len(ep))
    stats = {'строк на входе': n0, 'договор-дней': len(d), 'эпизодов': len(ep),
             'договоров с просрочкой': ep['contract'].nunique(), 'клиентов с просрочкой': ep['iin'].nunique()}
    return ep, d, stats


with Timer('Эпизоды просрочки'):
    episodes, dpd_rows, ep_stats = build_episodes(ovrd_contr, data_end=DATA_END)
DATA_END_TS = pd.Timestamp(DATA_END) if DATA_END else dpd_rows['act_date'].max().normalize()
print({k: f'{fmt_int(v)}' for k, v in ep_stats.items()}, '| конец данных по DPD:', DATA_END_TS.date())

span = (episodes['last_seen'].clip(upper=DATA_END_TS) - episodes['dlq_start']).dt.days + 1
dq = pd.Series({
    'Строк на один день эпизода (1.0 = строка за каждый день)': (episodes['n_rows'] / span.clip(lower=1)).median(),
    'Доля эпизодов, представленных только одной строкой': (episodes['n_rows'] == 1).mean(),
    'Доля договор-дней со смещённой датой начала просрочки (дата среза − (DPD − 1))': ((dpd_rows['dlq_start'] - dpd_rows.groupby(['contract', 'ep_seq'])['dlq_start'].transform('min')).dt.days > 0).mean(),
    'Доля эпизодов, достигших DPD 31+': (episodes['max_dpd'] >= 31).mean(),
    'Доля эпизодов, начавшихся внутри открытого эпизода (частичная оплата)': episodes['overlap'].mean(),
    'Медиана максимального DPD': episodes['max_dpd'].median()})
display(dq.to_frame('value'))
del dpd_rows
gc.collect()

## 5. Покрытие мониторинга ПКБ и качество данных
Ежедневные объёмы событий мониторинга ПКБ (Первое кредитное бюро, PKB) показывают, когда мониторинг фактически начался
(первые дни часто загружены лишь частично); задержка между `event_date` (в ПКБ) и `created_date` (в банке) показывает,
насколько свежие данные поступают по каждому правилу.

Проверки потока событий (только таблицы и предупреждения, на расчёты следующих разделов не влияют):
* **провалы загрузки по правилам** — дни, когда событий правила меньше 35% (`DQ_GAP_SHARE`) его обычного уровня
  не меньше 2 дней подряд (`DQ_GAP_MIN_DAYS`). Обычный уровень — медиана 15 предыдущих нормальных дней правила (для
  выходных — медиана тех же дней недели). Праздники Казахстана (`DQ_HOLIDAYS`) не проверяются: в них поток как
  в выходной. Если не меньше 2 дней подряд поток упал вдвое у большинства правил (массовый провал), эти дни закрашены
  на графике, и ноутбук показывает, у каких дат скоринга популяции A окна наблюдения 7 / 14 / 30 дней попадают на эти
  дни и задевает ли провал тестовую выборку вне времени (OOT, разделы 10 и 15). Это же предупреждение повторяется под
  выводом по правилам (раздел 10), под приростом Джини (разделы 15 и 17) и в резюме (раздел 18). Для провалов отдельных правил в таблице есть доля эпизодов
  теста OOT, окно которых задевает провал;
* **изменение потока событий после периода выборки** — события в день за последние 14 полных дней (`DQ_RECENT_DAYS`)
  против периода, по которому считаются признаки популяции A. Резкое изменение — больше чем в 3 раза
  (`DQ_REGIME_RATIO`) в любую сторону или нет событий в последние дни: бины, WoE и пороги, подобранные на периоде
  выборки, тогда могут не переноситься на новые данные;
* **повторы событий** — по неделям: строки таблицы, строки без ID и уникальные ID события (`internal_event_id`).
  Таблица событий по правилам и неделям считает все строки, а признаки (раздел 7) считаются после удаления повторов.
  Строки без ID раздел 7 считает одним и тем же ID и оставляет из них одну — ноутбук предупредит, если такие строки
  есть.

Провалы загрузки и резкие изменения потока по каждому правилу повторяются в колонке «Качество данных» вывода по правилам
(раздел 10).

In [ ]:
# ------------------------------------------------------------------ проверки потока событий ПКБ (только показ и предупреждения)
# Ничего из проверок ниже не используется в расчётах следующих разделов. Результаты для просмотра:
# hes_daily_rule — число событий по правилам и дням, hes_gaps — провалы загрузки, hes_regime — поток событий после периода
# выборки, hes_dups — строки и уникальные ID событий по неделям. _dq_mass — дни массового провала загрузки: по ним
# _dq_oot_note печатает предупреждение под выводом по правилам (раздел 10), рядом с приростом Джини (разделы 15, 17)
# и в резюме (раздел 18).
hes_daily_rule = hes_gaps = hes_regime = hes_dups = None
_dq_mass = pd.DatetimeIndex([])
_dq_sd, _dq_cut = pd.Series([], dtype='datetime64[ns]'), None
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
_DQ_WD = ('пн', 'вт', 'ср', 'чт', 'пт', 'сб', 'вс')
_DQ_RULES = ('правило', 'правила', 'правил')            # «проверено 1 правило / 2 правила / 5 правил»
_DQ_RULES_GEN = ('правила', 'правил', 'правил')       # «у 1 правила / у 2 правил / у 5 правил»


def _dq_fmt(d, wd=False):
    # дата в виде дд.мм (wd=True — с днём недели: «26.10 (пн)»)
    d = pd.Timestamp(d)
    return d.strftime('%d.%m') + (f' ({_DQ_WD[d.dayofweek]})' if wd else '')


def _dq_period(a, b, wd=False):
    return _dq_fmt(a, wd) if pd.Timestamp(a) == pd.Timestamp(b) else f'{_dq_fmt(a, wd)}–{_dq_fmt(b, wd)}'


def _dq_num(x):
    # число для таблиц: целое с пробелами между разрядами, малые значения — с одним знаком после точки
    if not np.isfinite(x):
        return '—'
    return fmt_int(x) if abs(x) >= 10 else f'{x:.1f}'


def _dq_pct(x):
    return f'{x:.0%}' if np.isfinite(x) else '—'


def _dq_runs(mask):
    """Отрезки подряд идущих True: список (первый индекс, последний индекс)."""
    d = np.diff(np.concatenate([[0], np.asarray(mask, np.int8), [0]]))
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1) - 1))


def _dq_holidays():
    # праздничные дни из DQ_HOLIDAYS (первая ячейка кода)
    return pd.DatetimeIndex(pd.to_datetime(list(globals().get('DQ_HOLIDAYS') or []), errors='coerce')).dropna().normalize()


def _dq_weekly_duplicates(ids, ok, wk, w0, nw, chunk=5_000_000):
    """Строки, строки без ID и уникальные ID события по неделям без копии таблицы: 64-битный хеш ID (считается частями
    по chunk строк), в младших битах — номер недели. После сортировки первая строка каждого ID приходится на самую
    раннюю неделю, остальные строки — повторы (их удаляет раздел 7 перед расчётом признаков). Строки без ID (пусто,
    None, NaN, 'nan' …) не хешируются и считаются отдельно: drop_duplicates раздела 7 считает их одним и тем же ID.
    Совпадение хешей у разных ID практически исключено. Возвращает (строки, без ID, уникальные ID) по неделям."""
    bits = np.uint64(max(8, int(nw).bit_length()))
    w_all = (wk - w0).astype(np.int64)
    miss = np.zeros(len(ids), bool)
    for i in range(0, len(ids), chunk):
        part = ids.iloc[i:i + chunk]
        m = np.array(part.isna().to_numpy(), dtype=bool)                # копия: to_numpy() может быть только для чтения
        if not (pd.api.types.is_numeric_dtype(part.dtype) or pd.api.types.is_bool_dtype(part.dtype)):
            m |= part.isin(_NULLS).to_numpy()
        miss[i:i + chunk] = m
    use = ok & ~miss
    h = np.empty(int(use.sum()), np.uint64)
    k = 0
    for i in range(0, len(ids), chunk):
        sel = use[i:i + chunk]
        if sel.any():
            hh = pd.util.hash_pandas_object(ids.iloc[i:i + chunk], index=False, categorize=False).to_numpy()[sel]
            h[k:k + len(hh)] = hh
            k += len(hh)
            del hh
    w = w_all[use]
    h >>= bits
    h <<= bits
    h |= w.astype(np.uint64)
    h.sort()
    hi = h >> bits
    first = np.ones(len(h), bool)
    first[1:] = hi[1:] != hi[:-1]
    del hi
    rows = np.bincount(w_all[ok], minlength=nw)
    no_id = np.bincount(w_all[ok & miss], minlength=nw)
    uniq = np.bincount((h[first] & ((np.uint64(1) << bits) - np.uint64(1))).astype(np.int64), minlength=nw)
    del h, first, w, w_all, miss, use
    return rows, no_id, uniq


def _dq_find_gaps(cnt, first_day, last_day, share=DQ_GAP_SHARE, min_days=DQ_GAP_MIN_DAYS, hist=15, min_level=20):
    """Провалы загрузки по правилам. cnt — таблица правило x день (число событий по дате EVENT_TIME_COL).
    Ожидаемое число событий правила за день = медиана его последних hist нормальных дней x поправка на день недели
    (типичное отношение этого дня недели к медиане по 4 последним нормальным таким дням; так выходные с обычно меньшим
    потоком не считаются провалом). День провальный, если событий меньше share от ожидаемого; провал — не меньше
    min_days провальных дней подряд (два провала, разделённые одним днём, объединяются). Нормальный день — не меньше
    половины ожидаемого: дни провала и дни, когда поток упал вдвое, не попадают в медиану, чтобы долгий провал не стал
    «обычным уровнем». Праздники (DQ_HOLIDAYS) не проверяются и в медиану не попадают: поток в них как в выходной.
    Не проверяются также дни до first_day и до первого события правила, дни начиная с last_day (последний, неполный
    день) и правила, у которых обычно меньше min_level событий в день.
    Возвращает (таблица провалов, флаги провалов правило x день, флаги «поток упал вдвое» правило x день,
    число проверенных правил)."""
    cols = cnt.columns
    i_beg = int(np.searchsorted(cols.values, np.datetime64(first_day)))
    i_end = int(np.searchsorted(cols.values, np.datetime64(last_day)))
    wday = cols.dayofweek.values
    hol = cols.normalize().isin(_dq_holidays())
    flags, half = np.zeros(cnt.shape, bool), np.zeros(cnt.shape, bool)
    rows, n_checked = [], 0
    for r_i, (code, s) in enumerate(cnt.iterrows()):
        v = s.values.astype(float)
        nz = np.flatnonzero(v[:i_end] > 0)
        nz = nz[nz >= i_beg]
        if len(nz) == 0 or i_end - nz[0] < 7:
            continue
        i0 = nz[0]
        first = np.arange(i0, min(i0 + hist, i_end))
        first = first[~hol[first]] if (~hol[first]).any() else first
        start_level = np.median(v[first])
        if max(start_level, np.median(v[i0:i_end])) < min_level:
            continue
        n_checked += 1
        # начальная поправка на день недели — по первым hist дням правила
        f0 = {d: float(np.clip(np.median(v[first[wday[first] == d]]) / start_level, 0.2, 1.5))
              for d in range(7) if start_level > 0 and (wday[first] == d).any()}
        clean, low = [], np.zeros(len(v), bool)
        med, exp = np.full(len(v), np.nan), np.full(len(v), np.nan)
        for i in range(i0, i_end):
            med[i] = np.median(v[clean[-hist:]]) if len(clean) >= 7 else start_level
            same = [k for k in clean if wday[k] == wday[i] and med[k] > 0][-4:]
            f = (float(np.clip(np.median(v[same] / med[same]), 0.2, 1.5)) if len(same) >= 2
                 else f0.get(wday[i], 1.0))
            exp[i] = med[i] * f
            if hol[i]:                                # праздник: не провал и не «нормальный» день
                continue
            if exp[i] >= min_level and v[i] < share * exp[i]:
                low[i] = True
            if exp[i] >= min_level and v[i] < 0.5 * exp[i]:
                half[r_i, i] = True
            else:
                clean.append(i)
        runs = [(a, b) for a, b in _dq_runs(low) if b - a + 1 >= min_days]
        merged = []
        for a, b in runs:                             # провалы, разделённые одним днём (например, праздником), — один провал
            if merged and a - merged[-1][1] == 2:
                merged[-1] = (merged[-1][0], b)
            else:
                merged.append((a, b))
        for a, b in merged:
            flags[r_i, a:b + 1] = True
            e, act = exp[a:b + 1], v[a:b + 1]
            rows.append({'rule': code, 'start': cols[a], 'end': cols[b], 'n_days': int(b - a + 1),
                         'expected_per_day': float(e.mean()), 'actual_per_day': float(act.mean()),
                         'share_of_expected': float(act.sum() / e.sum()),
                         'missing_events': float(np.clip(e - act, 0, None).sum()), 'until_end': bool(b == i_end - 1)})
    return (pd.DataFrame(rows), pd.DataFrame(flags, index=cnt.index, columns=cols),
            pd.DataFrame(half, index=cnt.index, columns=cols), n_checked)


def _dq_mass_days(half, n_checked, min_days=DQ_GAP_MIN_DAYS):
    """Массовый провал: дни, когда поток упал хотя бы вдвое у большинства (не меньше половины) проверенных правил,
    не меньше min_days дней подряд (один такой день — скорее нерабочий день, чем сбой загрузки). Праздник между
    днями массового провала считается его частью. Возвращает (число правил с падением вдвое по дням, порог, дни)."""
    per_day = half.sum(axis=0)
    thr = max(2, int(np.ceil(0.5 * n_checked)))
    m = (per_day >= thr).values.copy()
    hol = half.columns.normalize().isin(_dq_holidays())
    for i in np.flatnonzero(hol):
        a, b = i - 1, i + 1
        while a >= 0 and hol[a]:
            a -= 1
        while b < len(m) and hol[b]:
            b += 1
        if a >= 0 and b < len(m) and m[a] and m[b]:
            m[i] = True
    keep = np.zeros(len(m), bool)
    for a, b in _dq_runs(m):
        if b - a + 1 >= min_days:
            keep[a:b + 1] = True
    return per_day, thr, per_day.index[keep]


def _dq_pop_a_score_dates():
    """Даты скоринга популяции A по тем же условиям, что build_collection_population (раздел 6): популяция A строится
    позже, поэтому условия повторены здесь только для предупреждений."""
    e = episodes[(episodes['max_dpd'] >= OBS_DPD) & ~episodes['overlap']]
    sd = e['dlq_start'] + days(OBS_DPD - 1)
    m = ((sd >= HES_START_TS + days(MIN_HISTORY_DAYS)) & (e['dlq_start'] <= DATA_END_TS - days(BAD_DPD - 1))
         & (sd <= HES_END_TS) & e['iin'].notna())
    return sd[m].reset_index(drop=True)


def _dq_window_hits(score_dates, bad_days):
    """{W: True, если окно наблюдения признаков даты скоринга (дни событий [дата − EVENT_LAG_DAYS − W, дата −
    EVENT_LAG_DAYS − 1], как в разделе 8) содержит хотя бы один день из bad_days}."""
    bad = np.sort(pd.DatetimeIndex(bad_days).values.astype('datetime64[D]').astype(np.int64))
    s = pd.to_datetime(pd.Series(score_dates)).values.astype('datetime64[D]').astype(np.int64) - EVENT_LAG_DAYS
    return {W: (np.searchsorted(bad, s) - np.searchsorted(bad, s - W)) > 0 for W in WINDOWS}


def _dq_oot_note(score_dates, unit='эпизодов', where='раздел 15', tail=None):
    """Текст предупреждения, если окна наблюдения признаков тестовой выборки вне времени (OOT: даты скоринга позже
    квантиля TRAIN_SHARE, как в incremental_value) задевают дни массового провала загрузки ПКБ (_dq_mass, раздел 5);
    '' — не задевают или проверка не выполнялась. tail — свой конец предупреждения (раздел 10), None — о приросте Джини."""
    mass = globals().get('_dq_mass')
    sd = pd.Series(pd.to_datetime(pd.Series(score_dates))).dropna().reset_index(drop=True)
    if mass is None or not len(mass) or not len(sd):
        return ''
    test = (sd > sd.quantile(TRAIN_SHARE)).values
    if not test.any():
        return ''
    hits = _dq_window_hits(sd[test], mass)
    sh = {W: float(hits[W].mean()) for W in WINDOWS}
    if max(sh.values()) == 0:
        return ''
    runs = _dq_runs(pd.date_range(mass.min(), mass.max()).isin(mass))
    per = ', '.join(_dq_period(mass.min() + days(int(a)), mass.min() + days(int(b))) for a, b in runs)
    return (f'ВНИМАНИЕ: окна наблюдения признаков ПКБ задевают провал загрузки ПКБ {per} (раздел 5) у {unit} тестовой '
            'выборки: ' + ', '.join(f'{_dq_pct(sh[W])} при окне {W} дн.' for W in WINDOWS)
            + (tail if tail is not None else
               f'; прирост Джини на выборке вне времени (OOT, {where}) посчитан по неполным данным ПКБ; перед выводом о '
               'ценности ПКБ сравните его на периоде без провала.'))


def _dq_regime(cnt, win_from, win_to, rec_from, rec_to, ratio=DQ_REGIME_RATIO, min_rate=5):
    """По каждому правилу: событий в день в периоде выборки A [win_from, win_to] и в последние полные дни
    [rec_from, rec_to] (дни до первого события правила не учитываются), их отношение и вывод."""
    cols = cnt.columns
    rows = []
    for code, s in cnt.iterrows():
        pos = np.flatnonzero(s.values > 0)
        if len(pos) == 0:
            continue
        first = cols[pos[0]]
        a = s[(cols >= max(win_from, first)) & (cols <= win_to)]
        b = s[(cols >= max(rec_from, first)) & (cols <= rec_to)]
        r_pop = float(a.mean()) if len(a) else 0.0
        r_rec = float(b.mean()) if len(b) else 0.0
        upto = s[cols <= rec_to].values
        zeros = len(upto) - (np.flatnonzero(upto > 0)[-1] + 1) if (upto > 0).any() else len(upto)
        flag = True
        if r_pop == 0 and r_rec > 0:
            verdict = 'появилось после периода выборки A'
        elif r_pop > 0 and zeros >= 3 and r_pop >= min_rate:
            verdict = f'нет событий с {_dq_fmt(rec_to - days(zeros - 1))}'
        elif max(r_pop, r_rec) < min_rate:
            verdict, flag = 'мало событий для оценки', False
        elif r_rec > ratio * r_pop:
            verdict = f'рост в {r_rec / r_pop:.1f} раза'
        elif r_rec < r_pop / ratio:
            verdict = f'падение в {r_pop / r_rec:.1f} раза' if r_rec > 0 else 'нет событий'
        else:
            verdict, flag = 'без резких изменений', False
        rows.append({'rule': code, 'per_day_population': r_pop, 'per_day_recent': r_rec,
                     'ratio': r_rec / r_pop if r_pop > 0 else NAN, 'zero_days_at_end': int(zeros),
                     'verdict': verdict, 'flag': flag})
    out = pd.DataFrame(rows)
    if len(out):
        # сначала резкие изменения (самые сильные выше), затем остальные правила по убыванию потока, «мало событий» — в конце
        dev = np.abs(np.log(out['ratio'].where(out['ratio'] > 0))).fillna(np.inf)
        grp = np.where(out['flag'], 0, np.where(out['verdict'] == 'мало событий для оценки', 2, 1))
        out = (out.assign(_g=grp, _d=np.where(out['flag'], -dev, 0.0))
                  .sort_values(['_g', '_d', 'per_day_population'], ascending=[True, True, False])
                  .drop(columns=['_g', '_d']).reset_index(drop=True))
    return out


def _dq_report(flags, half, n_checked, thr, per_day, mass, sd, cut, last_full):
    """Таблицы и предупреждения проверок а) и б) раздела 5; возвращает (gaps, regime)."""
    gaps, regime = hes_gaps, None
    names_of = dict(zip(hes_daily_rule.index, code_names(hes_daily_rule.index)))
    test = (sd > cut).values if len(sd) and cut is not None else np.zeros(len(sd), bool)
    W_max = max(WINDOWS)
    # ---- а) провалы загрузки по правилам
    if n_checked == 0:
        print('\nПровалы загрузки не проверялись: ни у одного правила нет 7 дней истории в периоде покрытия '
              f'(с {_dq_fmt(HES_START_TS)}) при обычном уровне от 20 событий в день.')
    else:
        print(f'\nПровалы загрузки по правилам: за день событий меньше {DQ_GAP_SHARE:.0%} обычного уровня правила, не меньше '
              f'{DQ_GAP_MIN_DAYS} дней подряд (по дате {EVENT_TIME_COL}; проверено {n_checked} '
              f'{_plural_ru(n_checked, _DQ_RULES)} — с обычным уровнем от 20 событий в день; последний, неполный день '
              f'{_dq_fmt(HES_END_TS)} и праздники из DQ_HOLIDAYS не проверяются).')
    if len(gaps):
        gaps = gaps.sort_values(['start', 'rule']).reset_index(drop=True)
        # доля эпизодов тестовой выборки OOT, у которых окно W_max дней задевает провал этого правила
        oot = [float(_dq_window_hits(sd[test], pd.date_range(a, b))[W_max].mean()) if test.any() else NAN
               for a, b in zip(gaps['start'], gaps['end'])]
        t = pd.DataFrame({
            'Правило': [names_of[c] for c in gaps['rule']],
            'Период': [_dq_period(a, b) + (' (до конца данных)' if e else '')
                       for a, b, e in zip(gaps['start'], gaps['end'], gaps['until_end'])],
            'Дней': gaps['n_days'].values,
            'Ожидалось событий в день': [_dq_num(x) for x in gaps['expected_per_day']],
            'Фактически в среднем в день': [_dq_num(x) for x in gaps['actual_per_day']],
            'Доля от ожидаемого': [_dq_pct(x) for x in gaps['share_of_expected']],
            'Недополучено событий (оценка)': [_dq_num(x) for x in gaps['missing_events']],
            f'Задевает тест OOT (окно {W_max} дн.), доля эпизодов': [_dq_pct(x) for x in oot],
            CODE_COL: gaps['rule'].values}, index=pd.RangeIndex(1, len(gaps) + 1, name='№'))
        print('Провалы загрузки по правилам:')
        _display_orig(t)
        # провалы отдельных правил вне массового провала (о массовом провале — отдельное предупреждение ниже)
        hit = [(names_of[c], x) for c, x, a, b in zip(gaps['rule'], oot, gaps['start'], gaps['end'])
               if np.isfinite(x) and x > 0 and not pd.date_range(a, b).isin(mass).all()]
        if hit:
            best = {}
            for nm, x in hit:
                best[nm] = max(best.get(nm, 0), x)
            print(f'ВНИМАНИЕ: провалы загрузки задевают окна наблюдения ({W_max} дн.) тестовой выборки вне времени (OOT, '
                  'разделы 10 и 15): ' + '; '.join(f'«{nm}» — у {_dq_pct(x)} эпизодов теста' for nm, x in best.items())
                  + '. Признаки этих правил на тесте посчитаны по неполным данным.')
    elif n_checked:
        print('Провалов загрузки по правилам не найдено.')
    mass_runs = _dq_runs(per_day.index.isin(mass)) if len(mass) else []
    hol = per_day.index.normalize().isin(_dq_holidays())
    if mass_runs:
        print(f'ВНИМАНИЕ: массовый провал загрузки — не меньше {DQ_GAP_MIN_DAYS} дней подряд, когда поток событий упал '
              f'хотя бы вдвое у большинства правил (не меньше {thr} из {n_checked} проверенных):')
        for a, b in mass_runs:
            hit_n = half.iloc[:, a:b + 1].sum(axis=1)
            hit_n = hit_n[hit_n > 0].sort_values(ascending=False)
            deep = flags.iloc[:, a:b + 1].any(axis=1)
            work = [i for i in range(a, b + 1) if not hol[i]] or list(range(a, b + 1))
            lo, hi_ = int(per_day.iloc[work].min()), int(per_day.iloc[work].max())
            names = ', '.join(f'«{names_of[c]}»' for c in hit_n.index[:10]) + (f' и ещё {len(hit_n) - 10}' if len(hit_n) > 10 else '')
            print(f'  {_dq_period(per_day.index[a], per_day.index[b], wd=True)} ({b - a + 1} дн.): поток упал вдвое и больше '
                  + (f'у {lo} из {n_checked} проверенных правил' if lo == hi_ else
                     f'в разные дни — у {lo}–{hi_} из {n_checked} проверенных правил')   # «из N» — род. падеж
                  + f'; ниже {DQ_GAP_SHARE:.0%} обычного уровня (таблица выше) — у {int(deep.sum())} '
                  f'{_plural_ru(int(deep.sum()), _DQ_RULES_GEN)}. Правила: {names}')
    elif len(gaps):
        print('Массового провала (поток упал вдвое у большинства правил сразу) нет: провалы выше касаются отдельных правил.')

    # окна наблюдения популяции A и выборка вне времени (OOT), пересекающиеся с днями массового провала
    if len(mass) and len(sd):
        hits = _dq_window_hits(sd, mass)
        rows = []
        for W in WINDOWS:
            h = hits[W]
            rows.append({
                'Окно наблюдения признаков': f'Окно {W} дн.',
                'Даты скоринга с затронутым окном': _dq_period(sd[h].min(), sd[h].max()) if h.any() else '—',
                'Эпизодов популяции A': fmt_int(h.sum()),
                'Доля популяции A': _dq_pct(h.mean()),
                'Доля обучающей выборки': _dq_pct(h[~test].mean()) if (~test).any() else '—',
                'Доля тестовой выборки (OOT)': _dq_pct(h[test].mean()) if test.any() else '—'})
        t = pd.DataFrame(rows).set_index('Окно наблюдения признаков')
        print(f'\nПопуляция A (те же условия, что в разделе 6): {fmt_int(len(sd))} эпизодов, даты скоринга '
              f'{_dq_period(sd.min(), sd.max())}; выборка вне времени (OOT, разделы 10 и 15): обучение — даты скоринга '
              f'до {_dq_fmt(cut)} включительно, тест — {_dq_period(sd[test].min(), sd.max()) if test.any() else "—"}. '
              'Окна наблюдения, в которые попадают дни массового провала:')
        _display_orig(t)
        test_in_gap = test.any() and bool(pd.Series(sd[test].dt.normalize()).isin(mass).mean() > 0)
        note = _dq_oot_note(sd)
        if note:
            print(note + (' Сами даты скоринга части теста тоже попадают в дни провала.' if test_in_gap else ''))
    elif len(mass):
        print('Популяция A пуста: пересечение окон наблюдения с провалом не проверялось.')

    # ---- б) изменение потока событий после периода выборки A
    if not len(sd):
        print('\nПопуляция A пуста: изменение потока событий после периода выборки не проверялось.')
    else:
        win_from = max(sd.min().normalize() - days(EVENT_LAG_DAYS + max(WINDOWS)), hes_daily_rule.columns.min())
        win_to = sd.max().normalize() - days(EVENT_LAG_DAYS + 1)
        rec_from = last_full - days(DQ_RECENT_DAYS - 1)
        regime = _dq_regime(hes_daily_rule, win_from, win_to, rec_from, last_full)
        print(f'\nИзменение потока событий после периода выборки: событий в день за последние {DQ_RECENT_DAYS} полных '
              f'дней ({_dq_period(rec_from, last_full)}) против периода выборки A ({_dq_period(win_from, win_to)}: '
              'дни событий, которые попадают в признаки популяции A, включая дни провалов). Резкое изменение — '
              f'больше чем в {DQ_REGIME_RATIO} раза в любую сторону или нет событий в последние дни:')
        if rec_from <= win_to:
            print(f'  (последние {DQ_RECENT_DAYS} дней пересекаются с периодом выборки A: сравнение менее показательно)')
        if len(regime):
            t = pd.DataFrame({
                'Правило': [names_of[c] for c in regime['rule']],
                'Событий в день: период выборки A': [_dq_num(x) for x in regime['per_day_population']],
                f'Событий в день: последние {DQ_RECENT_DAYS} дн.': [_dq_num(x) for x in regime['per_day_recent']],
                'Во сколько раз изменился поток': [f'{x:.2f}' if np.isfinite(x) else '—' for x in regime['ratio']],
                'Вывод': [('РЕЗКОЕ ИЗМЕНЕНИЕ: ' if f else '') + v for v, f in zip(regime['verdict'], regime['flag'])],
                CODE_COL: regime['rule'].values}, index=pd.RangeIndex(1, len(regime) + 1, name='№'))
            _display_orig(t)
            fl = regime[regime['flag']]
            if len(fl):
                print(f'ВНИМАНИЕ: поток событий резко изменился у {len(fl)} {_plural_ru(len(fl), _DQ_RULES_GEN)}: '
                      + '; '.join(f'«{names_of[r.rule]}» — {r.verdict}' for r in fl.itertuples()) + '. '
                      'Бины, WoE и пороги признаков этих правил подобраны на периоде выборки A и могут не переноситься '
                      'на текущий поток событий: перед применением в стратегии проверьте их на новых данных.')
            else:
                print('Резких изменений потока событий после периода выборки A нет.')
    return gaps, regime


RULE_MAP = {}
if isinstance(hes, pd.DataFrame):
    with Timer('Ежедневные и еженедельные объёмы по правилам (вся таблица, подсчёт средствами numpy)'):
        t_col = hes[EVENT_TIME_COL]
        if not pd.api.types.is_datetime64_any_dtype(t_col):
            t_col = to_dt(t_col)
        day_num = t_col.values.astype('datetime64[D]').astype('int64')       # дни с 1970-01-01, NaT -> минимальное значение int
        ok = ~np.isnat(t_col.values)
        d0 = int(day_num[ok].min())
        daily_cnt = np.bincount(day_num[ok] - d0)
        daily = pd.Series(daily_cnt, index=pd.to_datetime(np.arange(len(daily_cnt)) + d0, unit='D'))
        daily = daily[daily > 0]
        if RUN_HES_DQ:
            rcode, runiq = pd.factorize(hes[H_RULE].values)                 # -1 = название правила отсутствует
            RULE_MAP = map_rule_names(runiq)
            wk = (day_num - 4) // 7                                         # недели, начинающиеся с понедельника
            w0 = int(wk[ok].min())
            nw = int(wk[ok].max()) - w0 + 1
            keep = ok & (rcode >= 0)
            cnt = np.bincount(rcode[keep] * nw + (wk[keep] - w0), minlength=len(runiq) * nw).reshape(len(runiq), nw)
            week_start = pd.to_datetime((np.arange(nw) + w0) * 7 + 4, unit='D')
            weekly_rule = (pd.DataFrame(cnt, index=[RULE_MAP[r] for r in runiq],
                                        columns=[d.strftime('%d.%m') for d in week_start])
                             .groupby(level=0).sum())
            # число событий по правилам и дням (для поиска провалов загрузки и изменений потока событий)
            nd = len(daily_cnt)
            ix = rcode[keep] * nd
            ix += day_num[keep] - d0
            hes_daily_rule = (pd.DataFrame(np.bincount(ix, minlength=len(runiq) * nd).reshape(len(runiq), nd),
                                           index=[RULE_MAP[r] for r in runiq],
                                           columns=pd.to_datetime(np.arange(nd) + d0, unit='D'))
                                .groupby(level=0).sum())
            del ix
            # строки и уникальные ID события по неделям (повторы удаляются в разделе 7)
            if H_ID in hes.columns and len(hes) <= DQ_DUP_MAX_ROWS:
                try:
                    _rows, _noid, _uniq = _dq_weekly_duplicates(hes[H_ID], ok, wk, w0, nw)
                    hes_dups = pd.DataFrame({'rows': _rows, 'no_id': _noid, 'unique_ids': _uniq,
                                             'repeats': _rows - _noid - _uniq},
                                            index=[d.strftime('%d.%m') for d in week_start])
                except Exception as e:                                      # например, не хватило памяти
                    print(f'  проверка повторов ID события пропущена: {type(e).__name__}: {e}')
            del rcode, wk, keep
            if H_EVENT_DATE in hes.columns:
                smp = _sample_rows(hes[[H_RULE, H_EVENT_DATE, EVENT_TIME_COL]], DQ_SAMPLE)
                lag_h = (to_dt(smp[EVENT_TIME_COL]) - to_dt(smp[H_EVENT_DATE])) / np.timedelta64(1, 'h')
                lat = (pd.DataFrame({'rule': smp[H_RULE].map(RULE_MAP).values, 'lag_h': lag_h.values})
                         .groupby('rule')['lag_h'].describe(percentiles=[.5, .9, .99])[['count', '50%', '90%', '99%']])
                del smp
        del t_col, day_num, ok
        gc.collect()
    med = daily.median()
    HES_START_TS = pd.Timestamp(HES_START) if HES_START else daily[daily >= 0.5 * med].index.min()
    HES_END_TS = daily.index.max()
    print(f'событий: {fmt_int(len(hes))} | первый день {_dq_fmt(daily.index.min())}.{daily.index.min().year} | '
          f'полное покрытие определено с {_dq_fmt(HES_START_TS)}.{HES_START_TS.year} | последний день '
          f'{_dq_fmt(HES_END_TS)}.{HES_END_TS.year}')

    # названия правил ПКБ, которые получили один и тот же код other_<...> (код обрезан до 30 символов) и считаются вместе
    for _dq_c, _dq_names in _merged_rule_names().items():
        print(f'ВНИМАНИЕ: {len(set(_dq_names))} '
              f'{_plural_ru(len(set(_dq_names)), ("разное правило", "разных правила", "разных правил"))} ПКБ получили один '
              'код и считаются как одно правило '
              f'«{code_name(_dq_c)}» (код в данных {_dq_c}): ' + '; '.join(f'«{n}»' for n in sorted(set(_dq_names))))

    # даты скоринга популяции A и граница выборки вне времени (OOT) — так же, как в разделах 6, 10 и 15
    try:
        _dq_sd = _dq_pop_a_score_dates()
        _dq_cut = pd.Series(_dq_sd).quantile(TRAIN_SHARE) if len(_dq_sd) else None
        _dq_last_full = HES_END_TS - days(1)                             # последний день данных считается неполным
        if hes_daily_rule is not None:
            hes_gaps, _dq_flags, _dq_half, _dq_n_checked = _dq_find_gaps(hes_daily_rule, HES_START_TS, HES_END_TS)
            # массовый провал: не меньше DQ_GAP_MIN_DAYS дней подряд, когда поток упал хотя бы вдвое у большинства правил
            _dq_per_day, _dq_thr, _dq_mass = _dq_mass_days(_dq_half, _dq_n_checked)
    except Exception as e:                                               # проверки только для показа: ноутбук продолжается
        print(f'  проверки потока событий пропущены: {type(e).__name__}: {e}')
        hes_daily_rule, _dq_mass = None, pd.DatetimeIndex([])

    fig, ax = plt.subplots(figsize=(10, 3.0))
    ax.plot(daily.index, daily.values, color=PAL['blue'])
    ax.axvline(HES_START_TS, color=PAL['gray'], lw=0.8, label='начало полного покрытия')
    for _dq_k, (_dq_a, _dq_b) in enumerate(_dq_runs(daily.index.isin(_dq_mass))):
        ax.axvspan(daily.index[_dq_a], daily.index[_dq_b] + days(1), color=PAL['red'], alpha=0.12, lw=0,
                   label='провал загрузки: поток упал вдвое у большинства правил' if _dq_k == 0 else None)
    if _dq_cut is not None and (_dq_sd > _dq_cut).any():           # первая дата скоринга тестовой выборки
        ax.axvline(_dq_sd[_dq_sd > _dq_cut].min(), color=PAL['orange'], lw=1, ls='--',
                   label='начало тестовой выборки (OOT), популяция A')
    ax.set_ylim(bottom=0)                                           # от нуля: обычный шум ±3% не выглядит провалом
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: fmt_int(v)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
    ax.set_title('События мониторинга ПКБ по дням (все правила)')
    ax.set_ylabel('число событий')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=3, fontsize=7.5)   # под графиком, не закрывает линию
    plt.tight_layout(); plt.show()
    if RUN_HES_DQ:
        weekly_rule.insert(0, 'description', [RULE_DESC.get(c, '') for c in weekly_rule.index])
        print('События по правилам и неделям (в заголовке колонки — начало недели, понедельник; считаются все строки '
              'таблицы, в том числе повторы одного события):')
        display(weekly_rule)
        if hes_dups is not None:
            _dq_r, _dq_n, _dq_u = int(hes_dups['rows'].sum()), int(hes_dups['no_id'].sum()), int(hes_dups['unique_ids'].sum())
            print(f'Строки и уникальные ID события ({H_ID}) по тем же неделям: всего строк {fmt_int(_dq_r)}, без ID '
                  f'{fmt_int(_dq_n)}, уникальных ID {fmt_int(_dq_u)}, повторов ID {fmt_int(_dq_r - _dq_n - _dq_u)} '
                  f'({(_dq_r - _dq_n - _dq_u) / max(_dq_r - _dq_n, 1):.1%} строк с ID). Признаки (раздел 7) считаются '
                  'после удаления повторов; повтор относится к неделе, в которой ID встретился не первым:')
            _dq_t = pd.DataFrame([[fmt_int(n) for n in hes_dups['rows']], [fmt_int(n) for n in hes_dups['no_id']],
                                  [fmt_int(n) for n in hes_dups['unique_ids']], [fmt_int(n) for n in hes_dups['repeats']],
                                  [f'{q / (n - m):.1%}' if n > m else '—'
                                   for n, m, q in zip(hes_dups['rows'], hes_dups['no_id'], hes_dups['repeats'])]],
                                 index=['Строк в таблице событий', 'Строк без ID события', 'Уникальных ID события',
                                        'Повторов ID события', 'Доля повторов среди строк с ID'],
                                 columns=list(hes_dups.index), dtype=object)
            _display_orig(_dq_t)
            if _dq_n:
                print(f'ВНИМАНИЕ: {fmt_int(_dq_n)} {_plural_ru(_dq_n, ("строка", "строки", "строк"))} без {H_ID} '
                      f'({_dq_n / max(_dq_r, 1):.1%} таблицы). Раздел 7 удаляет '
                      'повторы по ID и из всех строк без ID (по клиентам популяции) оставит одну: почти все эти события '
                      f'не попадут в признаки. Заполните {H_ID} или проверьте выгрузку.')
        elif H_ID not in hes.columns:
            print(f'Колонки {H_ID} нет: повторы событий не проверялись, и раздел 7 их не удаляет.')
        elif len(hes) > DQ_DUP_MAX_ROWS:
            print(f'Повторы ID события по неделям не считались: строк больше DQ_DUP_MAX_ROWS = {fmt_int(DQ_DUP_MAX_ROWS)}')
        if H_EVENT_DATE in hes.columns:
            print(f'Задержка поступления события в банк, часов ({EVENT_TIME_COL} − {H_EVENT_DATE}; выборка из '
                  f'{fmt_int(DQ_SAMPLE)} строк):')
            display(lat)

        if hes_daily_rule is not None:
            try:
                hes_gaps, hes_regime = _dq_report(_dq_flags, _dq_half, _dq_n_checked, _dq_thr, _dq_per_day, _dq_mass,
                                                  _dq_sd, _dq_cut, _dq_last_full)
            except Exception as e:                                       # проверки только для показа
                print(f'  отчёт проверок потока событий пропущен: {type(e).__name__}: {e}')
else:
    if HES_START is None:
        raise ValueError('hes является итератором: задайте HES_START явно')
    HES_START_TS = pd.Timestamp(HES_START)
    HES_END_TS = DATA_END_TS
    print('hes читается по частям: проверки покрытия и качества данных пропущены')
print('Найдено названий правил:', len(RULE_MAP), '| не распознаны:', [k for k, v in RULE_MAP.items() if v.startswith('other_')])

## 6. Популяция A: выходы в софт-коллекшн, целевая переменная и переменные договора
Софт-коллекшн (soft collection) — ранняя стадия взыскания, просрочка 1–30 дней.
* дата скоринга `score_date = dlq_start + OBS_DPD − 1`, начиная с `HES_START + MIN_HISTORY_DAYS`;
* исход должен быть наблюдаемым: `dlq_start ≤ DATA_END − (BAD_DPD − 1)`;
* целевая переменная (таргет, target): `target = 1`, если число дней просрочки (DPD) в эпизоде достигает `BAD_DPD` (DPD ≥ `BAD_DPD`).

**Переменные договора** (коды `bl_…`; в коде — группа `baseline`) — это данные банка о договоре и клиенте, известные на
дату скоринга, а не признаки ПКБ: продукт, канал, сумма, срок с момента выдачи в днях (months on book, `bl_mob_days`),
доля истекшего срока договора, предыдущие просрочки по договору и по клиенту у нас (количество, худший DPD на дату
скоринга, число дней с момента последней), другие наши договоры клиента, находящиеся в просрочке сейчас, и число
открытых наших договоров. С признаками ПКБ они не сравниваются и в рейтинги признаков ПКБ (разделы 10–14, 16, 17) не
входят: они нужны только в дополнительном разделе 15 (добавляют ли правила ПКБ точности к переменным договора).
В Excel их IV и бины — на последнем листе «Переменные договора (справочно)».

In [ ]:
POP_START = HES_START_TS + days(MIN_HISTORY_DAYS)
LAST_START = DATA_END_TS - days(BAD_DPD - 1)


def build_collection_population(ep):
    p = ep[(ep['max_dpd'] >= OBS_DPD) & ~ep['overlap']].copy()
    p['score_date'] = p['dlq_start'] + days(OBS_DPD - 1)
    p = p[(p['score_date'] >= POP_START) & (p['dlq_start'] <= LAST_START) & (p['score_date'] <= HES_END_TS)]
    p['target'] = (p['max_dpd'] >= BAD_DPD).astype('int8')
    for name, k in EXTRA_TARGETS.items():
        observed = p['dlq_start'] <= DATA_END_TS - days(k - 1)
        p[name] = np.where(observed, (p['max_dpd'] >= k).astype(float), np.nan)
    p = p.dropna(subset=['iin']).reset_index(drop=True)
    p['ep_id'] = np.arange(len(p))
    p['week'] = p['score_date'].dt.to_period('W-SUN').dt.start_time
    p['hist_days'] = (p['score_date'] - HES_START_TS).dt.days
    return p


def build_baseline(pop, ep_all, cinfo):
    """Внутренние признаки банка на дату скоринга (без данных ПКБ)."""
    b = pop[['ep_id', 'contract', 'iin', 'score_date']].merge(
        cinfo.drop(columns=['iin']), on='contract', how='left')
    out = pd.DataFrame({'ep_id': b['ep_id'].values})
    out['bl_product'] = b['product'].fillna('NA').astype(str).values
    out['bl_channel'] = b['channel'].fillna('NA').astype(str).values
    out['bl_amount'] = b['amount'].values
    mob = (b['score_date'] - b['open_date']).dt.days
    term = (b['end_date'] - b['open_date']).dt.days
    out['bl_mob_days'] = mob.values
    out['bl_term_days'] = term.values
    out['bl_term_elapsed'] = (mob / term.where(term > 0)).values
    hist = ep_all[['contract', 'iin', 'dlq_start', 'max_dpd', 'ep_uid']]
    for lvl, key in (('c', 'contract'), ('i', 'iin')):
        h = pop[['ep_id', key, 'score_date', 'ep_uid']].merge(
            hist[[key, 'dlq_start', 'max_dpd', 'ep_uid']], on=key, suffixes=('', '_h'))
        h = h[(h['dlq_start'] < h['score_date']) & (h['ep_uid_h'] != h['ep_uid'])]
        days_in = (h['score_date'] - h['dlq_start']).dt.days          # DPD, который был у старого эпизода вчера (накануне даты скоринга)
        h['dpd_asof'] = np.minimum(h['max_dpd'], days_in)
        h['end_asof'] = h['dlq_start'] + pd.to_timedelta(h['dpd_asof'] - 1, unit='D')
        h['ongoing'] = h['max_dpd'] >= days_in
        h['recent'] = h['dlq_start'] >= h['score_date'] - days(180)
        g = h.groupby('ep_id')
        agg = pd.DataFrame({
            f'bl_{lvl}_n_prev_ep': g.size(),
            f'bl_{lvl}_n_prev_ep_180d': g['recent'].sum(),
            f'bl_{lvl}_max_prev_dpd': g['dpd_asof'].max(),
            f'bl_{lvl}_days_since_prev': (g['score_date'].first() - g['end_asof'].max()).dt.days})
        if lvl == 'i':
            agg['bl_i_n_dlq_now'] = g['ongoing'].sum()
            agg['bl_i_n_prev_roll'] = (h['dpd_asof'] >= 31).groupby(h['ep_id']).sum()
        agg = agg.reindex(out['ep_id'].values)
        for c in agg.columns:
            vals = agg[c].values.astype(float)
            out[c] = vals if 'days_since' in c else np.nan_to_num(vals, nan=0.0)
    first_dlq = ep_all.groupby('contract')['dlq_start'].min().rename('first_dlq')
    oc = pop[['ep_id', 'iin', 'score_date']].merge(
        cinfo[['contract', 'iin', 'open_date', 'close_date', 'amount', 'from_contracts']].join(first_dlq, on='contract'),
        on='iin')
    known = oc['from_contracts'].astype(bool) | (oc['first_dlq'] <= oc['score_date'])
    oc = oc[known & (oc['open_date'] <= oc['score_date']) & ~(oc['close_date'] < oc['score_date'])]
    g = oc.groupby('ep_id')
    out['bl_i_n_open_contracts'] = g.size().reindex(out['ep_id'].values).fillna(0).values
    out['bl_i_open_amount'] = g['amount'].sum().reindex(out['ep_id'].values).fillna(0).values
    return out.set_index('ep_id')


with Timer('Популяция A и переменные договора'):
    pop = build_collection_population(episodes)
    base_A = build_baseline(pop, episodes, cinfo)
print(f'Популяция A: {fmt_int(len(pop))} эпизодов, {fmt_int(pop["iin"].nunique())} клиентов, '
      f'даты скоринга {pop["score_date"].min().date()} .. {pop["score_date"].max().date()}, '
      f'доля «плохих» (bad rate) {pop["target"].mean():.2%} ({fmt_int(int(pop["target"].sum()))} «плохих»)')
cohorts = pop.groupby('week').agg(episodes=('target', 'size'), bads=('target', 'sum'), bad_rate=('target', 'mean'),
                                  **{f'{k}_rate': (k, 'mean') for k in EXTRA_TARGETS})
display(cohorts)
if len(pop) == 0:
    raise ValueError('Пустая популяция: проверьте HES_START / MIN_HISTORY_DAYS / DATA_END')

### Популяция B (необязательная): действующие договоры без просрочки, раннее предупреждение (early warning, EWS)
Для каждого договора выбирается один случайный еженедельный срез (snapshot) внутри периода покрытия мониторинга ПКБ
(Первое кредитное бюро, PKB). На дату среза договор должен быть открыт и не находиться в просрочке. `target = 1`
(целевая переменная, таргет), если в течение `EWS_HORIZON` дней начинается эпизод просрочки, в котором число
дней просрочки (DPD) достигает `EWS_MIN_DPD` (DPD ≥ `EWS_MIN_DPD`). Статус «открыт / закрыт» определяется по `close_date`; если в этой колонке
хранится плановая дата окончания договора, то досрочно погашенные договоры остаются в выборке как «хорошие» (good):
это размывает сравнение, но не вносит в него смещения.

In [ ]:
def build_ews_population(cinfo, ep_all, rng):
    last_snap = DATA_END_TS - days(EWS_HORIZON + EWS_MIN_DPD - 2)
    snaps = pd.date_range(POP_START, min(last_snap, HES_END_TS), freq=f'{EWS_SNAPSHOT_FREQ_DAYS}D')
    if len(snaps) == 0:
        return None
    c = cinfo[cinfo['from_contracts'].astype(bool)].dropna(subset=['iin', 'open_date']).copy()   # только полный перечень выданных договоров
    c['score_date'] = snaps[rng.integers(0, len(snaps), len(c))]
    c = c[(c['open_date'] < c['score_date']) & ~(c['end_date'] < c['score_date'])
          & ~((c['closed'] == 1) & (c['close_date'] < c['score_date']))]
    if len(c) > EWS_MAX_ROWS:
        c = c.sample(n=EWS_MAX_ROWS, random_state=SEED)
    m = c[['contract', 'score_date']].merge(ep_all[['contract', 'dlq_start', 'max_dpd']], on='contract')
    dlq_now = (m['dlq_start'] < m['score_date']) & (m['dlq_start'] + pd.to_timedelta(m['max_dpd'] - 1, unit='D') >= m['score_date'] - days(1))
    future = ((m['dlq_start'] >= m['score_date']) & (m['dlq_start'] < m['score_date'] + days(EWS_HORIZON))
              & (m['max_dpd'] >= EWS_MIN_DPD))
    flags = pd.DataFrame({'contract': m['contract'], 'dlq_now': dlq_now, 'future': future}).groupby('contract').max()
    c = c.merge(flags, left_on='contract', right_index=True, how='left')
    c = c[~c['dlq_now'].fillna(False).astype(bool)]
    p = pd.DataFrame({'contract': c['contract'].values, 'iin': c['iin'].values, 'score_date': c['score_date'].values,
                      'target': c['future'].fillna(False).astype('int8').values})
    p['ep_uid'] = -1
    p['ep_id'] = np.arange(len(p))
    p['week'] = p['score_date'].dt.to_period('W-SUN').dt.start_time
    p['hist_days'] = (p['score_date'] - HES_START_TS).dt.days
    return p


pop_B, base_B = None, None
if RUN_EWS and contracts is not None:
    with Timer('Популяция B (раннее предупреждение) и переменные договора'):
        pop_B = build_ews_population(cinfo, episodes, np.random.default_rng(SEED))
        if pop_B is not None and len(pop_B):
            base_B = build_baseline(pop_B, episodes, cinfo)
    if pop_B is not None and len(pop_B):
        print(f'Популяция B: {fmt_int(len(pop_B))} срезов договоров, доля «плохих» (bad rate) {pop_B["target"].mean():.2%} '
              f'({fmt_int(int(pop_B["target"].sum()))} выходов на просрочку)')
else:
    print('Часть B пропущена (нужны RUN_EWS = True и таблица `contracts`)')

## 7. События мониторинга ПКБ (PKB) по клиентам популяции: фильтрация, разбор JSON, отметка наших договоров
Сохраняются только строки по клиентам популяции с `created_date` раньше последней даты скоринга: это сокращает
полную таблицу ещё до разбора JSON. Извлекаемые поля:

| поле | ключи JSON |
|---|---|
| `code` | `contract_code` (сравнивается с нашими `contract_number` / `process_ref`) |
| `cred` | `creditor_type.id` (1 = банк, 2 = МФО, микрофинансовая организация) |
| `v_new`, `v_old` | первое найденное из `overdue_sum`, `debt_sum`, `instalment`, `overdue_instalment_count` |
| `f_new` | первый найденный флаг `is_*` / `has_*` / `stopcredit` (статусы, урегулирование задолженности, рефинансирование) |
| `amt`, `debt`, `outst` | `summa_zaima`, `zadolzhenost`, `outstanding_debt` |
| `prolong_days` | `prolongation_end_date − prolongation_start_date` |

#### Как определяется, чей договор
Номер договора из события (`contract_code`) приводится к общему виду (`norm_code`: без пробелов по краям, в верхнем
регистре) и сравнивается с номерами наших договоров `OWN_CODES` (раздел 3: `contract_number` и `process_ref` из
`ovrd_contr` и `contracts`, плюс таблица `OWN_CODES_VAR`, если она задана).

| что в событии | чьё событие | группа признаков |
|---|---|---|
| номер договора совпал с нашим | наш договор | «ПКБ: наши договоры» (суффикс `_own`) |
| номер договора есть, но не наш | договор другого банка или МФО | «ПКБ: другие кредиторы» (суффикс `_oth`) |
| номера договора нет (онлайн-заявки, запросы кредитного отчёта, поиск данных, статусы клиента) | событие о клиенте в целом | считается вместе с другими кредиторами: «ПКБ: другие кредиторы» (без суффикса) |
| правила «частичная просрочка» и «у кого до погашения не менее 80% от ОД займа — появился кредит в другом банке» | правило, настроенное банком: сравнивает нас с другими кредиторами | «ПКБ: правила, настроенные банком» |

Правило считается правилом «с номером договора», если номер есть хотя бы в половине его событий; только такие правила
делятся на «наши договоры» и «другие кредиторы». Если событие такого правила пришло без номера, оно относится к
«другим кредиторам» (суффикс `_oth`). События по нашим договорам повторяют то, что банк уже знает, поэтому
в вывод по правилам (раздел 10) и в модель «Переменные договора + ПКБ» (раздел 15) они не входят: они нужны только для
проверок и справки — таблица сопоставления ниже (если для правила «Изм. DPD» доля событий по нашим договорам близка
к нулю, номера не совпали), справочная колонка раздела 10 и модель «Переменные договора + ПКБ + события по нашим
договорам».

**Внимание:** событие без номера договора может быть вызвано и нашим банком (например, наш запрос кредитного отчёта при взыскании или заявка клиента к нам). ПКБ не передаёт, кто запросил, поэтому такие события отнесены к другим кредиторам. Сильный лифт у «Запрос кредитного отчёта» стоит сверить с датами наших запросов в ПКБ.

**Ограничение:** наши договоры, которых нет в `ovrd_contr` / `contracts` (другие продукты банка, договоры без
просрочки), распознаются как договоры других кредиторов. Задайте `OWN_CODES_VAR` — имя таблицы в кавычках или саму
таблицу со всеми номерами договоров банка по этим клиентам, — и раздел 3 добавит их к `OWN_CODES`.

In [ ]:
def prefilter_hes(hes_src, iins, t_to):
    """Строки `hes` по заданным клиентам со временем события < t_to. hes_src: DataFrame или итерируемый набор DataFrame (по частям)."""
    keys = set(pd.unique(pd.Series(list(iins))))
    keys_alt = keys | {k.lstrip('0') for k in keys}            # bin_iin может храниться без ведущих нулей
    chunks = [hes_src] if isinstance(hes_src, pd.DataFrame) else hes_src
    out = []
    for ch in chunks:
        t = ch[EVENT_TIME_COL]
        if not pd.api.types.is_datetime64_any_dtype(t):
            t = to_dt(t)
        idx = np.flatnonzero((t < t_to).values)
        is_num = pd.api.types.is_numeric_dtype(ch[H_IIN])
        raw = ch[H_IIN].values[idx]
        if is_num:
            m = pd.Series(raw).isin({int(k) for k in keys if k.isdigit()}).values
        else:
            probe = pd.Series(raw[:10_000]).astype(str)
            clean = probe.str.fullmatch(r'\d+').fillna(False).astype(bool).mean() > 0.999
            ser = pd.Series(raw) if clean else pd.Series(raw).astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            m = ser.isin(keys_alt).values
        idx = idx[m]
        cols = [H_IIN, H_RULE, H_OLD, H_NEW] + ([H_ID] if H_ID in ch.columns else [])
        part = ch.iloc[idx][cols].copy()
        part['t'] = t.values[idx]
        out.append(part)
    raw = pd.concat(out, ignore_index=True)
    if H_ID in raw.columns:
        n0 = len(raw)
        raw = raw.drop_duplicates(subset=[H_ID]).drop(columns=[H_ID])
        print(f'  дубликатов по {H_ID} удалено: {fmt_int(n0 - len(raw))}')
    raw['iin'] = norm_iin(raw[H_IIN]).values
    raw = raw[raw['iin'].isin(keys)].drop(columns=[H_IIN])
    return raw.reset_index(drop=True)


def parse_events(raw, own_codes, chunk=1_000_000):
    names = raw[H_RULE].astype('category')
    rmap = map_rule_names(names.cat.categories)
    RULE_MAP.update(rmap)
    rule = names.cat.rename_categories([rmap[c] for c in names.cat.categories]).astype(str).values
    parts = []
    for i0 in range(0, len(raw), chunk):
        ch = raw.iloc[i0:i0 + chunk]
        nw = list(zip(*[_parse_new(s) for s in ch[H_NEW].values]))
        od = list(zip(*[_parse_old(s) for s in ch[H_OLD].values]))
        parts.append(pd.DataFrame({
            'iin': ch['iin'].values, 't': ch['t'].values, 'rule': rule[i0:i0 + chunk],
            'code': np.array(nw[0], dtype=object), 'cred': np.array(nw[1], dtype='float32'),
            'v_new': np.array(nw[2], dtype=float), 'f_new': np.array(nw[3], dtype='float32'),
            'amt': np.array(nw[4], dtype=float), 'debt': np.array(nw[5], dtype=float),
            'outst': np.array(nw[6], dtype=float), 'prolong_days': np.array(nw[7], dtype='float32'),
            'v_old': np.array(od[0], dtype=float), 'f_old': np.array(od[1], dtype='float32')}))
        print(f'  разобрано {fmt_int(min(i0 + chunk, len(raw)))} / {fmt_int(len(raw))}', flush=True)
    ev = pd.concat(parts, ignore_index=True)
    ev['own'] = np.where(ev['code'].isna(), -1, np.where(ev['code'].isin(own_codes), 1, 0)).astype('int8')
    return ev


iins_all = set(pop['iin'])
t_to = pop['score_date'].max()
if pop_B is not None:
    iins_all |= set(pop_B['iin'])
    t_to = max(t_to, pop_B['score_date'].max())
with Timer(f'Фильтрация hes: оставляем {fmt_int(len(iins_all))} клиентов и события до {t_to.date()}'):
    hes_raw = prefilter_hes(hes, iins_all, t_to)
print(f'{fmt_int(len(hes_raw))} строк сохранено')
with Timer('Разбор JSON'):
    ev = parse_events(hes_raw, OWN_CODES)
del hes_raw
gc.collect()

ev = ev[ev['rule'] != 'unsubscribe'].reset_index(drop=True)          # техническое правило
ev['fam'] = ev['rule'].map(FAMILY).fillna('unknown')
code_share = ev.groupby('rule')['own'].apply(lambda x: (x >= 0).mean())
CONTRACT_RULES = set(code_share[code_share >= 0.5].index)               # правила, в событиях которых передаётся contract_code
ev['rkey'] = np.where(ev['rule'].isin(CONTRACT_RULES),
                      ev['rule'] + np.where(ev['own'] == 1, '_own', '_oth'), ev['rule'])
ev['rkey'] = ev['rkey'].astype('category')
ev['oth'] = (ev['own'] != 1) & ~ev['rule'].isin(BANK_RULES)
ev = ev.drop(columns=['code'])
own_diag = ev.groupby('rule').agg(events=('own', 'size'), share_with_contract_code=('own', lambda x: (x >= 0).mean()),
                                  share_own_contract=('own', lambda x: (x == 1).mean()))
own_diag.insert(0, 'description', [RULE_DESC.get(c, '') for c in own_diag.index])
print('Сопоставление с нашими договорами по правилам (если для правила «Изм. DPD» «Доля событий по нашим договорам» '
      'близка к 0, коды договоров не совпали):')
display(own_diag.sort_values('events', ascending=False))

## 8. Признаки ПКБ (PKB)
Для каждого окна наблюдения W (7 / 14 / 30 дней до даты скоринга):
* `n_<rule>_<W>d`: число событий по правилу; правила, в событиях которых есть код договора, разделяются на `_oth`
  (другие кредиторы) и `_own` (наши договоры);
* счётчики по семействам правил (`fam_<family>_oth_n`), общее число событий и число различных сработавших правил
  у других кредиторов;
* значения из JSON: худшее значение дней просрочки (DPD) и число увеличений DPD у других кредиторов, сумма просрочки (максимум,
  увеличения), основной долг (увеличения, уменьшения, чистое изменение), изменение ежемесячного платежа, суммы заявок,
  число заявок, запросов кредитного отчёта и новых займов от МФО, сумма новых займов, дни пролонгации, установка статусов («Лудоман», «Призывник», «Стоп-Кредит», «Банкротство»);
* `days_since_<rule>`: число дней с последнего события по правилу (по всей истории с начала мониторинга).

Группы признаков (как они определяются — раздел 7, «Как определяется, чей договор»): `pkb_other` (другие кредиторы
и события о клиенте без номера договора: в основном новая для банка информация, см. оговорку в разделе 7),
`pkb_bank_rule` (правила, настроенные банком: сравнивают наш договор с другими кредиторами), `pkb_own` (наши договоры:
повторяют внутренние данные банка, используются только для проверок — модель «Переменные договора + ПКБ + события по
нашим договорам» и проверка на утечку данных (leakage)). В таблицах и на графиках они подписаны как «ПКБ: другие
кредиторы», «ПКБ: правила, настроенные банком» и «ПКБ: наши договоры». Переменные договора (`baseline`, раздел 6) —
не признаки ПКБ: в таблицах они подписаны «Переменные договора» и в рейтинги признаков ПКБ не входят.

#### Как читать названия признаков
На графиках и в первой колонке таблиц признак назван коротко по-русски, полное описание — в колонке «Описание признака»
и в «Расшифровке обозначений на графике», код — в колонке «Код в данных». Слова в названии:

| в названии | что означает |
|---|---|
| «др. кредиторы» | договоры других банков и МФО, а также события о клиенте без номера договора (онлайн-заявки, запросы кредитного отчёта, статусы); группа «ПКБ: другие кредиторы». Часть событий без номера может исходить от нашего банка (раздел 7) |
| «наши договоры» | события ПКБ по договорам нашего банка (номер договора совпал с нашим); группа «ПКБ: наши договоры» |
| «у нас» | переменные договора: данные банка, без ПКБ |
| «Клиент:» / «Договор:» | уровень расчёта переменной договора: все наши договоры клиента (по ИИН) / только этот договор |
| «, 7 дн.» / «, 14 дн.» / «, 30 дн.» | окно наблюдения: события за последние 7 / 14 / 30 дней до даты скоринга |
| «: число» / «: всего» | число событий правила / всех правил семейства |
| «: дн. с последнего» | дней с последнего такого события (по всей истории); бин «не было события» — события не было |

Название правила без тега («Онлайн-заявки на кредит: число, 30 дн.») — правило без номера договора: учитываются все его
события. Если у двух разных правил короткие названия совпали, к названию добавляется номер « (1)», « (2)» — один и тот же
во всех таблицах и графиках.
Все признаки ПКБ считаются по клиенту (ИИН): учитываются события по всем договорам клиента, а не только по договору,
вышедшему в просрочку.
Код признака складывается из частей:

| часть имени | что означает |
|---|---|
| `_7d`, `_14d`, `_30d` (`_<W>d`) | окно наблюдения: события за последние 7 / 14 / 30 дней до даты скоринга (время события — колонка `EVENT_TIME_COL`, по умолчанию дата поступления в банк; окна задаются в `WINDOWS`; если `EVENT_LAG_DAYS` > 0 — до даты скоринга минус `EVENT_LAG_DAYS`, от этой же точки считаются `days_since_`) |
| `_oth` | другие кредиторы: код договора в событии не совпал с кодами наших договоров или не передан |
| `_own` | наши договоры: код договора в событии совпал с нашим `contract_number` или `process_ref` |
| `n_<правило>_<W>d` | число событий по правилу за окно. Если в событиях правила обычно (≥ 50% событий) передаётся код договора, счётчик делится на `_oth` и `_own` (`n_dpd_chg_oth_7d`), иначе без суффикса — все события правила (`n_app_online_7d`) |
| `days_since_<правило>` | дней с последнего события по правилу (суффиксы те же), по всей истории событий до даты скоринга; пусто — события не было. `days_since_any_oth` — с последнего события у других кредиторов по любому правилу, кроме `partial_ovd` и `refin_out_80` |
| `fam_<семейство>_oth_n_<W>d` | число событий семейства правил, кроме событий по нашим договорам. Семейства: `credit_hunger` — кредитный голод (`app_online`, `report_req`, `lookup_other`, `new_loan`, `new_loan_refin`, `new_contract`); `delinquency` — просрочка (`dpd_chg`, `ovd_sum_chg`, `debt_terminated`, `loan_assigned`); `debt_burden` — долговая нагрузка (`debt_chg`, `instalment_chg`); `relief` — облегчение условий (`prolong`, `restruct`); `positive` — позитивные события (`settlement`, `loan_closed`); `status` — особые статусы клиента (`st_gambler`, `st_recruit`, `st_stopcredit`, `st_bankrupt`) |
| `all_oth_n_<W>d` / `all_oth_nrules_<W>d` | число всех событий / число различных сработавших правил у других кредиторов |
| `dpd_` | дни просрочки (DPD) из событий `dpd_chg`; `dpd_own_max_<W>d` — худший DPD по всем нашим договорам клиента, переданный в ПКБ |
| `ovdsum_` | сумма просрочки из событий `ovd_sum_chg` |
| `debt_` | основной долг из событий `debt_chg` |
| `inst_` | общий ежемесячный платёж из событий `instalment_chg` |
| `app_` | онлайн-заявки на кредит (`app_online`); `amt` — сумма заявки (`summa_zaima`) |
| `inq_` | запросы кредитного отчёта (`report_req`) |
| `newloan_` | новые займы и договоры (`new_loan`, `new_contract`, `new_loan_refin`); `amt` — сумма займа (`summa_zaima`; в сумму входят только события, где ПКБ передаёт это поле) |
| `prolong_` | пролонгации (`prolong`); `days` — дни пролонгации (дата окончания − дата начала пролонгации) |
| `status_on_` | установка статусов «Лудоман», «Призывник», «Стоп-Кредит», «Банкротство» (флаг в новом значении события = 1); счётчики `n_st_…` и `days_since_st_…` учитывают любое изменение статуса — и установку, и снятие |
| `max` | максимум за окно: худший (наибольший) новый DPD, наибольшая сумма просрочки, наибольшая сумма заявки (`app_amt_max`) |
| `n_up` / `n_down` | число событий, в которых значение выросло / снизилось (новое значение больше / меньше старого; для DPD пустое старое значение считается 0) |
| `up_sum` | сумма приростов: только увеличения, Σ max(новое − старое, 0) |
| `net` | чистое изменение: Σ (новое − старое), увеличения и уменьшения вместе |
| `sum` | сумма за окно: суммы заявок (`app_amt_sum`), суммы новых займов (`newloan_amt_sum`), дни пролонгации (`prolong_days_sum`) |
| `n_ge30` | число событий `dpd_chg`, в которых новый DPD ≥ 30 |
| `n_mfo` | число событий от МФО (тип кредитора `creditor_type.id` = 2) |
| `n` (`all_oth_n`, `fam_..._n`, `status_on_n`) | число событий |
| `bl_` | переменные договора из раздела 6 (данные банка, без ПКБ): `bl_c_` — по этому договору, `bl_i_` — по клиенту (все наши договоры клиента по ИИН) |

Признаки `all_oth_`, `days_since_any_oth`, `fam_`, `dpd_oth_`, `ovdsum_oth_`, `debt_oth_`, `inst_`, `app_`, `inq_`, `newloan_`, `prolong_`, `status_on_`
считаются только по событиям других кредиторов (у `inst_` … `status_on_` суффикса `_oth` в имени нет); правила,
настроенные банком (`partial_ovd`, `refin_out_80`), в них не входят и есть только в счётчиках `n_` и `days_since_`.

**Коды правил** (`RULES` в разделе 1; к коду правила может добавляться суффикс `_oth` / `_own`):

| код | правило ПКБ | код | правило ПКБ |
|---|---|---|---|
| `dpd_chg` | Изменилось количество дней просрочки | `ovd_sum_chg` | Общая сумма просрочки выросла/снизилась |
| `debt_terminated` | Обязательство прекращено без полного погашения | `loan_assigned` | Заём закрыт переуступкой (продан) |
| `debt_chg` | Общий основной долг вырос/снизился | `instalment_chg` | Изменился общий ежемесячный платёж |
| `app_online` | Онлайн-заявка на кредит | `report_req` | Запрос кредитного отчёта |
| `lookup_other` | Другой поставщик (не ПКБ) запросил данные по клиенту | `new_loan` | Появился новый заём |
| `new_loan_refin` | Новый заём с целью рефинансирования | `new_contract` | Появился новый договор |
| `prolong` | Пролонгация займа | `restruct` | Заём реструктурирован / рефинансирован / пересмотрен |
| `settlement` | Урегулирование задолженности | `loan_closed` | Заём закрыт (погашен) |
| `st_gambler` | Статус «Лудоман» (самозапрет на азартные игры) установлен или снят | `st_recruit` | Статус «Призывник» (призыв на воинскую службу) установлен или снят |
| `st_stopcredit` | Статус «Стоп-Кредит» (добровольный запрет на займы) установлен или снят | `st_bankrupt` | Статус «Банкротство» установлен или снят |
| `partial_ovd` | Частичная просрочка (клиент просрочил платёж у нас, но платит другим кредиторам); правило, настроенное банком | `refin_out_80` | «У кого до погашения не менее 80% от ОД займа — появился кредит в другом банке» (правило, настроенное банком; направление 80% уточнить у владельца правила) |
| `collateral_chg` | Изменение по залогу | `unsubscribe` | Отписка: нет действующих договоров (техническое событие, в признаки не входит) |
| `other_<…>` | правило, которого нет в `RULES`: код строится из названия правила ПКБ, описание — само название | | |

In [ ]:
def rkey_group(rk):
    if rk.endswith('_own'):
        return 'pkb_own'
    base = rk[:-4] if rk.endswith('_oth') else rk
    return 'pkb_bank_rule' if base in BANK_RULES else 'pkb_other'


def rkey_desc(rk):
    base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
    tail = ' (другие кредиторы)' if rk.endswith('_oth') else ' (наши договоры)' if rk.endswith('_own') else ''
    return RULE_DESC.get(base, base) + tail


def build_pkb_features(pop, ev, windows=WINDOWS, lag_days=EVENT_LAG_DAYS):
    j = pop[['ep_id', 'iin', 'score_date']].merge(ev, on='iin', how='inner')
    cut = j['score_date'] - days(lag_days)
    j = j[(j['t'] < cut).values].copy()
    j['age'] = ((cut.loc[j.index] - j['t']) / np.timedelta64(1, 'D')).astype('float32')
    feats, meta = {}, {}

    def put(name, s, fill, group, family, rule, window, desc):
        feats[name] = (s, fill)
        meta[name] = dict(group=group, family=family, rule=rule, window=window, description=desc)

    for W in windows:
        sub = j[j['age'] <= W]
        cnt = sub.groupby(['ep_id', 'rkey'], observed=True).size().unstack(fill_value=0)
        for rk in cnt.columns:
            base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
            put(f'n_{rk}_{W}d', cnt[rk], 0, rkey_group(rk), FAMILY.get(base, 'unknown'), base, W,
                f'Число событий: {rkey_desc(rk)}, за последние {W} дн.')
        o = sub[sub['oth']]
        go = o.groupby('ep_id')
        put(f'all_oth_n_{W}d', go.size(), 0, 'pkb_other', 'all', 'all', W, f'Число событий у других кредиторов по всем правилам, кроме правил, настроенных банком, за последние {W} дн.')
        put(f'all_oth_nrules_{W}d', go['rule'].nunique(), 0, 'pkb_other', 'all', 'all', W,
            f'Число различных сработавших правил у других кредиторов (без правил, настроенных банком), за последние {W} дн.')
        for fam in ('credit_hunger', 'delinquency', 'status', 'relief', 'positive', 'debt_burden'):
            x = o[o['fam'] == fam]
            put(f'fam_{fam}_oth_n_{W}d', x.groupby('ep_id').size(), 0, 'pkb_other', fam, 'family', W,
                f'Число событий семейства «{RU_VALUES.get(fam, fam)}» (кроме событий по нашим договорам), за последние {W} дн.')

        def rows(rule, frame=o):
            return frame[frame['rule'] == rule]

        x = rows('dpd_chg')
        put(f'dpd_oth_max_{W}d', x.groupby('ep_id')['v_new'].max(), NAN, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Худшее (максимальное) новое значение DPD у других кредиторов, за последние {W} дн.')
        x_up = x[x['v_new'] > x['v_old'].fillna(0)]
        put(f'dpd_oth_n_up_{W}d', x_up.groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Число увеличений DPD у других кредиторов, за последние {W} дн.')
        put(f'dpd_oth_n_ge30_{W}d', x[x['v_new'] >= 30].groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Число событий изменения DPD у других кредиторов с новым значением DPD ≥ 30, за последние {W} дн.')
        x = rows('ovd_sum_chg')
        dl = x['v_new'] - x['v_old']
        put(f'ovdsum_oth_max_{W}d', x.groupby('ep_id')['v_new'].max(), NAN, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Максимальная сумма просрочки у других кредиторов, за последние {W} дн.')
        put(f'ovdsum_oth_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Число увеличений суммы просрочки у других кредиторов, за последние {W} дн.')
        put(f'ovdsum_oth_up_sum_{W}d', dl.clip(lower=0).groupby(x['ep_id']).sum(), 0, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Рост суммы просрочки у других кредиторов: сумма всех увеличений (уменьшения не вычитаются), за последние {W} дн.')
        x = rows('debt_chg')
        dl = x['v_new'] - x['v_old']
        put(f'debt_oth_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Число увеличений основного долга у других кредиторов, за последние {W} дн.')
        put(f'debt_oth_n_down_{W}d', x[dl < 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Число уменьшений основного долга у других кредиторов, за последние {W} дн.')
        put(f'debt_oth_up_sum_{W}d', dl.clip(lower=0).groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Рост основного долга у других кредиторов: сумма всех увеличений (уменьшения не вычитаются), за последние {W} дн.')
        put(f'debt_oth_net_{W}d', dl.groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Чистое изменение основного долга у других кредиторов (увеличения минус уменьшения), за последние {W} дн.')
        x = rows('instalment_chg')
        dl = x['v_new'] - x['v_old']
        put(f'inst_net_{W}d', dl.groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'instalment_chg', W,
            f'Чистое изменение общего ежемесячного платежа у других кредиторов (увеличения минус уменьшения), за последние {W} дн.')
        put(f'inst_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'instalment_chg', W,
            f'Число увеличений общего ежемесячного платежа у других кредиторов, за последние {W} дн.')
        x = rows('app_online')
        put(f'app_amt_sum_{W}d', x.groupby('ep_id')['amt'].sum(), 0, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Общая сумма онлайн-заявок на кредит, за последние {W} дн.')
        put(f'app_amt_max_{W}d', x.groupby('ep_id')['amt'].max(), NAN, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Максимальная сумма онлайн-заявки на кредит, за последние {W} дн.')
        put(f'app_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Число онлайн-заявок в МФО, за последние {W} дн.')
        x = rows('report_req')
        put(f'inq_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'report_req', W,
            f'Число запросов кредитного отчёта со стороны МФО, за последние {W} дн.')
        x = o[o['rule'].isin(['new_loan', 'new_contract', 'new_loan_refin'])]
        put(f'newloan_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'new_loan', W,
            f'Число событий о новых займах в МФО (правила «новый заём», «новый договор», «заём на рефинансирование»), за последние {W} дн.')
        put(f'newloan_amt_sum_{W}d', x.groupby('ep_id')['amt'].sum(), 0, 'pkb_other', 'credit_hunger', 'new_loan', W,
            f'Сумма новых займов у других кредиторов (только события, где ПКБ передаёт сумму займа summa_zaima), за последние {W} дн.')
        x = rows('prolong')
        put(f'prolong_days_sum_{W}d', x.groupby('ep_id')['prolong_days'].sum(), 0, 'pkb_other', 'relief', 'prolong', W,
            f'Суммарное число дней пролонгации у других кредиторов, за последние {W} дн.')
        x = o[o['rule'].isin(STATUS_RULES) & (o['f_new'] == 1)]
        put(f'status_on_n_{W}d', x.groupby('ep_id').size(), 0, 'pkb_other', 'status', 'status', W,
            f'Число событий установки статуса «Лудоман», «Призывник», «Стоп-Кредит» или «Банкротство» (новое значение флага = 1), за последние {W} дн.')
        own = sub[(sub['own'] == 1) & (sub['rule'] == 'dpd_chg')]
        put(f'dpd_own_max_{W}d', own.groupby('ep_id')['v_new'].max(), NAN, 'pkb_own', 'delinquency', 'dpd_chg', W,
            f'Худшее (максимальное) значение DPD по всем нашим договорам клиента, переданное в ПКБ, за последние {W} дн.')
    rec = j.groupby(['ep_id', 'rkey'], observed=True)['age'].min().unstack()
    for rk in rec.columns:
        base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
        put(f'days_since_{rk}', rec[rk], NAN, rkey_group(rk), FAMILY.get(base, 'unknown'), base, 'all',
            f'Дней с последнего события: {rkey_desc(rk)}, за всю историю (пусто — не было)')
    put('days_since_any_oth', j[j['oth']].groupby('ep_id')['age'].min(), NAN, 'pkb_other', 'all', 'all', 'all',
        'Дней с последнего события у других кредиторов по любому правилу, кроме правил, настроенных банком, за всю историю (пусто — не было)')
    idx = pd.Index(pop['ep_id'].values, name='ep_id')
    cols = {}
    for name, (s, fill) in feats.items():
        v = s.reindex(idx)
        if fill == fill:
            v = v.fillna(fill)
        cols[name] = v.values.astype('float32')
    F = pd.DataFrame(cols, index=idx)
    keep = [c for c in F.columns if F[c].nunique(dropna=False) > 1]
    catalog = pd.DataFrame.from_dict(meta, orient='index').loc[keep]
    return F[keep], catalog


with Timer('Признаки ПКБ, популяция A'):
    F_A, catalog = build_pkb_features(pop, ev)
print(f'{F_A.shape[1]} признаков ПКБ | доля эпизодов хотя бы с одним событием ПКБ до даты скоринга: '
      f'{F_A["days_since_any_oth"].notna().mean():.1%}')
display(catalog.groupby(['group', 'family']).size().unstack(fill_value=0))

## 9. Биннинг, WoE и IV
Числовые признаки: до 20 квантильных бинов (интервалов) или бинов по отдельным значениям; малые бины
(< `MIN_BIN_SHARE`) присоединяются к соседнему бину с наиболее близкой долей «плохих» (bad rate); затем соседние бины
объединяются, пока доля «плохих» не станет монотонной, а затем — до тех пор, пока бинов не станет `MAX_BINS` или меньше.
Пропущенные значения образуют отдельный бин. Категориальные признаки: один бин на категорию, редкие категории
объединяются. WoE (вес свидетельства) и информационная ценность (IV): `WoE = ln(%good / %bad)` со сглаживанием 0,5,
`IV = Σ (%good − %bad) · WoE`, где %good / %bad — доля «хороших» / «плохих» в бине от их общего числа.
Джини (Gini) и KS рассчитываются для признака после биннинга. Признак помечается как *ненадёжный* (unreliable), если
вне его самого крупного бина находится меньше `MIN_BADS_RELIABLE` «плохих» наблюдений (типично для редких событий).

In [ ]:
# Форматирование границы бина для подписи интервала (целые и большие значения выводятся без дробной части).
def _fmt(v):
    if not np.isfinite(v):
        return 'inf'
    return f'{v:,.0f}' if abs(v - round(v)) < 1e-9 or abs(v) >= 100 else f'{v:,.3g}'


# Подбор бинов (интервалов) для одного признака по целевой переменной y.
# Категориальный признак: свой бин для каждой категории с долей >= min_share, остальные -> 'OTHER', пропуски -> 'MISSING'.
# Числовой признак: стартовые бины по квантилям, затем слияние мелких бинов, монотонизация доли «плохих» (bad rate)
# и сокращение числа бинов до max_bins. Последний код всегда отведён под пропуски ('MISSING').
def fit_bins(x, y, max_bins=MAX_BINS, min_share=MIN_BIN_SHARE, monotonic=True, n_init=20):
    x = pd.Series(x, copy=False)
    y = np.asarray(y, dtype=float)
    N = len(y)
    if not pd.api.types.is_numeric_dtype(x) or pd.api.types.is_bool_dtype(x):
        s = x.astype(object)
        s = s.where(s.notna(), '__MISSING__').astype(str)
        vc = s.value_counts()
        cats = [c for c in vc.index if c != '__MISSING__' and vc[c] >= min_share * N]
        return {'kind': 'cat', 'cats': cats, 'n_codes': len(cats) + 2,
                'labels': cats + ['OTHER', 'MISSING']}
    # числовой признак: пропуски выделяются в отдельный бин
    xv = x.values.astype(float)
    miss = np.isnan(xv)
    v, yv = xv[~miss], y[~miss]
    if len(v) == 0:
        return {'kind': 'num', 'edges': np.array([]), 'n_codes': 2, 'labels': ['all', 'MISSING']}
    # стартовые границы: все уникальные значения (если их не больше n_init) или n_init квантилей
    u = np.unique(v)
    if len(u) <= n_init:
        edges = u[:-1]
    else:
        edges = np.unique(np.quantile(v, np.linspace(0, 1, n_init + 1)[1:-1]))
        edges = edges[edges < u[-1]]
    k = len(edges) + 1
    code = np.searchsorted(edges, v, side='left')
    n = list(np.bincount(code, minlength=k).astype(float))
    b = list(np.bincount(code, weights=yv, minlength=k))
    hi = list(edges) + [np.inf]

    def merge(i):                      # объединить бин i с бином i + 1
        n[i] += n[i + 1]; b[i] += b[i + 1]; hi[i] = hi[i + 1]
        del n[i + 1], b[i + 1], hi[i + 1]

    def br(i):                         # доля «плохих» (bad rate) в бине i
        return b[i] / n[i] if n[i] > 0 else 0.0

    # 1) бины с долей меньше min_share сливаются с соседним бином с наиболее близкой долей «плохих»
    min_n = max(min_share * N, 1)
    while len(n) > 1:
        i = int(np.argmin(n))
        if n[i] >= min_n:
            break
        if i == 0:
            merge(0)
        elif i == len(n) - 1:
            merge(i - 1)
        elif abs(br(i) - br(i - 1)) <= abs(br(i) - br(i + 1)):
            merge(i - 1)
        else:
            merge(i)
    # 2) монотонность: направление тренда доли «плохих» берётся из взвешенной линейной регрессии,
    #    соседние бины, нарушающие это направление, сливаются
    if monotonic and len(n) > 2:
        rates = np.array([br(i) for i in range(len(n))])
        sign = np.sign(np.polyfit(np.arange(len(n)), rates, 1, w=np.sqrt(np.array(n)))[0]) or 1.0
        changed = True
        while changed and len(n) > 1:
            changed = False
            for i in range(len(n) - 1):
                if sign * (br(i + 1) - br(i)) < 0:
                    merge(i)
                    changed = True
                    break
    # 3) пока бинов больше max_bins, сливается пара соседних бинов с наиболее близкой долей «плохих»
    while len(n) > max_bins:
        merge(int(np.argmin([abs(br(i + 1) - br(i)) for i in range(len(n) - 1)])))
    edges = np.array(hi[:-1], dtype=float)
    # подписи интервалов: '<= a', '(a, b]', '> b' (служебные метки 'all', 'MISSING' остаются кодами; на русский их переводит ru_labels при показе)
    labels = []
    lo = -np.inf
    for h in hi:
        labels.append(f'<= {_fmt(h)}' if lo == -np.inf and np.isfinite(h) else
                      f'> {_fmt(lo)}' if not np.isfinite(h) and lo != -np.inf else
                      'all' if not np.isfinite(h) else f'({_fmt(lo)}, {_fmt(h)}]')
        lo = h
    return {'kind': 'num', 'edges': edges, 'n_codes': len(edges) + 2, 'labels': labels + ['MISSING']}


# Присвоение кода бина каждому значению по уже подобранной модели бинов.
# Категории: номер категории, k = 'OTHER', k + 1 = 'MISSING'; числа: номер интервала, пропуски — последний код.
def apply_bins(x, model):
    if model['kind'] == 'cat':
        s = pd.Series(x, copy=False).astype(object)
        lut = {c: i for i, c in enumerate(model['cats'])}
        k = len(model['cats'])
        codes = s.astype(str).map(lut).fillna(k).values.astype(int)
        codes[s.isna().values] = k + 1
        return codes
    xv = np.asarray(x, dtype=float)
    codes = np.searchsorted(model['edges'], xv, side='left')
    codes[np.isnan(xv)] = len(model['edges']) + 1
    return codes


# Статистики по бинам: число наблюдений n, «плохих» b, «хороших» g, WoE и вклад бина в IV.
# WoE = ln(доля «хороших» / доля «плохих») со сглаживанием +0.5, чтобы бин без «плохих» не давал бесконечность.
def bin_stats(codes, y, n_codes):
    y = np.asarray(y, dtype=float)
    n = np.bincount(codes, minlength=n_codes).astype(float)
    b = np.bincount(codes, weights=y, minlength=n_codes)
    g = n - b
    ne = n > 0
    k = max(int(ne.sum()), 1)
    B, G = b.sum(), g.sum()
    db = (b + 0.5) / (B + 0.5 * k)
    dg = (g + 0.5) / (G + 0.5 * k)
    woe = np.where(ne, np.log(dg / db), 0.0)
    iv = np.where(ne, (dg - db) * woe, 0.0)
    return n, b, g, woe, iv


def binned_auc(n, b, risk):
    """AUC скора по бинам; risk: одно значение на бин, больше = рискованнее (совпадающие значения учитываются)."""
    g = n - b
    B, G = b.sum(), g.sum()
    if B == 0 or G == 0:
        return NAN
    o = np.argsort(risk, kind='mergesort')
    bs, gs, rs = b[o], g[o], np.asarray(risk)[o]
    auc, cum_g, i = 0.0, 0.0, 0
    while i < len(rs):
        j = i
        while j + 1 < len(rs) and rs[j + 1] == rs[i]:
            j += 1
        bb, gg = bs[i:j + 1].sum(), gs[i:j + 1].sum()
        auc += bb * (cum_g + 0.5 * gg)
        cum_g += gg
        i = j + 1
    return auc / (B * G)


# Статистика Колмогорова–Смирнова (KS) для скора по бинам: максимум расхождения накопленных долей «плохих» и «хороших».
def binned_ks(n, b, risk):
    g = n - b
    if b.sum() == 0 or g.sum() == 0:
        return NAN
    o = np.argsort(-np.asarray(risk), kind='mergesort')
    return float(np.max(np.abs(np.cumsum(b[o]) / b.sum() - np.cumsum(g[o]) / g.sum())))


# Таблица бинов одного признака: подпись бина, n, число «плохих», доля бина, доля «плохих», WoE, IV (пустые бины скрыты).
def bin_table(x, y, model):
    codes = apply_bins(x, model)
    n, b, g, woe, iv = bin_stats(codes, y, model['n_codes'])
    t = pd.DataFrame({'bin': model['labels'], 'n': n, 'bads': b, 'share': n / n.sum(),
                      'bad_rate': np.where(n > 0, b / np.maximum(n, 1), np.nan), 'woe': woe, 'iv': iv})
    return t[t['n'] > 0]


# Сводные метрики одного признака: IV, Джини (Gini) и KS (скор = -WoE), число непустых бинов,
# число «плохих» вне самого крупного бина (мера надёжности) и направление зависимости риска от значения признака.
def feature_iv(x, y, model=None, **kw):
    model = model or fit_bins(x, y, **kw)
    codes = apply_bins(x, model)
    n, b, g, woe, iv = bin_stats(codes, y, model['n_codes'])
    ne = n > 0
    risk = -woe
    big = int(np.argmax(n))
    rate = np.where(ne, b / np.maximum(n, 1), np.nan)
    num_bins = [i for i in range(model['n_codes'] - 1) if ne[i]] if model['kind'] == 'num' else []
    direction = ''
    if len(num_bins) >= 2:
        direction = 'risk up with value' if rate[num_bins[-1]] > rate[num_bins[0]] else 'risk down with value'
    return model, dict(iv=float(iv.sum()), gini=2 * binned_auc(n[ne], b[ne], risk[ne]) - 1,
                       ks=binned_ks(n[ne], b[ne], risk[ne]), n_bins=int(ne.sum()),
                       bads_outside_main_bin=float(b.sum() - b[big]), direction=direction)


# Таблица IV по списку признаков: метрики feature_iv + доли пропусков и ненулевых значений,
# класс силы по IV (strength), флаг надёжности (reliable) и группа признака из каталога (по умолчанию 'baseline').
def compute_iv(X, y, features, catalog=None):
    y = np.asarray(y, dtype=float)
    rows, models = [], {}
    for f in features:
        x = X[f]
        m, r = feature_iv(x, y)
        models[f] = m
        xv = pd.to_numeric(x, errors='coerce') if m['kind'] == 'num' else None
        r.update(feature=f,
                 share_missing=float(pd.isna(x).mean()),
                 share_nonzero=float(((xv != 0) & xv.notna()).mean()) if xv is not None else NAN)
        rows.append(r)
    if not rows:
        return pd.DataFrame(columns=['iv', 'gini', 'ks', 'n_bins', 'bads_outside_main_bin', 'direction', 'share_missing',
                                     'share_nonzero', 'strength', 'reliable', 'group']), models
    t = pd.DataFrame(rows).set_index('feature')
    t['strength'] = t['iv'].map(iv_strength)
    t['reliable'] = t['bads_outside_main_bin'] >= MIN_BADS_RELIABLE
    if catalog is not None:
        t = t.join(catalog, how='left')
    t['group'] = t['group'].fillna('baseline') if 'group' in t else 'baseline'
    return t.sort_values('iv', ascending=False), models


# PSI (индекс стабильности популяции) между двумя распределениями по бинам; eps защищает от log(0).
def psi(expected, actual, eps=1e-4):
    e = np.asarray(expected, float) / max(np.sum(expected), 1)
    a = np.asarray(actual, float) / max(np.sum(actual), 1)
    e, a = np.clip(e, eps, None), np.clip(a, eps, None)
    return float(np.sum((a - e) * np.log(a / e)))

## 10. Влияние правил ПКБ на риск: вывод по каждому правилу
**Главный результат ноутбука.** Одна строка — одно правило мониторинга ПКБ (Первое кредитное бюро), найденное в данных,
даже если у клиентов выборки событий этого правила не было. Сводные признаки по нескольким правилам здесь не участвуют.
Переменные договора (раздел 6) здесь тоже не участвуют: оценивается только сам сигнал ПКБ.

**Термины одной фразой**
* **«Во сколько раз риск выше при событии»** — доля «плохих» среди эпизодов с событием, делённая на долю «плохих» без
  события. 1 — нет разницы, 2 — вдвое чаще доходят до DPD 31+, 0.5 — вдвое реже. Это **не лифт** из раздела 11 и из
  прежних версий: лифт делит долю «плохих» с событием на ОБЩУЮ долю «плохих» и считается за 14 дней, а здесь — на долю
  «плохих» БЕЗ события, окно 30 дней. При одном и том же окне это число всегда дальше от 1, чем лифт.
* **95% ДИ** (доверительный интервал) — диапазон, в котором почти наверняка лежит настоящее значение. Весь ДИ выше 1 —
  событие точно повышает риск; ДИ захватывает 1 — разница может быть случайной.
* **q-значение** — насколько вероятно получить такую разницу случайно, с поправкой на то, что правил много (поправка
  Бенджамини–Хохберга); меньше 0.05 — разница не случайна.
* **IV** (информационная ценность) — насколько признак разделяет «плохих» и «хороших» во всём портфеле с учётом того, как
  часто бывает событие: меньше 0.02 — почти не разделяет, от 0.1 — хорошо.
* **Джини** — насколько хорошо признак или модель ставит «плохих» выше «хороших»: 0 — не лучше случайного порядка,
  1 — идеально.
* **WoE** — значение признака, переведённое в шкалу риска; нужно модели, читать его не обязательно.

**Какие события считаются.** Событие правила — его события по договорам других банков и МФО и события без номера договора
(заявки, запросы, статусы клиента). События по нашим договорам не входят: они повторяют то, что банк и так знает (наш DPD,
наши платежи). Для них есть только справочная колонка «Справочно: риск при событии по нашему договору». Если у правила
события есть только по нашим договорам, вывод — «события только по нашим договорам — не оценивается».

**Что показано по каждому правилу** (популяция A: выход договора в просрочку; «плохой» — эпизод дошёл до DPD 31+).
Первые колонки таблицы — правило, вывод, риск при событии, как часто бывает событие, IV, устойчивость и качество данных;
остальные — подробности:
* как часто бывает событие: число и доля эпизодов, у клиента которых было хотя бы одно событие правила за 30 дней до
  даты скоринга (`RULE_WINDOW`), и сколько среди них «плохих»;
* доля «плохих» с событием и без него и **во сколько раз риск выше (или ниже) при событии** — с 95% ДИ. ДИ считается
  бутстрэпом по клиентам: все эпизоды одного клиента попадают в повтор вместе. То же для окон 7 и 14 дней и по неделям
  дат скоринга — справочно;
* **IV флага** события и **IV лучшего признака правила** (с 95% ДИ) — среди признаков этого правила из раздела 8 (число
  событий, максимум, суммы, дни с последнего события, все окна). Лучший признак выбирается только на **обучающем
  периоде** (ранние 70% дат скоринга, как в разделе 15) и проверяется на **тестовом** (поздние 30%): на тесте он должен
  ранжировать риск так же (Джини на тесте больше 0; проверка — бутстрэпом по клиентам, как у флага события);
* **устойчивость** — во сколько раз риск выше при событии на обучающем и на тестовом периоде: «устойчиво», «ослабло /
  сменило направление на тесте (в пределах случайности)» или «неустойчиво» (изменение не случайно);
* **Джини лучшего признака на тесте** — сила правила самого по себе. **Вклад сверх остальных правил ПКБ** — насколько
  падает Джини модели только на правилах ПКБ (по одному лучшему признаку на правило, логистическая регрессия на WoE),
  если убрать это правило. У похожих правил (например, «Изм. DPD» и «Изм. суммы просрочки») вклад делится: у каждого
  он может быть около нуля, хотя вместе они важны. Поэтому решение о правиле — по «Выводу» и по Джини самого правила,
  а вклад сверх остальных показывает, дублирует ли правило другие;
* **справочно: риск при событии среди эпизодов с другими событиями ПКБ** — то же сравнение только среди клиентов, у
  которых за 30 дней были события других правил ПКБ. Если разница там почти исчезает, событие скорее говорит о том, что
  у клиента есть активная кредитная история, а не о самом событии (пометка в «Выводе»);
* **качество данных** по правилу — провалы загрузки и резкие изменения потока событий из раздела 5.

**Как получается «Вывод»** (пороги — в первой ячейке кода):
1. «нет событий у клиентов выборки» — у клиентов выборки не было ни одного события правила до даты скоринга;
2. «мало данных — оценить нельзя» — событие было меньше чем у 200 эпизодов (`MIN_EVENT_EPISODES`) или среди них меньше
   30 «плохих» (`MIN_EVENT_BADS`) — и наблюдаемых, и ожидаемых, если бы событие не влияло (эпизоды с событием × общая
   доля «плохих»). Так правило, которое сильно снижает риск (у него «плохих» мало именно из-за влияния), оценивается;
3. если разница незначима (q ≥ 0.05, `RULE_ALPHA`, и лучший признак не подтвердился на тесте):
   «нет влияния» — когда весь 95% ДИ риска при событии лежит от 0.87 до 1.15 раза (`RULE_RR_NEGLIGIBLE`), то есть
   заметного влияния точно нет; иначе «влияние не подтверждено: данных мало для точного вывода» — разница может быть
   случайной, но может быть и заметной;
4. «практически нет влияния» — разница не случайна, но ничтожна: IV лучшего признака ниже 0.01 и риск отличается меньше
   чем в 1.15 раза (`RULE_IV_NEGLIGIBLE`, `RULE_RR_NEGLIGIBLE`; так бывает на большой выборке);
5. иначе сила — **большая из двух оценок**: (а) во сколько раз меняется риск у эпизодов с событием — для клиента: от 1.4
   раза — «среднее», от 2 раз — «сильное» (`RULE_RR_MEDIUM`, `RULE_RR_STRONG`; граница 95% ДИ тоже должна быть не
   ближе к 1, чем следующий порог ниже); (б) IV лучшего признака правила за весь период — сколько правило даёт всему
   портфелю: от 0.02 — «среднее», от 0.1 — «сильное» (`RULE_IV_MEDIUM`, `RULE_IV_STRONG`). Редкое событие может сильно
   менять риск клиента, но мало влиять на портфель — тогда в выводе есть пометка «редкое событие». Если 95% ДИ IV
   задевает порог, в выводе написано «на границе среднего и сильного» (или «слабого и среднего»). Шкала IV мягче общей
   шкалы IV раздела 12 (там 0.1–0.3 — «средний» признак): одно правило — узкий сигнал;
6. «слабое (неустойчиво)» — на тестовом периоде эффект ослаб не случайно (разница с обучением значима): сменил
   направление или стал меньше трети эффекта на обучении (проверяется, если в обоих периодах есть хотя бы 100 эпизодов с
   событием и 15 «плохих» среди них);
7. направление: «повышает риск» или «снижает риск», если разница в риске значима; иначе «направление зависит от
   значения». Пометка «на тестовом периоде не подтвердилось» — лучший признак незначим на тесте и риск при событии на
   тесте отличается меньше чем в 1.15 раза.

Правила, настроенные банком («Просрочка только у нас», «Кредит в др. банке, у нас ≥80% ОД»), в «Итоге» показаны отдельно:
«Просрочка только у нас» срабатывает, когда клиент просрочил платёж у нас, поэтому отчасти повторяет то, что банк уже
знает. События без номера договора (заявки, запросы кредитного отчёта, поиск данных) могут быть вызваны и нашим банком:
об этом — оговорка в «Итоге».

Ниже: «Итог» простыми словами, таблица по правилам (от сильного влияния к слабому, внутри — по IV), график и
«Расшифровка обозначений на графике». В Excel это первый лист «Итог по правилам ПКБ» (над таблицей — итог, под ней —
пороги и термины). Для популяции B (раннее предупреждение) такой же вывод — в части B (раздел 17). Лифт правил
с разбивкой на наши договоры и другие кредиторы — в разделе 11.


In [ ]:
# ================================================================== влияние правил ПКБ на риск: вывод по каждому правилу
# Одна строка на правило мониторинга ПКБ, найденное в данных (RULE_MAP). Событие правила — его события по договорам других
# кредиторов и события без номера договора (own != 1, раздел 7); события по НАШИМ договорам (own == 1) — только справочная
# колонка. Переменные договора (раздел 6) здесь не используются. Пороги вывода — в первой ячейке кода (RULE_WINDOW,
# MIN_EVENT_EPISODES, MIN_EVENT_BADS, RULE_ALPHA, RULE_IV_STRONG, RULE_IV_MEDIUM, RULE_RR_STRONG, RULE_RR_MEDIUM,
# RULE_IV_NEGLIGIBLE, RULE_RR_NEGLIGIBLE, RULE_RARE_SHARE, RULE_STAB_KEEP, RULE_BOOT). Раздел только добавляет новые
# таблицы: расчёты других разделов не меняются.
from scipy import sparse as _sp
from matplotlib.ticker import FuncFormatter, FixedLocator, NullLocator
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

_TECH_RULES = ('unsubscribe',)       # техническое правило (отписка): в признаки не входит (раздел 7), не оценивается
_MULTI_RULE_PREFIX = ('newloan_',)   # признаки, собранные сразу из нескольких правил: к одному правилу не относятся
_NO_RULE = ('', 'nan', 'None', 'NaN', '<NA>')   # пустое название правила в hes — не правило
# пометка к выводу по правилам, настроенным банком (BANK_RULES): что правило отчасти повторяет данные банка
_BANK_NOTE = {'partial_ovd': 'правило банка: срабатывает при просрочке у нас, поэтому отчасти повторяет то, что банк уже знает'}

# код вывода -> (порядок в таблице, текст)
_V_ORDER = {'strong': 0, 'medium': 1, 'weak': 2, 'unstable': 3, 'negligible': 4, 'none': 5, 'unproven': 6, 'few': 7,
            'own_only': 8, 'no_events': 9, 'technical': 10}
_V_TEXT = {'strong': 'сильное влияние', 'medium': 'среднее влияние', 'weak': 'слабое влияние',
           'unstable': 'слабое влияние (неустойчиво)', 'negligible': 'практически нет влияния', 'none': 'нет влияния',
           'unproven': 'влияние не подтверждено', 'few': 'мало данных — оценить нельзя',
           'own_only': 'события только по нашим договорам — не оценивается',
           'no_events': 'нет событий у клиентов выборки', 'technical': 'техническое событие — не оценивается'}
_V_NOTE = {'negligible': f'разница не случайна, но меньше чем в {RULE_RR_NEGLIGIBLE:g} раза',
           'unproven': 'данных мало для точного вывода'}
_V_EFFECT = ('strong', 'medium', 'weak', 'unstable')       # выводы «влияет» (с направлением)
_V_JUDGED = _V_EFFECT + ('negligible', 'none')             # выводы, сделанные по достаточным данным
_V_NA = ('technical', 'no_events', 'own_only')             # чисел по событию нет
_V_SHORT = {'negligible': 'практически нет влияния', 'none': 'нет влияния', 'unproven': 'не подтверждено',
            'few': 'мало данных'}                          # короткая пометка на графике (серые строки)
_DIR_TEXT = {1: 'повышает риск', -1: 'снижает риск', 0: 'направление зависит от значения'}
_CLS = {2: 'strong', 1: 'medium', 0: 'weak'}
_CLS_GEN = {0: 'слабого', 1: 'среднего', 2: 'сильного'}

# термины одной фразой: в Excel под таблицей (в тексте раздела 10 — те же определения)
_GLOSSARY = [
    'Термины одной фразой:',
    '«Во сколько раз риск выше при событии» — доля «плохих» среди эпизодов с событием, делённая на долю «плохих» без '
    'события; 1 — нет разницы, 2 — вдвое чаще доходят до плохого исхода, 0.5 — вдвое реже. Это не лифт из раздела 11: '
    'лифт делит на ОБЩУЮ долю «плохих» и считается за 14 дней.',
    '95% ДИ (доверительный интервал) — диапазон, в котором почти наверняка лежит настоящее значение; весь ДИ выше 1 — '
    'событие точно повышает риск, ДИ захватывает 1 — разница может быть случайной.',
    'q-значение — насколько вероятно получить такую разницу случайно, с поправкой на то, что правил много; меньше 0.05 — '
    'разница не случайна.',
    'IV (информационная ценность) — насколько признак разделяет «плохих» и «хороших» во всём портфеле с учётом того, как '
    'часто бывает событие: меньше 0.02 — почти не разделяет, от 0.1 — хорошо.',
    'Джини — насколько хорошо признак или модель ставит «плохих» выше «хороших»: 0 — не лучше случайного порядка, '
    '1 — идеально.',
    'WoE — значение признака, переведённое в шкалу риска; нужно модели, читать его не обязательно.']


def rule_list(ev_):
    """Все правила ПКБ, найденные в данных: коды из RULE_MAP (вся таблица hes, раздел 5, и события выборки, раздел 7)
    и из событий ev_; порядок — как в справочнике RULES, затем остальные по алфавиту. Пустое название правила
    (NaN, 'nan') — не правило."""
    codes = {v for v in (globals().get('RULE_MAP') or {}).values() if isinstance(v, str) and v not in _NO_RULE}
    codes |= {c for c in pd.unique(ev_['rule'].dropna()) if isinstance(c, str) and c not in _NO_RULE}
    order = [c for _, c, _, _ in RULES]
    return [c for c in order if c in codes] + sorted(c for c in codes if c not in order)


def rule_last_event_age(pop_, ev_, rules, lag_days=EVENT_LAG_DAYS, chunk=200_000):
    """Дней от последнего события правила до даты скоринга, по каждому эпизоду и правилу: (age_oth, age_own) — массивы
    эпизоды x правила; age_oth — события других кредиторов и без номера договора (own != 1), age_own — события по нашим
    договорам (own == 1); inf — события не было. События отбираются, как в разделе 8: время события < дата скоринга −
    lag_days. Без соединения таблиц: ключи (клиент, правило, наш договор, время в секундах) сортируются один раз,
    последнее событие до даты скоринга находится двоичным поиском (searchsorted) — быстро и без лишней памяти.
    События без названия правила (NaN) не относятся ни к одному правилу и не учитываются."""
    n, R = len(pop_), len(rules)
    out = [np.full((n, R), np.inf, np.float32), np.full((n, R), np.inf, np.float32)]
    if n == 0 or R == 0 or len(ev_) == 0:
        return out[0], out[1]
    codes, _ = pd.factorize(pd.concat([pd.Series(pop_['iin'].values), pd.Series(ev_['iin'].values)],
                                      ignore_index=True))
    p_cl, e_cl = codes[:n].astype(np.int64), codes[n:].astype(np.int64)
    r_ix = pd.Categorical(ev_['rule'].values, categories=rules).codes.astype(np.int64)   # NaN и 'nan' -> −1
    t = pd.to_datetime(ev_['t']).values
    ok = (r_ix >= 0) & ~np.isnat(t) & (p_cl.size > 0)
    if not ok.any():
        return out[0], out[1]
    t0 = np.datetime64(pd.Timestamp(t[ok].min()).normalize())
    t_s = np.zeros(len(t), np.int64)
    t_s[ok] = (t[ok] - t0) // np.timedelta64(1, 's')
    own = (np.asarray(ev_['own'].values) == 1).astype(np.int64)
    key = np.sort(((((e_cl * R + r_ix) * 2 + own) << 32) | t_s)[ok])
    cut = pd.to_datetime(pop_['score_date']).values - np.timedelta64(int(lag_days), 'D')
    c_s = ((cut - t0) // np.timedelta64(1, 's')).astype(np.int64)
    lo32 = np.int64(0xFFFFFFFF)
    for a in range(0, n, chunk):
        b = min(a + chunk, n)
        cs = c_s[a:b, None]
        for o in (0, 1):
            g = (p_cl[a:b, None] * R + np.arange(R, dtype=np.int64)[None, :]) * 2 + o
            q = (g << 32) | np.clip(cs, 0, lo32)
            pos = np.searchsorted(key, q.ravel(), side='left').reshape(q.shape) - 1
            kk = key[np.maximum(pos, 0)]
            hit = (pos >= 0) & ((kk >> 32) == g) & (cs > 0)
            out[o][a:b] = np.where(hit, (cs - (kk & lo32)) / 86400.0, np.inf)
    return out[0], out[1]


def _bh(p):
    """q-значения Бенджамини–Хохберга (поправка на число проверяемых правил); NaN остаются NaN."""
    p = np.asarray(p, float)
    q = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    m = int(ok.sum())
    if m:
        pv = p[ok]
        o = np.argsort(pv)
        r = pv[o] * m / np.arange(1, m + 1)
        r = np.minimum.accumulate(r[::-1])[::-1]
        qq = np.empty(m)
        qq[o] = np.minimum(r, 1.0)
        q[ok] = qq
    return q


def _ln_rr(n1, b1, n, b):
    # логарифм отношения долей «плохих» (с событием / без события) со сглаживанием 0.5: без log 0 у редких правил
    n0, b0 = n - n1, b - b1
    return np.log((b1 + 0.5) / (n1 + 1.0)) - np.log((b0 + 0.5) / (n0 + 1.0))


def _rr(n1, b1, n, b):
    # отношение долей «плохих»: с событием / без события; NaN — нет эпизодов с событием или без, нет «плохих» без события
    n1, b1, n, b = (np.asarray(v, float) for v in (n1, b1, n, b))
    n0, b0 = n - n1, b - b1
    ok = (n1 > 0) & (n0 > 0) & (b0 > 0)
    return np.where(ok, (b1 / np.where(ok, n1, 1.0)) / (np.where(ok, b0, 1.0) / np.where(ok, n0, 1.0)), np.nan)


def _boot_auc(Wt, P, N):
    """AUC для каждого повтора бутстрэпа: Wt — веса клиентов (клиенты x повторы); P / N — разреженные матрицы числа
    «плохих» / «хороших» эпизодов (уровни скора по возрастанию x клиенты). Совпадающие скоры считаются наполовину."""
    wp = np.asarray(P @ Wt, dtype=float)
    wn = np.asarray(N @ Wt, dtype=float)
    below = np.cumsum(wn, axis=0) - wn
    num = (wp * (below + 0.5 * wn)).sum(axis=0)
    den = wp.sum(axis=0) * wn.sum(axis=0)
    return np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)


def _score_levels(s, y, cl, n_cl):
    # уровни скора (по возрастанию) x клиенты: число «плохих» и «хороших» эпизодов
    u, inv = np.unique(np.asarray(s, float), return_inverse=True)
    P = _sp.csr_matrix((np.asarray(y, float), (inv, cl)), shape=(len(u), n_cl))
    N = _sp.csr_matrix((1.0 - np.asarray(y, float), (inv, cl)), shape=(len(u), n_cl))
    return P, N


def _gini_pt(y, s):
    # Джини = 2 * AUC − 1 (как в разделе 15)
    return 2 * roc_auc_score(y, s) - 1 if 0 < np.sum(y) < len(y) else np.nan


def _iv_from_counts(b, g):
    """IV по бинам из (взвешенных) чисел «плохих» b и «хороших» g (бины x повторы бутстрэпа); сглаживание — как в
    bin_stats (раздел 9), поэтому при весах 1 получается та же IV."""
    ne = (b + g) > 0
    k = np.maximum(ne.sum(axis=0), 1)
    db = (b + 0.5) / (b.sum(axis=0) + 0.5 * k)
    dg = (g + 0.5) / (g.sum(axis=0) + 0.5 * k)
    woe = np.where(ne, np.log(dg / db), 0.0)
    return np.where(ne, (dg - db) * woe, 0.0).sum(axis=0)


def _best_rule_feature(r, F_, catalog_, y_tr, y_te, tr, te):
    """Лучший признак правила r среди его признаков из каталога раздела 8 (кроме событий по нашим договорам и признаков
    из нескольких правил): наибольшая IV на ОБУЧАЮЩЕМ периоде (бины подбираются там же). Возвращает словарь: код,
    IV обучение / тест (бины обучения), WoE обучения для эпизодов обучения и теста; None — у правила нет признаков.
    Значимость на тесте проверяется в rule_verdicts бутстрэпом по клиентам (Джини на тесте больше 0)."""
    cand = [f for f in catalog_.index[(catalog_['rule'] == r) & (catalog_['group'] != 'pkb_own')]
            if f in F_.columns and not f.startswith(_MULTI_RULE_PREFIX)]
    best = None
    for f in cand:
        x_tr = F_[f].values[tr]
        if pd.Series(x_tr).nunique(dropna=False) < 2:
            continue
        m = fit_bins(x_tr, y_tr)
        c_tr = apply_bins(x_tr, m)
        _, _, _, woe, iv = bin_stats(c_tr, y_tr, m['n_codes'])
        if best is None or iv.sum() > best['iv_train'] + 1e-12:
            best = dict(feature=f, model=m, iv_train=float(iv.sum()), woe=woe, c_tr=c_tr)
    if best is None:
        return None
    m, woe = best['model'], best['woe']
    c_te = apply_bins(F_[best['feature']].values[te], m)
    _, _, _, _, iv_te = bin_stats(c_te, y_te, m['n_codes'])
    best.update(iv_test=float(iv_te.sum()), w_tr=woe[best.pop('c_tr')], w_te=woe[c_te])
    return best


def _rule_dq_notes(rules):
    """Пометки о качестве данных по правилу из раздела 5: провалы загрузки (hes_gaps) и резкое изменение потока событий
    после периода выборки (hes_regime). '' — замечаний нет; None — проверки раздела 5 не выполнялись."""
    gaps, reg = globals().get('hes_gaps'), globals().get('hes_regime')
    if globals().get('hes_daily_rule') is None:
        return {r: None for r in rules}
    notes = {r: [] for r in rules}
    if isinstance(gaps, pd.DataFrame) and len(gaps):
        for r, a, b, e in zip(gaps['rule'], gaps['start'], gaps['end'], gaps['until_end']):
            if r in notes:
                notes[r].append(f'провал загрузки {_dq_period(a, b)}' + (' (до конца данных)' if e else ''))
    if isinstance(reg, pd.DataFrame) and len(reg):
        for row in reg.itertuples():
            if row.flag and row.rule in notes:
                notes[row.rule].append(f'после периода выборки: {row.verdict}')
    return {r: '; '.join(v) for r, v in notes.items()}


def _rr_class(rr, lo, hi):
    # сила по риску у эпизодов с событием: 2 — сильное, 1 — среднее, 0 — слабое (точка и граница 95% ДИ, см. RULE_RR_*)
    if not (np.isfinite(rr) and np.isfinite(lo) and np.isfinite(hi)):
        return 0
    if (rr >= RULE_RR_STRONG and lo >= RULE_RR_MEDIUM) or (rr <= 1 / RULE_RR_STRONG and hi <= 1 / RULE_RR_MEDIUM):
        return 2
    if (rr >= RULE_RR_MEDIUM and lo >= RULE_RR_NEGLIGIBLE) or (rr <= 1 / RULE_RR_MEDIUM and hi <= 1 / RULE_RR_NEGLIGIBLE):
        return 1
    return 0


def _iv_class(iv):
    return 2 if iv >= RULE_IV_STRONG else 1 if iv >= RULE_IV_MEDIUM else 0


def rule_verdicts(pop_, F_, catalog_, y_, n_boot=RULE_BOOT, seed=SEED, unit='эпизодов'):
    """Вывод по каждому правилу ПКБ для популяции pop_ (признаки F_ и каталог catalog_ раздела 8, целевая y_ в порядке
    строк F_). Возвращает (таблица: строка — правило, колонки с внутренними именами; словарь модели «все правила
    вместе»: Джини на тесте и его 95% ДИ, граница обучение / тест, число эпизодов и доля «плохих»)."""
    y = np.asarray(y_, float)
    yb = y > 0
    pp = pop_.set_index('ep_id').loc[F_.index].reset_index()
    rules = rule_list(ev)
    R, n = len(rules), len(pp)
    age_oth, age_own = rule_last_event_age(pp, ev, rules)
    f_main = age_oth <= RULE_WINDOW                               # было событие правила за RULE_WINDOW дней
    f_own = age_own <= RULE_WINDOW
    n_hist = np.isfinite(age_oth).sum(axis=0)                     # эпизоды с событием правила за всю историю
    n_hist_own = np.isfinite(age_own).sum(axis=0)                 # ... только по нашим договорам
    # обучение / тест — как в разделе 15: квантиль TRAIN_SHARE по дате скоринга
    sd = pd.to_datetime(pp['score_date'])
    cut = sd.quantile(TRAIN_SHARE)
    tr = (sd <= cut).values
    te = ~tr
    cl, cl_uniq = pd.factorize(pp['iin'])
    C = len(cl_uniq)

    def counts(flag, mask):
        # точные числа: эпизоды с событием, «плохие» среди них, все эпизоды, все «плохие» (в пределах mask)
        fm = flag & mask[:, None]
        return fm.sum(0).astype(float), (fm & yb[:, None]).sum(0).astype(float), float(mask.sum()), float(y[mask].sum())

    def exp_bads(n1_, N, B):
        # «плохих» среди эпизодов с событием, если бы событие не влияло на риск
        return n1_ * B / max(N, 1.0)

    every = np.ones(n, bool)
    n1, b1, N_, B_ = counts(f_main, every)
    n1_tr, b1_tr, N_tr, B_tr = counts(f_main, tr)
    n1_te, b1_te, N_te, B_te = counts(f_main, te)
    rr, rr_tr, rr_te = _rr(n1, b1, N_, B_), _rr(n1_tr, b1_tr, N_tr, B_tr), _rr(n1_te, b1_te, N_te, B_te)
    n_own, b_own, _, _ = counts(f_own, every)
    extra = {}
    for W in RULE_EXTRA_WINDOWS:
        nw, bw, _, _ = counts(age_oth <= W, every)
        extra[W] = (nw / max(N_, 1.0), _rr(nw, bw, N_, B_))
    iv_flag = np.array([bin_stats(f_main[:, k].astype(int), y, 2)[4].sum() if 0 < n1[k] < n else np.nan
                        for k in range(R)])
    # справочно: риск при событии по неделям дат скоринга (недели, где «плохих» с событием — наблюдаемых или ожидаемых без
    # влияния — не меньше MIN_EVENT_BADS)
    wk = (pd.to_datetime(pp['week']).values if 'week' in pp.columns
          else sd.dt.to_period('W-SUN').dt.start_time.values)
    rr_wk = []
    for w_ in np.unique(wk):
        nw, bw, Nw, Bw = counts(f_main, wk == w_)
        rr_wk.append(np.where(np.maximum(bw, exp_bads(nw, Nw, Bw)) >= MIN_EVENT_BADS, _rr(nw, bw, Nw, Bw), np.nan))
    rr_wk = np.vstack(rr_wk) if rr_wk else np.full((0, R), np.nan)
    n_wk = np.isfinite(rr_wk).sum(axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        wk_min = np.where(n_wk > 0, np.nanmin(np.where(np.isfinite(rr_wk), rr_wk, np.inf), axis=0), np.nan)
        wk_max = np.where(n_wk > 0, np.nanmax(np.where(np.isfinite(rr_wk), rr_wk, -np.inf), axis=0), np.nan)
    # справочно: риск при событии среди эпизодов, у клиента которых за то же окно были события ДРУГИХ правил ПКБ
    # (не по нашим договорам): отделяет влияние самого события от того, что у клиента просто есть активная кредитная история
    tech = np.array([r in _TECH_RULES for r in rules])
    f_real = f_main & ~tech[None, :]
    other_any = (f_real.sum(axis=1, dtype=np.int16)[:, None] - f_real) > 0     # эпизоды x правила
    n1_a, b1_a = (f_main & other_any).sum(0).astype(float), (f_main & other_any & yb[:, None]).sum(0).astype(float)
    N_a, B_a = other_any.sum(0).astype(float), (other_any & yb[:, None]).sum(0).astype(float)
    rr_act = _rr(n1_a, b1_a, N_a, B_a)
    act_ok = ((np.minimum(n1_a, N_a - n1_a) >= MIN_EVENT_EPISODES)
              & (np.maximum(b1_a, n1_a * B_a / np.maximum(N_a, 1.0)) >= MIN_EVENT_BADS))
    del other_any, f_real

    # ---- лучший признак правила (выбор на обучающем периоде) и модель только на правилах ПКБ
    y_tr, y_te = y[tr], y[te]
    can_split = min(y_tr.sum(), y_te.sum(), (y_tr == 0).sum(), (y_te == 0).sum()) >= 10
    best = {}
    for r in rules:
        if r in _TECH_RULES or not can_split:
            continue
        bf = _best_rule_feature(r, F_, catalog_, y_tr, y_te, tr, te)
        if bf is not None:
            x_ = F_[bf['feature']].values
            m_all, st_ = feature_iv(x_, y)                        # весь период, бины как в разделе 12
            bf.update(iv_all=st_['iv'], c_all=apply_bins(x_, m_all), n_codes_all=m_all['n_codes'])
            best[r] = bf
    in_model = [r for r in rules if r in best and np.unique(best[r]['w_tr']).size > 1]
    model = {'rules': in_model, 'gini_test': np.nan, 'gini_train': np.nan, 'ci': (np.nan, np.nan), 'cut': cut,
             'n_train': int(tr.sum()), 'n_test': int(te.sum()), 'score_dates': sd.values, 'n_all': int(N_),
             'bad_rate': B_ / max(N_, 1.0)}
    s_full, s_minus = None, {}
    if in_model and can_split:
        Xtr = np.column_stack([best[r]['w_tr'] for r in in_model])
        Xte = np.column_stack([best[r]['w_te'] for r in in_model])
        lr = LogisticRegression(C=1.0, max_iter=2000).fit(Xtr, y_tr)
        s_full = lr.predict_proba(Xte)[:, 1]
        model['gini_test'] = _gini_pt(y_te, s_full)
        model['gini_train'] = _gini_pt(y_tr, lr.predict_proba(Xtr)[:, 1])
        for j, r in enumerate(in_model):                          # модель без правила r (обучается заново)
            keep = [q for q in range(len(in_model)) if q != j]
            if keep:
                s_minus[r] = LogisticRegression(C=1.0, max_iter=2000).fit(Xtr[:, keep], y_tr).predict_proba(
                    Xte[:, keep])[:, 1]
            else:
                s_minus[r] = np.zeros(int(te.sum()))              # без единственного правила модель не различает эпизоды
    drop = {r: model['gini_test'] - _gini_pt(y_te, s) for r, s in s_minus.items()}
    # Джини лучшего признака правила на тесте (один признак, без других правил): WoE обучения, «плохие» — низкий WoE
    g_best_pt = {r: _gini_pt(y_te, -bf['w_te']) for r, bf in best.items()}
    # IV лучшего признака за весь период по клиентам: бины x клиенты («плохие» и «хорошие»), бины зафиксированы
    seg, off, rows_iv, cols_iv = {}, 0, [], []
    for r, bf in best.items():
        seg[r] = (off, off + bf['n_codes_all'])
        rows_iv.append(off + bf.pop('c_all'))
        cols_iv.append(cl)
        off += bf['n_codes_all']
    if best:
        ri, ci_ = np.concatenate(rows_iv), np.concatenate(cols_iv)
        yy = np.tile(yb.astype(float), len(best))
        IVB = _sp.csr_matrix((yy, (ri, ci_)), shape=(off, C))
        IVG = _sp.csr_matrix((1.0 - yy, (ri, ci_)), shape=(off, C))
        del ri, ci_, yy

    # ---- бутстрэп по клиентам: вес клиента ~ Пуассон(1), один вес на все эпизоды клиента; все правила сразу через
    # суммы по клиентам (матрица «клиенты x счётчики»): 95% ДИ отношения рисков, p-значения, изменение эффекта обучение ->
    # тест, 95% ДИ IV лучшего признака, значимость лучшего признака на тесте (Джини > 0), 95% ДИ Джини модели на правилах
    # ПКБ и падения Джини без каждого правила (модели не переобучаются)
    def block(mask):
        fm = f_main & mask[:, None]
        return [fm, fm & yb[:, None], mask[:, None], (mask & yb)[:, None]]

    X = _sp.hstack([_sp.csr_matrix(a.astype(np.float32)) for a in block(tr) + block(te)]).tocsr()   # эпизоды x (4R + 4)
    S = _sp.csr_matrix((np.ones(n, np.float32), (cl, np.arange(n))), shape=(C, n))
    M = (S @ X).T.tocsr()                                                                             # (4R + 4) x клиенты
    lv = {}
    if s_full is not None:
        lv['all'] = _score_levels(s_full, y_te, cl[te], C)
        for r, s in s_minus.items():
            lv[('drop', r)] = _score_levels(s, y_te, cl[te], C)
    for r, bf in best.items():
        lv[('best', r)] = _score_levels(-bf['w_te'], y_te, cl[te], C)
    rng = np.random.default_rng(seed)
    step = max(5, min(50, int(2e7 // max(C, 1))))
    ln_tr_b, ln_te_b, ln_all_b, rr_b, g_all = [], [], [], [], []
    g_drop, g_best_b, iv_b_rep = {r: [] for r in s_minus}, {r: [] for r in best}, {r: [] for r in best}
    done = 0
    while done < n_boot:
        bsz = min(step, n_boot - done)
        Wt = rng.poisson(1.0, size=(C, bsz)).astype(np.float32)
        Z = np.asarray(M @ Wt, dtype=float)                                                          # (4R + 4) x повторы
        k1, k1y, kn, kb = Z[:R], Z[R:2 * R], Z[2 * R], Z[2 * R + 1]                                 # обучение
        e1, e1y, en, eb = Z[2 * R + 2:3 * R + 2], Z[3 * R + 2:4 * R + 2], Z[4 * R + 2], Z[4 * R + 3]   # тест
        ln_tr_b.append(_ln_rr(k1, k1y, kn, kb))
        ln_te_b.append(_ln_rr(e1, e1y, en, eb))
        ln_all_b.append(_ln_rr(k1 + e1, k1y + e1y, kn + en, kb + eb))
        rr_b.append(_rr(k1 + e1, k1y + e1y, kn + en, kb + eb))
        if 'all' in lv:
            ga = 2 * _boot_auc(Wt, *lv['all']) - 1
            g_all.append(ga)
            for r in s_minus:
                g_drop[r].append(ga - (2 * _boot_auc(Wt, *lv[('drop', r)]) - 1))
        for r in best:
            g_best_b[r].append(2 * _boot_auc(Wt, *lv[('best', r)]) - 1)
        if best:
            bb, gg = np.asarray(IVB @ Wt, dtype=float), np.asarray(IVG @ Wt, dtype=float)
            for r, (a_, b_) in seg.items():
                iv_b_rep[r].append(_iv_from_counts(bb[a_:b_], gg[a_:b_]))
        done += bsz
    ln_tr_b, ln_te_b, ln_all_b, rr_b = (np.hstack(v) for v in (ln_tr_b, ln_te_b, ln_all_b, rr_b))  # правила x повторы
    ok_ci = np.isfinite(rr_b).sum(axis=1) >= 0.9 * n_boot
    rr_bf = np.where(np.isfinite(rr_b), rr_b, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        rr_lo = np.where(ok_ci, np.nanpercentile(rr_bf, 2.5, axis=1), np.nan)
        rr_hi = np.where(ok_ci, np.nanpercentile(rr_bf, 97.5, axis=1), np.nan)
    # p-значение: z = ln(отношение рисков) / его стандартная ошибка по бутстрэпу (учитывает повторы одного клиента)
    se = ln_all_b.std(axis=1)
    testable = (n1 > 0) & (n1 < N_) & ~tech
    with np.errstate(divide='ignore', invalid='ignore'):
        z = _ln_rr(n1, b1, N_, B_) / se
    p = np.where(testable & (se > 0), np.array([math.erfc(abs(v) / math.sqrt(2)) if np.isfinite(v) else np.nan
                                                for v in z]), np.nan)
    q = _bh(p)
    # лучший признак на тесте: Джини (порядок риска с обучения) больше 0 — односторонняя проверка, стандартная ошибка —
    # по бутстрэпу клиентов (все эпизоды клиента вместе), как у флага события
    p_best = np.full(R, np.nan)
    g_best, g_best_lo, g_best_hi = np.full(R, np.nan), np.full(R, np.nan), np.full(R, np.nan)
    iv_lo, iv_hi = np.full(R, np.nan), np.full(R, np.nan)
    for k, r in enumerate(rules):
        if r not in best:
            continue
        gb = np.hstack(g_best_b[r])
        g_best[k] = g_best_pt[r]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            sg = float(np.nanstd(gb)) if np.isfinite(gb).sum() >= 2 else np.nan
            if np.isfinite(gb).any():
                g_best_lo[k], g_best_hi[k] = np.nanpercentile(gb, [2.5, 97.5])
        if np.isfinite(g_best[k]) and np.isfinite(sg) and sg > 0:
            p_best[k] = 0.5 * math.erfc(g_best[k] / (sg * math.sqrt(2)))
        # IV по бутстрэпу в среднем завышена (случайный шум добавляет IV): ДИ сдвигается на это завышение
        ivr = np.hstack(iv_b_rep[r])
        iv_lo[k], iv_hi[k] = np.clip(np.percentile(ivr, [2.5, 97.5]) - (ivr.mean() - best[r]['iv_all']), 0.0, None)
    q_best = _bh(p_best)
    if g_all:
        model['ci'] = tuple(np.nanpercentile(np.hstack(g_all), [2.5, 97.5]))
    drop_ci = {r: tuple(np.nanpercentile(np.hstack(v), [2.5, 97.5])) for r, v in g_drop.items()}
    # устойчивость: эффект на тесте против эффекта на обучении (при достаточном числе событий в обоих периодах)
    d_se = (ln_te_b - ln_tr_b).std(axis=1)
    l_tr, l_te = _ln_rr(n1_tr, b1_tr, N_tr, B_tr), _ln_rr(n1_te, b1_te, N_te, B_te)
    enough = ((np.minimum(n1_tr, n1_te) >= MIN_EVENT_EPISODES / 2)
              & (np.minimum(np.maximum(b1_tr, exp_bads(n1_tr, N_tr, B_tr)), np.maximum(b1_te, exp_bads(n1_te, N_te, B_te)))
                 >= MIN_EVENT_BADS / 2)
              & ((N_tr - n1_tr) > 0) & ((N_te - n1_te) > 0))
    ln_neg = np.log(RULE_RR_NEGLIGIBLE)
    with np.errstate(divide='ignore', invalid='ignore'):
        changed = np.abs(l_te - l_tr) > 1.96 * d_se
        weaker = (np.abs(l_tr) >= ln_neg) & (l_te / l_tr < RULE_STAB_KEEP)
        kept = ((np.abs(l_tr) < ln_neg) & (np.abs(l_te) < ln_neg)) | ((l_tr * l_te > 0) & (l_te / l_tr >= RULE_STAB_KEEP))
    unstable = enough & changed & weaker

    # ---- вывод по правилу
    dq = _rule_dq_notes(rules)
    rows = []
    for k, r in enumerate(rules):
        bf = best.get(r)
        iv_b = bf['iv_all'] if bf is not None else iv_flag[k]
        share = n1[k] / max(N_, 1.0)
        rr_cls = _rr_class(rr[k], rr_lo[k], rr_hi[k])
        iv_cls = _iv_class(iv_b) if np.isfinite(iv_b) else 0
        border = ''
        if r in _TECH_RULES:
            v = 'technical'
        elif n_hist[k] == 0:
            v = 'own_only' if n_hist_own[k] > 0 else 'no_events'
        elif (min(n1[k], N_ - n1[k]) < MIN_EVENT_EPISODES or max(b1[k], exp_bads(n1[k], N_, B_)) < MIN_EVENT_BADS
              or B_ - b1[k] < MIN_EVENT_BADS):
            v = 'few'
        else:
            sig = (np.isfinite(q[k]) and q[k] < RULE_ALPHA) or (np.isfinite(q_best[k]) and q_best[k] < RULE_ALPHA)
            tiny = ((not np.isfinite(iv_b) or iv_b < RULE_IV_NEGLIGIBLE)
                    and (not np.isfinite(rr[k]) or 1 / RULE_RR_NEGLIGIBLE < rr[k] < RULE_RR_NEGLIGIBLE))
            if not sig:
                # «нет влияния» — только если весь 95% ДИ риска при событии в пределах 1/1.15–1.15 раза (и IV мала);
                # иначе разница может быть и случайной, и заметной: «влияние не подтверждено»
                inside = (np.isfinite(rr_lo[k]) and np.isfinite(rr_hi[k]) and rr_lo[k] >= 1 / RULE_RR_NEGLIGIBLE
                          and rr_hi[k] <= RULE_RR_NEGLIGIBLE and (not np.isfinite(iv_b) or iv_b < RULE_IV_MEDIUM))
                v = 'none' if inside else 'unproven'
            elif tiny:
                v = 'negligible'
            elif unstable[k]:
                v = 'unstable'
            else:
                cls = max(iv_cls, rr_cls)                         # большая из двух оценок силы
                v = _CLS[cls]
                if r in best and np.isfinite(iv_lo[k]):           # IV близко к порогу: класс мог бы быть соседним
                    down = cls >= 1 and rr_cls < cls and iv_lo[k] < (RULE_IV_STRONG if cls == 2 else RULE_IV_MEDIUM)
                    up = cls < 2 and iv_hi[k] >= (RULE_IV_STRONG if cls == 1 else RULE_IV_MEDIUM)
                    if down and up:
                        border = 'оценка силы неточная'
                    elif down:
                        border = f'на границе {_CLS_GEN[cls - 1]} и {_CLS_GEN[cls]}'
                    elif up:
                        border = f'на границе {_CLS_GEN[cls]} и {_CLS_GEN[cls + 1]}'
        direction = 0
        if v in _V_EFFECT and np.isfinite(rr[k]) and ((np.isfinite(q[k]) and q[k] < RULE_ALPHA) or rr_cls >= 1):
            direction = 1 if rr[k] > 1 else -1
        if v in _V_NA or not (np.isfinite(rr_tr[k]) and np.isfinite(rr_te[k])):
            stab = ''
        elif unstable[k]:
            stab = 'неустойчиво'
        elif not enough[k]:
            stab = 'мало данных для проверки'
        elif kept[k]:
            stab = 'устойчиво'
        elif np.abs(l_tr[k]) < ln_neg:
            stab = 'на тесте эффект сильнее' + ('' if changed[k] else ' (в пределах случайности)')
        elif l_te[k] / l_tr[k] < 0:
            stab = 'сменило направление на тесте (в пределах случайности)'
        else:
            stab = 'ослабло на тесте (в пределах случайности)'
        # пометки к выводу (о редкости события — только в тексте вывода: в «Итоге» доля события показана числом)
        notes, rare = [], []
        if v in _V_EFFECT and share < RULE_RARE_SHARE:
            rare.append(f'редкое событие: {share:.1%} {unit}')
        if (v in ('strong', 'medium', 'weak') and np.isfinite(q_best[k]) and q_best[k] >= RULE_ALPHA
                and np.isfinite(rr_te[k]) and 1 / RULE_RR_NEGLIGIBLE < rr_te[k] < RULE_RR_NEGLIGIBLE):
            notes.append('на тестовом периоде не подтвердилось')
        if (v in _V_EFFECT and act_ok[k] and np.isfinite(rr_act[k]) and rr_act[k] > 0 and np.isfinite(rr[k])
                and abs(np.log(rr[k])) >= ln_neg and np.log(rr_act[k]) / np.log(rr[k]) < 0.5):
            notes.append('похоже на признак активной кредитной истории, а не самого события')
        if v in _V_EFFECT and r in _BANK_NOTE:
            notes.append(_BANK_NOTE[r])
        head = _V_TEXT[v] + (f' ({border})' if border else '')
        head += f': {_DIR_TEXT[direction]}' if v in _V_EFFECT else (f': {_V_NOTE[v]}' if v in _V_NOTE else '')
        rows.append(dict(
            rule=r, n=N_, n_event=n1[k], share_event=share, bads_event=b1[k], exp_bads_event=exp_bads(n1[k], N_, B_),
            br_event=b1[k] / n1[k] if n1[k] else np.nan,
            br_no=(B_ - b1[k]) / (N_ - n1[k]) if N_ > n1[k] else np.nan,
            rr=rr[k], rr_lo=rr_lo[k], rr_hi=rr_hi[k], p=p[k], q=q[k],
            **{f'share_{W}': extra[W][0][k] for W in RULE_EXTRA_WINDOWS},
            **{f'rr_{W}': extra[W][1][k] for W in RULE_EXTRA_WINDOWS},
            rr_week_min=wk_min[k], rr_week_max=wk_max[k], n_weeks=int(n_wk[k]),
            iv_flag=iv_flag[k], best_feature=bf['feature'] if bf else None,
            iv_best_all=bf['iv_all'] if bf else np.nan, iv_best_lo=iv_lo[k], iv_best_hi=iv_hi[k],
            iv_best_train=bf['iv_train'] if bf else np.nan, iv_best_test=bf['iv_test'] if bf else np.nan,
            gini_best_test=g_best[k], gini_best_lo=g_best_lo[k], gini_best_hi=g_best_hi[k],
            p_best_test=p_best[k], q_best_test=q_best[k],
            rr_train=rr_tr[k], rr_test=rr_te[k], stability=stab,
            gini_drop=drop.get(r, np.nan), gini_drop_lo=drop_ci.get(r, (np.nan,) * 2)[0],
            gini_drop_hi=drop_ci.get(r, (np.nan,) * 2)[1],
            share_event_active=n1_a[k] / max(N_, 1.0), rr_active=rr_act[k] if act_ok[k] else np.nan,
            share_own=n_own[k] / max(N_, 1.0), rr_own=_rr(n_own[k], b_own[k], N_, B_) if n_own[k] else np.nan,
            dq_note=dq[r], strength_iv=iv_cls if v in ('strong', 'medium', 'weak') else np.nan,
            strength_rr=rr_cls if v in ('strong', 'medium', 'weak') else np.nan,
            verdict=v, direction=direction, verdict_text=head + ''.join('; ' + s for s in rare + notes),
            verdict_notes='; '.join(notes)))
    res = pd.DataFrame(rows).set_index('rule')
    res.index.name = None
    srt = pd.DataFrame({'o': res['verdict'].map(_V_ORDER), 'iv': -res['iv_best_all'].fillna(res['iv_flag']).fillna(-1),
                        'cov': -res['share_event']}, index=res.index)
    res = res.loc[srt.sort_values(['o', 'iv', 'cov'], kind='mergesort').index]
    return res, model


# ------------------------------------------------------------------ показ: таблица, график, итог
def _f4(x):
    # IV — четыре знака: 0.0199 не превращается в «0.020» рядом с порогом 0.02
    return f'{x:.4f}' if np.isfinite(x) else '—'


def _times(x):
    # «в 1.65 раза выше» / «в 1.39 раза ниже» (x — отношение долей «плохих»)
    if not np.isfinite(x):
        return '—'
    if x == 0:
        return '«плохих» с событием нет'
    return f'в {x:.2f} раза выше' if x >= 1 else f'в {1 / x:.2f} раза ниже'


def _rr_ci_text(rr, lo, hi):
    # «в 1.65 раза выше (1.57–1.73)»; ниже 1 — в той же записи «во сколько раз ниже»: «в 1.39 раза ниже (1.32–1.47)»;
    # ДИ по обе стороны от 1 — словами: «в 1.13 раза выше (ДИ: от 1.15 раза ниже до 1.37 раза выше)»
    s = _times(rr)
    if s == '—' or not (np.isfinite(lo) and np.isfinite(hi)):
        return s
    if lo < 1 < hi:
        low = f'от {1 / lo:.2f} раза ниже' if lo > 0 else 'от нуля'
        return f'{s} (ДИ: {low} до {hi:.2f} раза выше)'
    if rr >= 1:
        return f'{s} ({lo:.2f}–{hi:.2f})'
    a = 1 / hi if hi > 0 else np.inf
    b = 1 / lo if lo > 0 else np.inf
    return f'{s} ({a:.2f}–{b:.2f})' if np.isfinite(b) else f'{s} (ДИ: риск ниже как минимум в {a:.2f} раза)'


def _r3(x):
    # три знака после точки без «−0.000»
    return f'{round(float(x), 3) + 0.0:.3f}' if np.isfinite(x) else '—'


def _sd_word(unit):
    # «даты скоринга» (эпизоды, популяция A) или «даты среза» (срезы договоров, популяция B)
    return 'даты среза' if unit == 'срезов' else 'даты скоринга'


def _q_text(q):
    return '—' if not np.isfinite(q) else '< 0.001' if q < 0.001 else f'{q:.3f}'


def _U(unit):
    return unit[:1].upper() + unit[1:]


def rule_table_screen(res, unit='эпизодов'):
    """Таблица для экрана: строка — правило, русские колонки; сначала правило и вывод, коды — в последних колонках."""
    na = lambda v: v in _V_NA                                                          # noqa: E731
    t = pd.DataFrame({
        'Правило': code_names(res.index),
        'Вывод': list(res['verdict_text']),
        'Риск при событии (95% ДИ)': [_rr_ci_text(a, b, c) for a, b, c in zip(res['rr'], res['rr_lo'], res['rr_hi'])],
        f'{_U(unit)} с событием за {RULE_WINDOW} дн. (доля)': [
            '—' if na(v) else f'{fmt_int(a)} ({s:.1%})' for a, s, v in zip(res['n_event'], res['share_event'], res['verdict'])],
        'IV лучшего признака за весь период (95% ДИ)': [
            '—' if not isinstance(f, str) else f'{_f4(a)} ({_f4(b)}–{_f4(c)})'
            for f, a, b, c in zip(res['best_feature'], res['iv_best_all'], res['iv_best_lo'], res['iv_best_hi'])],
        'Устойчивость: риск при событии, обучение → тест, раз': [
            '—' if not s else f'{a:.2f} → {b:.2f}: {s}' for a, b, s in zip(res['rr_train'], res['rr_test'], res['stability'])],
        'Качество данных (раздел 5)': ['не проверялось' if not isinstance(d, str) else d or 'замечаний нет'
                                       for d in res['dq_note']],
        'Что означает': [code_desc(r) for r in res.index],
        '«Плохих» среди них': ['—' if na(v) else fmt_int(b) for b, v in zip(res['bads_event'], res['verdict'])],
        'Доля «плохих»: с событием / без события': [
            '—' if na(v) or not np.isfinite(a) else f'{a:.1%} / {b:.1%}'
            for a, b, v in zip(res['br_event'], res['br_no'], res['verdict'])],
        'Значимость: q-значение флага / лучшего признака на тесте': [
            f'{_q_text(a)} / {_q_text(b)}' if np.isfinite(a) or np.isfinite(b) else '—'
            for a, b in zip(res['q'], res['q_best_test'])],
        **{f'Риск при событии за {W} дн., раз': [f'{x:.2f}' if np.isfinite(x) else '—' for x in res[f'rr_{W}']]
           for W in RULE_EXTRA_WINDOWS},
        'Риск при событии по неделям: от – до, раз (число недель)': [
            f'{a:.2f}–{b:.2f} ({k})' if k >= 2 else '—'
            for a, b, k in zip(res['rr_week_min'], res['rr_week_max'], res['n_weeks'])],
        f'IV флага (событие за {RULE_WINDOW} дн.)': [_f4(x) for x in res['iv_flag']],
        'Лучший признак правила': ['—' if not isinstance(f, str) else code_name(f) for f in res['best_feature']],
        'IV лучшего признака: обучение / тест': [
            '—' if not isinstance(f, str) else f'{_f4(a)} / {_f4(b)}'
            for f, a, b in zip(res['best_feature'], res['iv_best_train'], res['iv_best_test'])],
        'Джини лучшего признака на тесте (95% ДИ)': [
            '—' if not np.isfinite(g) else f'{_r3(g)} ({_r3(a)} … {_r3(b)})'
            for g, a, b in zip(res['gini_best_test'], res['gini_best_lo'], res['gini_best_hi'])],
        'Вклад сверх остальных правил ПКБ: падение Джини без правила (95% ДИ)': [
            '—' if not np.isfinite(d) else f'{_r3(d)} ({_r3(a)} … {_r3(b)})'
            for d, a, b in zip(res['gini_drop'], res['gini_drop_lo'], res['gini_drop_hi'])],
        f'Справочно: риск при событии среди {unit} с другими событиями ПКБ': [
            '—' if not np.isfinite(x) else _times(x) for x in res['rr_active']],
        'Справочно: риск при событии по нашему договору': [
            '—' if not np.isfinite(x) else f'{_times(x)} (событие у {s:.1%})' for x, s in zip(res['rr_own'], res['share_own'])],
        'Код лучшего признака в данных': ['—' if not isinstance(f, str) else f for f in res['best_feature']],
        'Код правила в данных': list(res.index)}, index=pd.RangeIndex(1, len(res) + 1, name='№'))
    return t


def rule_table_excel(res, unit='эпизодов'):
    """Та же таблица для Excel: числа — числами (доли — от 0 до 1), русские колонки; сначала правило и вывод, коды —
    в последних колонках."""
    num = lambda c: res[c].astype(float).values                                      # noqa: E731
    U = _U(unit)
    cols = {'Правило': code_names(res.index), 'Вывод': list(res['verdict_text']),
            'Во сколько раз риск выше при событии': num('rr'), '95% ДИ: от': num('rr_lo'), '95% ДИ: до': num('rr_hi'),
            f'Доля {unit} с событием': num('share_event'), f'{U} с событием за {RULE_WINDOW} дн.': num('n_event'),
            '«Плохих» с событием': num('bads_event'),
            '«Плохих» с событием, если бы событие не влияло': num('exp_bads_event'),
            'IV лучшего признака: весь период': num('iv_best_all'), 'IV лучшего признака: 95% ДИ от': num('iv_best_lo'),
            'IV лучшего признака: 95% ДИ до': num('iv_best_hi'), 'Устойчивость': list(res['stability']),
            'Качество данных (раздел 5)': ['не проверялось' if not isinstance(d, str) else d or 'замечаний нет'
                                           for d in res['dq_note']],
            'Что означает': [code_desc(r) for r in res.index], f'{U} всего': num('n'),
            'Доля «плохих» с событием': num('br_event'), 'Доля «плохих» без события': num('br_no'),
            'p-значение': num('p'), 'q-значение (поправка Бенджамини–Хохберга)': num('q')}
    for W in RULE_EXTRA_WINDOWS:
        cols[f'Доля {unit} с событием за {W} дн.'] = num(f'share_{W}')
        cols[f'Во сколько раз риск выше при событии за {W} дн.'] = num(f'rr_{W}')
    cols.update({
        'Во сколько раз риск выше при событии по неделям: от': num('rr_week_min'),
        'Во сколько раз риск выше при событии по неделям: до': num('rr_week_max'),
        'Число недель с достаточными данными': num('n_weeks'),
        f'IV флага (событие за {RULE_WINDOW} дн.)': num('iv_flag'),
        'Лучший признак правила': ['' if not isinstance(f, str) else code_name(f) for f in res['best_feature']],
        'IV лучшего признака: обучение': num('iv_best_train'), 'IV лучшего признака: тест': num('iv_best_test'),
        'Джини лучшего признака на тесте': num('gini_best_test'),
        'Джини лучшего признака на тесте: 95% ДИ от': num('gini_best_lo'),
        'Джини лучшего признака на тесте: 95% ДИ до': num('gini_best_hi'),
        'p-значение лучшего признака на тесте (Джини > 0)': num('p_best_test'),
        'q-значение лучшего признака на тесте': num('q_best_test'),
        'Во сколько раз риск выше при событии: обучение': num('rr_train'),
        'Во сколько раз риск выше при событии: тест': num('rr_test'),
        'Падение Джини без правила (вклад сверх остальных правил ПКБ)': num('gini_drop'),
        'Падение Джини: 95% ДИ от': num('gini_drop_lo'), 'Падение Джини: 95% ДИ до': num('gini_drop_hi'),
        f'Справочно: доля {unit} с событием и с другими событиями ПКБ': num('share_event_active'),
        f'Справочно: во сколько раз риск выше при событии среди {unit} с другими событиями ПКБ': num('rr_active'),
        f'Справочно: доля {unit} с событием по нашему договору': num('share_own'),
        'Справочно: во сколько раз риск выше при событии по нашему договору': num('rr_own'),
        'Код лучшего признака в данных': ['' if not isinstance(f, str) else f for f in res['best_feature']],
        'Код правила в данных': list(res.index)})
    return pd.DataFrame(cols)


def plot_rule_verdicts(res, title, unit='эпизодов'):
    """Слева — во сколько раз риск выше (ниже) при событии с 95% ДИ (логарифмическая шкала, 1 — нет разницы; точечный
    пунктир — пороги силы по риску), справа — IV лучшего признака правила с порогами силы по IV; цвет — направление,
    серый — нет влияния / не подтверждено / мало данных / нет событий; полый кружок — неустойчиво. Правила — в порядке
    таблицы (первая строка таблицы — вверху)."""
    n = len(res)
    if n == 0:
        return
    names = code_names(res.index)
    ypos = np.arange(n)[::-1]
    effect = res['verdict'].isin(_V_EFFECT).values
    dirs = res['direction'].values
    col = np.where(~effect, PAL['gray'], np.where(dirs > 0, PAL['red'], np.where(dirs < 0, PAL['blue'], PAL['violet'])))
    rr, lo, hi = (res[c].astype(float).values for c in ('rr', 'rr_lo', 'rr_hi'))
    vals = np.concatenate([v[np.isfinite(v) & (v > 0)] for v in (rr, lo, hi)])
    xmin = float(np.clip(min(0.5, vals.min() * 0.85) if len(vals) else 0.5, 0.1, 0.5))
    xmax = float(np.clip(max(2.0, vals.max() * 1.15) if len(vals) else 2.0, 2.0, 10.0))
    xtxt = xmax * 1.02                                             # подписи значений — справа от оси
    h = 0.36 * n + 2.6
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13.5, h), sharey=True, gridspec_kw={'width_ratios': [1.35, 1]})
    a1.set_xscale('log')
    a1.axvline(1, color=PAL['muted'], lw=0.9, zorder=1)
    for thr in (1 / RULE_RR_STRONG, 1 / RULE_RR_MEDIUM, RULE_RR_MEDIUM, RULE_RR_STRONG):
        if xmin < thr < xmax:
            a1.axvline(thr, color=PAL['muted'], lw=0.9, ls=(0, (1, 2)), zorder=1)
    for y_, r_, l_, h_, c_, v_ in zip(ypos, rr, lo, hi, col, res['verdict']):
        if v_ == 'technical':
            a1.text(1.0, y_, '  не оценивается', va='center', fontsize=7, color=PAL['muted'])
            continue
        if v_ in ('no_events', 'own_only') or not np.isfinite(r_):
            a1.text(1.0, y_, {'no_events': '  нет событий', 'own_only': '  только наши договоры'}.get(
                v_, '  не посчитать: мало событий'), va='center', fontsize=7, color=PAL['muted'])
            continue
        pt = float(np.clip(r_, xmin, xmax))
        l2 = float(np.clip(l_, xmin, xmax)) if np.isfinite(l_) and l_ > 0 else (xmin if np.isfinite(l_) else pt)
        h2 = float(np.clip(h_, xmin, xmax)) if np.isfinite(h_) else pt
        a1.plot([l2, h2], [y_, y_], color=c_, lw=1.8, solid_capstyle='butt', zorder=2)
        if np.isfinite(l_) and l_ < xmin:                          # ДИ уходит за левый край оси
            a1.plot([xmin], [y_], marker='<', color=c_, ms=5, zorder=3, clip_on=False)
        if np.isfinite(h_) and h_ > xmax:                          # ... за правый край
            a1.plot([xmax], [y_], marker='>', color=c_, ms=5, zorder=3, clip_on=False)
        a1.plot([pt], [y_], 'o', color=c_, mfc='white' if v_ == 'unstable' else c_, ms=6, mew=1.5, zorder=3)
        a1.text(xtxt, y_, f'{r_:.2f}', va='center', fontsize=7, color=PAL['muted'])
    ticks = [t_ for t_ in (0.1, 0.2, 0.25, 0.33, 0.5, 0.67, 0.8, 1, 1.25, 1.5, 2, 3, 4, 5, 7, 10) if xmin <= t_ <= xmax]
    if len(ticks) > 9:
        ticks = [t_ for t_ in (0.1, 0.2, 0.5, 1, 2, 5, 10) if xmin <= t_ <= xmax]
    a1.set_xlim(xmin, xmax)
    a1.xaxis.set_major_locator(FixedLocator(ticks))
    a1.xaxis.set_minor_locator(NullLocator())
    a1.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    a1.set_yticks(ypos)
    a1.set_yticklabels([chart_label(nm, 44) for nm in names])
    a1.set_ylim(-0.7, n - 0.3)
    a1.set_xlabel('во сколько раз доля «плохих» выше при событии (1 — нет разницы, 0.5 — в 2 раза ниже;\n'
                  f'линия — 95% ДИ; число справа — оценка; точки — пороги силы: в {RULE_RR_MEDIUM:g} и '
                  f'{RULE_RR_STRONG:g} раза)')
    a1.set_title('Во сколько раз меняется риск при событии', loc='left')
    a1.grid(axis='y', visible=False)
    # справа: IV лучшего признака правила (если признака нет — IV флага)
    iv = res['iv_best_all'].astype(float).where(res['iv_best_all'].notna(), res['iv_flag']).values
    ivf = np.where(np.isfinite(iv), iv, 0.0)
    xm = max(float(np.nanmax(ivf)) * 2.1 if len(ivf) else 0.0, RULE_IV_STRONG * 1.6)
    for thr in (RULE_IV_MEDIUM, RULE_IV_STRONG):
        a2.axvline(thr, color=PAL['muted'], lw=0.9, ls='--', zorder=1)
    a2.barh(ypos, ivf, color=col, height=0.6, zorder=2)
    for y_, v_, s_, vd in zip(ypos, ivf, res['share_event'], res['verdict']):
        if vd in _V_NA:
            continue
        a2.text(v_ + xm * 0.01, y_, f'{v_:.4f}; событие у {s_:.1%} {unit}'
                + (f'; {_V_SHORT[vd]}' if vd in _V_SHORT else ''), va='center', fontsize=7, color=PAL['muted'],
                zorder=3, bbox=dict(facecolor='white', edgecolor='none', pad=0.6))
    a2.set_xlim(0, xm)
    a2.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    a2.set_xlabel(f'IV лучшего признака правила за весь период\n(пунктир — пороги IV: {RULE_IV_MEDIUM:g} — среднее, '
                  f'{RULE_IV_STRONG:g} — сильное влияние)')
    a2.set_title('Вклад в портфель: IV', loc='left')
    a2.grid(axis='y', visible=False)
    hd = [Patch(color=PAL['red']), Patch(color=PAL['blue'])]
    lb = ['повышает риск', 'снижает риск']
    if (effect & (dirs == 0)).any():
        hd.append(Patch(color=PAL['violet']))
        lb.append('влияет, направление зависит от значения')
    hd += [Patch(color=PAL['gray']),
           Line2D([], [], marker='o', ls='none', mfc='white', mec=PAL['muted'], color=PAL['muted'], ms=6)]
    lb += ['нет влияния, не подтверждено, мало данных', 'неустойчиво (полый кружок)']
    fig.legend(handles=hd, labels=lb, loc='lower center', ncol=len(hd), fontsize=8)
    fig.suptitle(title, fontsize=10.5, fontweight='bold', y=1 - 0.08 / h, va='top')
    plt.tight_layout(rect=(0, 0.45 / h, 1, 1 - 0.5 / h))
    fig.subplots_adjust(top=1 - 0.95 / h)                         # без лишнего промежутка под общим заголовком
    plt.show()
    show_chart_legend(list(res.index), names)


def rule_summary_lines(res, model, unit='эпизодов'):
    """Короткий итог простыми словами: какие правила повышают / снижают риск, какие влияют слабо, не влияют, какие
    оценить нельзя; правила, настроенные банком, — отдельно; Джини всех правил ПКБ вместе на тестовом периоде."""
    def nm(r):
        # «Статус „Призывник“»: кавычки внутри названия — „…“
        return '«' + code_name(r).replace('«', '„').replace('»', '“') + '»'

    def lst(sel, fmt):
        return '; '.join(fmt(r, row) for r, row in res.loc[sel].iterrows()) if len(sel) else 'нет'

    def why(x):
        # почему такая сила: риск при событии, IV лучшего признака, как часто бывает событие
        f = (f' по признаку «{code_name(x.best_feature)}»' if isinstance(x.best_feature, str) else '')
        return (f'риск при событии {_times(x.rr)}, IV {_f4(x.iv_best_all if np.isfinite(x.iv_best_all) else x.iv_flag)}'
                f'{f}, событие у {x.share_event:.1%} {unit}')

    bank = [r for r in res.index if r in BANK_RULES]
    v, d = res['verdict'], res['direction']
    nb = ~res.index.isin(bank)
    strong_med = v.isin(['strong', 'medium']) & nb
    up = list(res.index[strong_med & (d > 0)])
    down = list(res.index[strong_med & (d < 0)])
    both = list(res.index[strong_med & (d == 0)])
    weak = list(res.index[v.isin(['weak', 'unstable']) & nb])
    vt = lambda x: x.verdict_text.split(': ')[0]                                      # noqa: E731
    nt = lambda x: f' ({x.verdict_notes})' if x.verdict_notes else ''                 # noqa: E731
    lines = [
        f'Правил с достаточными данными для вывода: {int(v.isin(_V_JUDGED).sum())} из {len(res)} '
        f'(событие правила за {RULE_WINDOW} дн. до {_sd_word(unit)}, кроме событий по нашим договорам).',
        f'Сила влияния — большая из двух оценок: во сколько раз меняется риск у {unit} с событием (от '
        f'{RULE_RR_MEDIUM:g} раза — среднее, от {RULE_RR_STRONG:g} раз — сильное, если это подтверждает 95% ДИ) и IV '
        f'лучшего признака правила — насколько правило разделяет «плохих» и «хороших» во всём портфеле с учётом того, '
        f'как часто бывает событие (от {RULE_IV_MEDIUM:g} — среднее, от {RULE_IV_STRONG:g} — сильное). Редкое событие '
        'может сильно менять риск клиента, но мало влиять на портфель.',
        'Повышают риск (сильное и среднее влияние): ' + lst(up, lambda r, x: f'{nm(r)} — {vt(x)}: {why(x)}{nt(x)}'),
        'Снижают риск (сильное и среднее влияние): ' + lst(down, lambda r, x: f'{nm(r)} — {vt(x)}: {why(x)}{nt(x)}')]
    if both:
        lines.append('Влияют, направление зависит от значения признака: '
                     + lst(both, lambda r, x: f'{nm(r)} — {vt(x)}: {why(x)}{nt(x)}'))
    lines += [
        'Слабое влияние: ' + lst(weak, lambda r, x: f'{nm(r)} ({_DIR_TEXT[x.direction]}, {_times(x.rr)}, IV '
                                                    f'{_f4(x.iv_best_all)}'
                                                    + (', неустойчиво' if x.verdict == 'unstable' else '')
                                                    + (f'; {x.verdict_notes}' if x.verdict_notes else '') + ')'),
        f'Нет влияния (разница незначима, и весь 95% ДИ риска при событии — от {1 / RULE_RR_NEGLIGIBLE:.2f} до '
        f'{RULE_RR_NEGLIGIBLE:g} раза): ' + lst(list(res.index[(v == 'none') & nb]), lambda r, x: nm(r))]
    if ((v == 'negligible') & nb).any():
        lines.append(f'Практически нет влияния (разница не случайна, но меньше чем в {RULE_RR_NEGLIGIBLE:g} раза): '
                     + lst(list(res.index[(v == 'negligible') & nb]), lambda r, x: f'{nm(r)} ({_times(x.rr)})'))
    lines += [
        'Влияние не подтверждено — разница может быть случайной, но ДИ слишком широкий, чтобы сказать «нет влияния»: '
        + lst(list(res.index[(v == 'unproven') & nb]), lambda r, x: f'{nm(r)} — {_rr_ci_text(x.rr, x.rr_lo, x.rr_hi)}'),
        f'Мало данных — оценить нельзя (событие меньше чем у {MIN_EVENT_EPISODES} {unit} или меньше {MIN_EVENT_BADS} '
        '«плохих» среди них — и наблюдаемых, и ожидаемых, если бы событие не влияло): '
        + lst(list(res.index[(v == 'few') & nb]), lambda r, x: nm(r))]
    if bank:
        lines.append('Правила, настроенные банком (сравнивают наш договор с другими кредиторами): '
                     + lst(bank, lambda r, x: f'{nm(r)} — {x.verdict_text}'))
    if ((v == 'own_only') & nb).any():
        lines.append('События только по нашим договорам — не оцениваются (в выводе учитываются только события других '
                     'кредиторов и без номера договора): ' + lst(list(res.index[(v == 'own_only') & nb]), lambda r, x: nm(r)))
    if ((v == 'no_events') & nb).any():
        lines.append('Нет событий у клиентов выборки: ' + lst(list(res.index[(v == 'no_events') & nb]), lambda r, x: nm(r)))
    if (v == 'technical').any():
        lines.append('Техническое событие, не оценивается: ' + lst(list(res.index[v == 'technical']), lambda r, x: nm(r)))
    if np.isfinite(model['gini_test']):
        lo, hi = model['ci']
        lines.append(f'Все правила ПКБ вместе (модель только на правилах ПКБ, по одному лучшему признаку на правило, '
                     f'без переменных договора): Джини на тестовом периоде {model["gini_test"]:.3f} (95% ДИ '
                     f'{lo:.3f} … {hi:.3f}); правил в модели: {len(model["rules"])}; обучение — {_sd_word(unit)} до '
                     f'{pd.Timestamp(model["cut"]):%d.%m.%Y} включительно ({fmt_int(model["n_train"])} {unit}), тест — '
                     f'позже ({fmt_int(model["n_test"])} {unit}).')
    else:
        lines.append('Модель на правилах ПКБ не строилась: мало «плохих» на обучающем или тестовом периоде '
                     'или нет признаков правил.')
    # события без номера договора (заявки, запросы, поиск данных) могут быть вызваны и нашим банком (раздел 7)
    cr = globals().get('CONTRACT_RULES')
    who = [r for r in res.index[v.isin(_V_EFFECT)]
           if cr is not None and r not in cr and r not in BANK_RULES and code_family(r) == 'credit_hunger']
    if who:
        lines.append(f'Оговорка: события без номера договора ({", ".join(nm(r) for r in who)}) могут быть вызваны и нашим '
                     'банком (например, наш запрос кредитного отчёта или заявка клиента к нам) — ПКБ не передаёт, кто '
                     'запросил (раздел 7). Перед решением сверьте с датами наших запросов в ПКБ.')
    return lines


def show_rule_verdicts(res, model, title, unit='эпизодов', oot_note=''):
    """Итог, таблица по правилам, график и «Расшифровка обозначений на графике»."""
    print(f'Найдено правил: {len(res)}')
    n_noname = int(ev['rule'].isna().sum() + ev['rule'].dropna().astype(str).isin(_NO_RULE).sum()) if len(ev) else 0
    if n_noname:
        print(f'Событий без названия правила: {fmt_int(n_noname)} — ни к одному правилу не относятся и не учитываются.')
    print('\nИтог:')
    for s in rule_summary_lines(res, model, unit):
        print('- ' + s)
    if oot_note:
        print(oot_note)
    print(f'\nВывод по каждому правилу ПКБ (событие правила за {RULE_WINDOW} дн. до {_sd_word(unit)}, кроме событий по '
          'нашим договорам; сортировка — от сильного влияния к слабому, внутри — по IV лучшего признака):')
    _display_orig(rule_table_screen(res, unit))
    plot_rule_verdicts(res, title, unit)


def rule_excel_notes(res, model, unit='эпизодов', pop_text=''):
    """Строки листа Excel «Итог по правилам ПКБ»: (над таблицей — что за популяция и итог; под таблицей — пороги вывода,
    сравнение со шкалой IV листа признаков и термины)."""
    top = ([pop_text] if pop_text else []) + ['Итог:'] + rule_summary_lines(res, model, unit)
    bottom = [
        f'Пороги вывода: мало данных — событие меньше чем у {MIN_EVENT_EPISODES} {unit} или меньше {MIN_EVENT_BADS} '
        f'«плохих» среди них (и наблюдаемых, и ожидаемых, если бы событие не влияло); влияние есть, если q-значение < '
        f'{RULE_ALPHA:g} (по флагу события или по лучшему признаку на тесте) и эффект не ничтожен (IV лучшего признака ≥ '
        f'{RULE_IV_NEGLIGIBLE:g} или риск отличается хотя бы в {RULE_RR_NEGLIGIBLE:g} раза); нет влияния — разница '
        f'незначима и весь 95% ДИ риска при событии от {1 / RULE_RR_NEGLIGIBLE:.2f} до {RULE_RR_NEGLIGIBLE:g} раза, иначе '
        f'«влияние не подтверждено»; сила — большая из двух оценок: по IV лучшего признака за весь период (≥ '
        f'{RULE_IV_STRONG:g} — сильное, {RULE_IV_MEDIUM:g}–{RULE_IV_STRONG:g} — среднее) и по риску при событии (в '
        f'{RULE_RR_STRONG:g} раза и больше — сильное, в {RULE_RR_MEDIUM:g} раза и больше — среднее, если это подтверждает '
        f'95% ДИ); неустойчиво — на тесте эффект ослаб не случайно (разница с обучением значима): сменил направление или '
        f'стал меньше {RULE_STAB_KEEP:.2g} эффекта на обучении. Шкала IV здесь мягче общей шкалы листа «IV признаков»: '
        'там 0.02–0.1 — «слабый», 0.1–0.3 — «средний» признак.'] + [''] + _GLOSSARY
    return top, bottom


def format_rule_sheet(xw, sheet, df, header_row):
    """Оформление листа «Итог по правилам ПКБ»: закреплены колонки «Правило» и «Вывод», заголовок жирный с переносом,
    ширина колонок, формат чисел (доли — проценты). Работает с обоими движками pandas.ExcelWriter (openpyxl и
    xlsxwriter). header_row — номер строки заголовка таблицы (с 1)."""
    ws = xw.sheets[sheet]
    spec = []
    for c in df.columns:
        fmt = ('0.0%' if c.startswith(('Доля', 'Справочно: доля')) else
               '0.00' if c.startswith(('Во сколько раз', 'Справочно: во сколько', '95% ДИ')) else
               '0.0000' if c.startswith(('IV', 'p-значение', 'q-значение')) else
               '0.000' if c.startswith(('Падение Джини', 'Джини')) else
               '#,##0' if c.startswith(('Эпизодов', 'Срезов', '«Плохих»', 'Число')) else None)
        spec.append((c, {'Правило': 34, 'Вывод': 40, 'Что означает': 45}.get(c, 14), fmt))
    rows = range(header_row + 1, header_row + 1 + len(df))
    if hasattr(ws, 'cell'):                                      # openpyxl
        from openpyxl.styles import Alignment, Font
        from openpyxl.utils import get_column_letter
        ws.freeze_panes = 'C1'
        for j, (c, wd, fmt) in enumerate(spec, start=1):
            hc = ws.cell(row=header_row, column=j)
            hc.font = Font(bold=True)
            hc.alignment = Alignment(wrap_text=True, vertical='top')
            ws.column_dimensions[get_column_letter(j)].width = wd
            for i in rows:
                if fmt:
                    ws.cell(row=i, column=j).number_format = fmt
                if c == 'Вывод':
                    ws.cell(row=i, column=j).alignment = Alignment(wrap_text=True, vertical='top')
        ws.row_dimensions[header_row].height = 60
    else:                                                        # xlsxwriter
        book = xw.book
        hdr = book.add_format({'bold': True, 'text_wrap': True, 'valign': 'top'})
        ws.freeze_panes(0, 2)
        for j, (c, wd, fmt) in enumerate(spec):
            f = (book.add_format({'text_wrap': True, 'valign': 'top'}) if c == 'Вывод'
                 else book.add_format({'num_format': fmt}) if fmt else None)
            ws.set_column(j, j, wd, f)
            ws.write(header_row - 1, j, c, hdr)
        ws.set_row(header_row - 1, 60)


def rule_population_text(model, pop_name):
    # первая строка листа Excel: что за популяция, что такое «плохой», сколько эпизодов
    if pop_name == 'B':
        return (f'Популяция B: договоры без просрочки на дату среза; «плохой» — договор вышел на просрочку DPD ≥ '
                f'{EWS_MIN_DPD} в течение {EWS_HORIZON} дн.; срезов {fmt_int(model["n_all"])}, доля «плохих» '
                f'{model["bad_rate"]:.1%}. Одна строка — одно правило ПКБ; события по нашим договорам не учитываются.')
    return (f'Популяция A: выход договора в просрочку (DPD {OBS_DPD}); «плохой» — эпизод дошёл до DPD {BAD_DPD}+; '
            f'эпизодов {fmt_int(model["n_all"])}, доля «плохих» {model["bad_rate"]:.1%}. Одна строка — одно правило ПКБ; '
            'события по нашим договорам не учитываются.')


# ------------------------------------------------------------------ расчёт и показ для популяции A
_y_rules_A = pop.set_index('ep_id').loc[F_A.index, 'target'].values.astype(float)
with Timer('Влияние правил ПКБ на риск, популяция A'):
    rules_A, rules_model_A = rule_verdicts(pop, F_A, catalog, _y_rules_A)
# провал загрузки ПКБ (раздел 5) в окнах наблюдения тестового периода: IV на тесте, устойчивость и Джини — по неполным данным
def _rules_oot_tail(where):
    # конец предупреждения о провале загрузки (раздел 5) для вывода по правилам
    return (f'; IV на тестовом периоде, устойчивость, проверка лучшего признака на тесте и Джини правил ПКБ на тесте '
            f'({where}) посчитаны по неполным данным ПКБ: сила влияния в выводе считается по всему периоду, но перед '
            'решением сравните тест с периодом без провала.')


_rules_note_A = _dq_oot_note(rules_model_A['score_dates'], 'эпизодов', 'раздел 10', tail=_rules_oot_tail('раздел 10'))
show_rule_verdicts(rules_A, rules_model_A,
                   f'Популяция A: влияние правил ПКБ на перекат в DPD {BAD_DPD}+ (событие за {RULE_WINDOW} дн. '
                   'до даты скоринга)\nкаждое правило отдельно; события по нашим договорам не учитываются',
                   'эпизодов', _rules_note_A)


## 11. Подробно: лифт правил ПКБ с разбивкой на наши договоры и другие кредиторы
Главный вывод по правилам — в разделе 10. Здесь — прежняя таблица лифта за `IMPACT_WINDOW` = 14 дней, где события по
нашим договорам показаны отдельными строками («наши договоры»); числа те же, что в прежних версиях ноутбука.
Для каждого правила мониторинга ПКБ (Первое кредитное бюро, PKB) — с разбивкой на другие кредиторы / наши договоры там,
где в событии есть код договора, — флаг «хотя бы одно событие за последние `IMPACT_WINDOW` дней» сравнивается с общей
долей «плохих» (bad rate). **Лифт (lift)** = доля «плохих» среди эпизодов с флагом / общая доля «плохих».
Лифт — не то же самое, что «во сколько раз риск выше при событии» в разделе 10: там сравнение с эпизодами БЕЗ события
и окно 30 дней, поэтому то число дальше от 1, чем лифт. Эту таблицу
стоит обсуждать с подразделением взыскания (collection): как часто происходит событие и насколько сильно оно меняет долю
перекатов (roll rate).

In [ ]:
# Целевая переменная (таргет) популяции A в порядке строк матрицы признаков F_A и общая доля «плохих».
y_A = pop.set_index('ep_id').loc[F_A.index, 'target'].values.astype(float)
BR_A = y_A.mean()


# Влияние каждого правила: флаг «было хотя бы одно событие за окно window» против доли «плохих».
# Используются только признаки-счётчики n_<правило>_<window>d; лифт = доля «плохих» с флагом / общая доля «плохих».
# reliable: среди эпизодов с флагом не меньше MIN_BADS_RELIABLE / 2 «плохих» и флаг стоит не у всех эпизодов.
def rule_impact(F, y, catalog, window=IMPACT_WINDOW):
    y = np.asarray(y, float)
    rows = []
    br = y.mean()
    for f, meta in catalog.iterrows():
        if not (f.startswith('n_') and meta['window'] == window):
            continue
        flag = (F[f].values > 0)
        nf = int(flag.sum())
        if nf == 0:
            continue
        bf = float(y[flag].sum())
        n_, b_, g_, woe, iv = bin_stats(flag.astype(int), y, 2)
        rows.append(dict(rule_key=f[2:-(len(str(window)) + 2)], group=meta['group'], family=meta['family'],
                         description=rkey_desc(f[2:-(len(str(window)) + 2)]),
                         share_flagged=nf / len(y), n_flagged=nf, bads_flagged=int(bf),
                         bad_rate_flagged=bf / nf, bad_rate_not=float(y[~flag].mean()) if nf < len(y) else NAN,
                         lift=(bf / nf) / br if br > 0 else NAN, iv_flag=float(iv.sum()),
                         reliable=bf >= MIN_BADS_RELIABLE / 2 and (len(y) - nf) > 0))
    if not rows:
        return pd.DataFrame(columns=['rule_key', 'group', 'family', 'description', 'share_flagged', 'n_flagged',
                                     'bads_flagged', 'bad_rate_flagged', 'bad_rate_not', 'lift', 'iv_flag', 'reliable'])
    return pd.DataFrame(rows).sort_values('iv_flag', ascending=False).reset_index(drop=True)


impact_A = rule_impact(F_A, y_A, catalog)
print(f'Популяция A: влияние правил ПКБ (флаг «было событие за последние {IMPACT_WINDOW} дн.»):')
display(impact_A)


# График лифта по правилам: красный — событие повышает риск, синий — снижает (ключ цветов — в легенде);
# правила, флаг которых стоит менее чем у min_share эпизодов, не показываются.
# Подпись строки — короткое русское название правила (code_names); под графиком — «Расшифровка обозначений на графике»
# (полное описание правила и код в данных) в том же порядке, сверху вниз.
from matplotlib.patches import Patch


def plot_rule_lift(impact, title, min_share=0.001):
    d = impact[impact['share_flagged'] >= min_share].sort_values('lift')
    if d.empty:
        return
    names = code_names(d['rule_key'])
    fig, ax = plt.subplots(figsize=(11, 0.34 * len(d) + 1.5))
    col = np.where(d['lift'] >= 1, PAL['red'], PAL['blue'])
    ax.barh(range(len(d)), d['lift'] - 1, left=1, color=col, height=0.6)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels([chart_label(n, 56) for n in names])  # подпись до 55 символов — в одну строку
    ax.axvline(1, color=PAL['muted'], lw=0.8)
    for i, (l, s, rel) in enumerate(zip(d['lift'], d['share_flagged'], d['reliable'])):
        ax.text(max(l, 1) + 0.03, i, f'{s:.1%} с событием' + ('' if rel else ', мало «плохих» — ненадёжно'), va='center',
                fontsize=7, color=PAL['muted'])
    ax.set_xlabel('лифт = доля «плохих» при наличии события / общая доля «плохих»')
    ax.set_title(title)
    # ключ цветов — под графиком, чтобы не закрывать столбцы и подписи
    fig.legend(handles=[Patch(color=PAL['red']), Patch(color=PAL['blue'])],
               labels=['риск выше среднего (лифт > 1)', 'риск ниже среднего (лифт < 1)'], loc='lower center', ncol=2,
               fontsize=8)
    ax.grid(axis='y', visible=False)
    plt.tight_layout(rect=(0, 0.35 / fig.get_figheight(), 1, 1)); plt.show()
    show_chart_legend(list(d['rule_key'])[::-1], names[::-1])    # верхний столбец графика — первая строка


plot_rule_lift(impact_A, f'Популяция A: перекат в DPD {BAD_DPD}+, событие за последние {IMPACT_WINDOW} дн.')

## 12. Подробно: информационная ценность (IV) каждого признака ПКБ
Признаки ПКБ (Первое кредитное бюро, PKB) на полной популяции A. Переменные договора (раздел 6) — не признаки ПКБ:
в таблицы и графики раздела они не входят (их IV считается, как и раньше, и лежит в Excel на листе «Переменные
договора (справочно)»). Шкала силы IV в таблице — общая: < 0.02 бесполезный, 0.02–0.1 слабый, 0.1–0.3 средний,
0.3–0.5 сильный признак; вывод о влиянии каждого правила — в разделе 10.

In [ ]:
# Матрица признаков популяции A: признаки ПКБ + переменные договора (IV считается для всех, как раньше; показываются
# только признаки ПКБ, переменные договора — в Excel на листе «Переменные договора (справочно)»).
X_A = F_A.join(base_A)
features_A = list(X_A.columns)
with Timer(f'Расчёт IV для {len(features_A)} признаков'):
    iv_A, models_A = compute_iv(X_A, y_A, features_A, catalog)
# колонки, выводимые в таблице IV
cols_show = ['group', 'family', 'window', 'iv', 'strength', 'gini', 'ks', 'n_bins', 'share_nonzero', 'share_missing',
             'bads_outside_main_bin', 'reliable', 'direction', 'description']
# только признаки ПКБ: переменные договора (группа 'baseline') — не признаки ПКБ и в рейтинг не входят
_iv_A_pkb = iv_A[iv_A['group'] != 'baseline']
print('Колонка «Сила признака по IV» — общая шкала для признаков скоринговой модели (< 0.02 бесполезный, 0.02–0.1 '
      'слабый, 0.1–0.3 средний, 0.3–0.5 сильный). Вывод о правилах в разделе 10 использует свою, более мягкую шкалу (по '
      'IV лучшего признака правила: от 0.1 — сильное влияние, 0.02–0.1 — среднее; ещё учитывается, во сколько раз '
      'меняется риск), поэтому один и тот же признак здесь может быть «средним», а в разделе 10 давать «сильное влияние».')
print('Топ-40 признаков ПКБ по IV (переменные договора сюда не входят: они в Excel на листе «Переменные договора '
      '(справочно)»):')
display(_iv_A_pkb[cols_show].head(40))
print('IV признаков ПКБ по группам (число признаков в каждом классе силы IV):')
display(iv_class_order(pd.crosstab(_iv_A_pkb['group'], _iv_A_pkb['strength'])))   # классы от сильных к бесполезным


# Горизонтальная диаграмма IV топ-признаков: цвет столбца = группа признака, у конца столбца — значение IV;
# ось X — по данным (до максимальной IV + 15%), вертикальные линии — пороги силы IV (0.02 слабый, 0.1 средний,
# 0.3 сильный), попадающие в эту ось.
# Подпись строки — короткое русское название признака (code_names); под графиком — «Расшифровка обозначений на графике»
# (полное описание признака и код в данных) в том же порядке, сверху вниз.
def plot_iv(iv, title, top=30):
    d = iv.head(top).iloc[::-1]
    if d.empty:
        return
    names = code_names(d.index)
    xmax = max(float(d['iv'].max()) * 1.15, 0.05)
    fig, ax = plt.subplots(figsize=(11, 0.34 * len(d) + 1.5))
    ax.barh(range(len(d)), d['iv'], color=[GROUP_COLOR.get(g, PAL['gray']) for g in d['group']], height=0.62)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels([chart_label(n, 56) for n in names])  # подпись до 55 символов — в одну строку
    thrs = [thr for thr in (0.02, 0.1, 0.3) if thr < xmax]
    for thr in thrs:
        ax.axvline(thr, color='#c9c8c2', lw=0.8)
    for i, v in enumerate(d['iv']):
        ax.text(v + xmax * 0.005, i, f'{v:.3f}', va='center', fontsize=7, color=PAL['muted'])
    ax.set_xlim(0, xmax)
    groups = [g for g in GROUP_COLOR if g in set(d['group'])]
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOR[g]) for g in groups],
              labels=[RU_VALUES.get(g, g) for g in groups], loc='lower right')
    ax.set_xlabel('Информационная ценность, IV' + (f' (линии — пороги силы IV: {" / ".join(f"{t:g}" for t in thrs)})'
                                                    if thrs else ''))
    ax.set_title(title)
    ax.grid(axis='y', visible=False)
    plt.tight_layout(); plt.show()
    show_chart_legend(list(d.index)[::-1], names[::-1])          # верхний столбец графика — первая строка


pkb_groups = ('pkb_other', 'pkb_bank_rule')
plot_iv(iv_A[iv_A['group'].isin(pkb_groups)], 'Популяция A: топ признаков ПКБ по IV (другие кредиторы и правила, '
        'настроенные банком)')

### Профили WoE (вес свидетельства) самых сильных признаков ПКБ (Первое кредитное бюро, PKB)
Столбцы: доля «плохих» (bad rate) в каждом бине (интервале), ось Y — в процентах; над каждым столбцом — доля
наблюдений в бине («9% набл.»). Горизонтальная линия: общая доля «плохих». Подписи бинов: «0», «1–2», «≥ 7» — число
событий; «≤ 7 дн.» — дни; «78–150 тыс.» — суммы в тенге; «не было события» — в окне наблюдения такого события не было
(признаки «максимум» и «дней с последнего» тогда пусты); «было событие» — единственный бин для всех, у кого событие
было. Точные границы бинов — в таблицах ниже («Бин (интервал)») и в Excel.

In [ ]:
# Сетка графиков: доля «плохих» по бинам для каждого признака (столбец — доля «плохих» в бине, ось Y в процентах;
# число над столбцом — доля наблюдений в бине; горизонтальная линия — общая доля «плохих»; в заголовке — короткое
# русское название признака и IV; подписи бинов — bin_names: «0», «1–2», «≥ 7», «≤ 7 дн.», «не было события»;
# под сеткой — «Расшифровка обозначений на графике» в порядке панелей, по строкам).
import matplotlib.ticker as mtick


def plot_woe_grid(X, y, models, feats, ncols=3, title=''):
    feats = list(feats)
    if not feats:
        return
    names = code_names(feats)
    nrows = int(math.ceil(len(feats) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.8 * nrows + 0.3), squeeze=False)
    br = np.mean(y)
    for k, (ax, f, nm) in enumerate(zip(axes.ravel(), feats, names)):
        t = bin_table(X[f], y, models[f])
        ax.bar(range(len(t)), t['bad_rate'], color=PAL['blue'], width=0.6)
        ax.axhline(br, color=PAL['muted'], lw=0.8)
        for i, s in enumerate(t['share']):
            ax.text(i, t['bad_rate'].iloc[i], f'{s:.0%} набл.', ha='center', va='bottom', fontsize=7, color=PAL['muted'])
        ax.set_xticks(range(len(t)))
        bl = bin_names(f, t['bin'], X[f])
        short = max(len(b) for b in bl) <= 7                       # короткие подписи («0», «1–2», «≥ 7») — без наклона
        ax.set_xticklabels(bl, rotation=0 if short else 30, ha='center' if short else 'right', fontsize=7)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
        ax.set_ylim(0, max(float(t['bad_rate'].max()), br) * 1.15)
        if k % ncols == 0:
            ax.set_ylabel('доля «плохих» в бине', fontsize=7)
        ax.set_title(f'{chart_label(nm, 38)}\nIV {bin_table(X[f], y, models[f])["iv"].sum():.3f}', fontsize=8.5)
        ax.grid(axis='x', visible=False)
    for ax in axes.ravel()[len(feats):]:
        ax.axis('off')
    fig.suptitle(f'{title}\nстолбец — доля «плохих» в бине; число над столбцом — доля наблюдений в бине; '
                 f'линия — средняя доля «плохих» {br:.1%}', fontsize=10.5, fontweight='bold')
    plt.tight_layout(rect=(0, 0, 1, 1 - 0.6 / fig.get_figheight()))    # 0.6 дюйма сверху — под общий заголовок
    plt.show()
    show_chart_legend(feats, names)


# до 12 самых сильных по IV надёжных признаков ПКБ
top_pkb = iv_A[iv_A['group'].isin(pkb_groups) & iv_A['reliable']].head(12).index
plot_woe_grid(X_A, y_A, models_A, top_pkb, title='Доля «плохих» по бинам: самые сильные признаки ПКБ')
print('Таблицы бинов для топ-5 признаков ПКБ (первая колонка — бин, как на графике; «Бин (интервал)» — точные границы):')
_bt = _lab = None                     # без надёжных признаков ПКБ (очень маленькая выборка) цикл ниже пуст
for f in top_pkb[:5]:
    _bt = bin_table(X_A[f], y_A, models_A[f])
    _lab = list(_bt['bin'])
    print(f'\n«{code_name(f)}» — {catalog.loc[f, "description"]} (код в данных: {f})'
          + (f'; бин «{bin_exact(f, "MISSING")}» — в окне наблюдения такого события не было' if 'MISSING' in _lab
             and _bin_is_pkb(f) else ''))
    _bt.insert(0, 'bin_chart', bin_names(f, _lab, X_A[f]))
    _bt['bin'] = [bin_exact(f, b, 'MISSING' in _lab) for b in _lab]
    display(_bt)
del _bt, _lab

## 13. Стабильность признаков ПКБ
* **Понедельные IV и Джини (Gini)** при бинах, зафиксированных на полной выборке; Джини за неделю считается по порядку
  WoE (вес свидетельства) полной выборки, поэтому смена направления зависимости проявляется как низкое или отрицательное
  значение.
* **PSI** (индекс стабильности популяции) распределения по бинам каждой недели относительно первой недели.
* **Полная 30-дневная история**: у 30-дневных признаков эпизодов с датой скоринга в первые недели мониторинга окно
  наблюдения покрывает меньше 30 дней событий; их IV пересчитывается на эпизодах с полной историей.
* **IV в разрезе продуктов**.

In [ ]:
# Понедельная стабильность при бинах, зафиксированных на полной выборке: IV и Джини (Gini) по неделям,
# PSI распределения по бинам относительно первой недели. Признак считается стабильным (stable),
# если минимальный понедельный Джини > 0 и максимальный PSI < 0.25.
def weekly_stability(X, y, week, models, feats):
    y = np.asarray(y, float)
    wk = np.sort(pd.unique(week))
    rows = []
    for f in feats:
        m = models[f]
        codes = apply_bins(X[f], m)
        n_all, b_all, _, woe_all, _ = bin_stats(codes, y, m['n_codes'])
        ref = None
        ivs, ginis, psis = [], [], []
        for w in wk:
            mk = (week == w)
            n, b, g, woe, iv = bin_stats(codes[mk], y[mk], m['n_codes'])
            ne = n > 0
            ivs.append(iv.sum())
            ginis.append(2 * binned_auc(n[ne], b[ne], -woe_all[ne]) - 1)
            if ref is None:
                ref = n
            psis.append(psi(ref, n))
        rows.append(dict(feature=f, iv_min=np.min(ivs), iv_max=np.max(ivs), iv_cv=np.std(ivs) / max(np.mean(ivs), 1e-9),
                         gini_min_week=np.nanmin(ginis), gini_max_week=np.nanmax(ginis), psi_max=np.max(psis),
                         **{f'iv_{pd.Timestamp(w):%d.%m}': v for w, v in zip(wk, ivs)}))
    if not rows:
        return pd.DataFrame(columns=['stable'])
    t = pd.DataFrame(rows).set_index('feature')
    t['stable'] = (t['gini_min_week'] > 0) & (t['psi_max'] < 0.25)
    return t


# признаки для проверки стабильности: IV >= IV_SELECT и надёжные (не более 40)
stab_feats = list(iv_A[(iv_A['iv'] >= IV_SELECT) & iv_A['reliable']].index[:40])
week_A = pop.set_index('ep_id').loc[X_A.index, 'week'].values
stab_A = weekly_stability(X_A, y_A, week_A, models_A, stab_feats)
print('Понедельная стабильность признаков ПКБ (IV, Джини, PSI по неделям скоринга; «Стабилен» — мин. Джини > 0 и PSI < 0.25):')
display(iv_A.loc[[f for f in stab_feats if iv_A.loc[f, 'group'] != 'baseline'], ['group', 'iv']].join(stab_A))   # только ПКБ

# график понедельной IV для 6 сильнейших признаков ПКБ (в легенде — короткое русское название признака; у каждой
# линии свой тип линии и маркер; под датой недели — число эпизодов; неделя, где эпизодов меньше половины медианы
# (обычно последняя, неполная), закрашена, её точки полые, а отрезки к ней бледные: IV там ненадёжна;
# под графиком — «Расшифровка обозначений на графике» в порядке легенды)
top6 = [f for f in iv_A[iv_A['group'].isin(pkb_groups)].index if f in stab_A.index][:6]
if top6:
    wcols = [c for c in stab_A.columns if c.startswith('iv_') and c[3:5].isdigit()]
    names6 = code_names(top6)
    _wk_n = pd.Series(week_A).value_counts()
    _wk_n = {pd.Timestamp(w).strftime('%d.%m'): int(n) for w, n in _wk_n.items()}
    _xs = np.arange(len(wcols))
    _ns = np.array([_wk_n.get(c[3:], 0) for c in wcols])
    _small = _ns < 0.5 * np.median(_ns)
    fig, ax = plt.subplots(figsize=(12, 4.8))
    for _j in np.flatnonzero(_small):
        ax.axvspan(_j - 0.5, _j + 0.5, color=PAL['gray'], alpha=0.12, lw=0)
    for i, (f, nm) in enumerate(zip(top6, names6)):
        _y = stab_A.loc[f, wcols].values.astype(float)
        _ls, _mk = ['-', '--', ':', '-.', '-', '--'][i % 6], 'osD^vP'[i % 6]
        _yf = np.where(_small, np.nan, _y)                         # полные недели — основная линия
        ax.plot(_xs, _yf, color=SERIES[i], ls=_ls, marker=_mk, ms=5, label=chart_label(nm, 40))
        for _j in np.flatnonzero(_small):                          # неполная неделя: бледный отрезок и полая точка
            for _k in (_j - 1, _j + 1):
                if 0 <= _k < len(_xs) and not _small[_k]:
                    ax.plot([_xs[_k], _xs[_j]], [_y[_k], _y[_j]], color=SERIES[i], ls=_ls, alpha=0.35, lw=1.5)
            ax.plot(_xs[_j], _y[_j], color=SERIES[i], marker=_mk, ms=6, mfc='white', ls='none')
    ax.set_xticks(_xs)
    ax.set_xticklabels([f'{c[3:]}\n{fmt_int(n)} эп.' for c, n in zip(wcols, _ns)])
    ax.set_title('Понедельная IV топ-признаков ПКБ (бины зафиксированы на полной выборке)'
                 + ('\nзакрашено: неделя с малым числом эпизодов' + (' (последняя, неполная)' if _small[-1] else '')
                    + ' — IV ненадёжна' if _small.any() else ''))
    ax.set_xlabel('неделя скоринга (начало) и число эпизодов')
    ax.set_ylabel('IV')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1))
    plt.tight_layout(); plt.show()
    show_chart_legend(top6, names6)

# признаки с окном 30 дней: IV на всех эпизодах против IV только на эпизодах с полностью заполненным окном наблюдения
full_hist = pop.set_index('ep_id').loc[X_A.index, 'hist_days'].values >= max(WINDOWS)
f30 = [f for f in iv_A.index if catalog.reindex([f])['window'].iloc[0] == max(WINDOWS)]
if full_hist.sum() > 0 and full_hist.mean() < 1 and f30:
    y_fh = y_A[full_hist]
    iv_fh = compute_iv(X_A.loc[full_hist], y_fh, f30)[0]['iv'] if y_fh.sum() > 0 else pd.Series(dtype=float)
    comp = pd.DataFrame({'iv_all_episodes': iv_A.loc[f30, 'iv'], 'iv_full_30d_history': iv_fh}).sort_values('iv_all_episodes', ascending=False)
    print(f'{max(WINDOWS)}-дневные признаки: у {fmt_int(full_hist.sum())} эпизодов ({full_hist.mean():.0%}) окно наблюдения заполнено полностью')
    display(comp.head(20))

# IV топ-15 признаков ПКБ по продуктам (бины с полной выборки; продукты, где меньше 20 «плохих», пропускаются)
prod_A = base_A.loc[X_A.index, 'bl_product'].values
top_iv_feats = list(iv_A[iv_A['group'].isin(pkb_groups)].head(15).index)
by_prod = {}
for p_ in pd.Series(prod_A).value_counts().index[:6]:
    mk = prod_A == p_
    if y_A[mk].sum() < 20:
        continue
    by_prod[f'Продукт {PRODUCT_NAMES.get(str(p_), p_)} ({fmt_int(mk.sum())} эп., доля «плохих» {y_A[mk].mean():.1%})'] = {
        f: bin_stats(apply_bins(X_A.loc[mk, f], models_A[f]), y_A[mk], models_A[f]['n_codes'])[4].sum() for f in top_iv_feats}
print('IV топ-признаков ПКБ в разрезе продуктов (бины построены на полной выборке):')
display(pd.DataFrame(by_prod))

## 14. Короткий список признаков ПКБ (short-list): фильтр по корреляции
Кандидаты: признаки ПКБ (Первое кредитное бюро, PKB) групп `pkb_other` (другие кредиторы) и `pkb_bank_rule` (правила
банка) с информационной ценностью (IV) ≥ `IV_SELECT`, надёжные (reliable) и стабильные.
Жадный отбор в порядке убывания IV: признак пропускается, если модуль его ранговой корреляции Спирмена (Spearman)
с уже отобранным признаком превышает `CORR_MAX`.

In [ ]:
# Ранговая корреляция Спирмена между признаками. Пропуск означает «события не было», поэтому перед ранжированием
# он заменяется: у признаков days_since_* (дней с последнего события) — на очень большое число, у остальных — на -1.
def spearman(X, feats):
    """Корреляция Спирмена; пропуск означает «события не было»: +inf для days_since_*, -1 для остальных признаков."""
    D = X[feats].astype(float)
    for f in feats:
        D[f] = D[f].fillna(1e9 if f.startswith('days_since') else -1)
    return D.rank().corr()


# Жадный отбор: признаки перебираются в порядке убывания IV; признак добавляется, только если его корреляция
# (по модулю) с каждым уже отобранным признаком меньше corr_max. Отбор останавливается на max_n признаках.
def greedy_select(iv_series, corr, corr_max=CORR_MAX, max_n=MAX_MODEL_FEATURES):
    sel = []
    for f in iv_series.index:
        if all(abs(corr.loc[f, s]) < corr_max for s in sel):
            sel.append(f)
        if len(sel) >= max_n:
            break
    return sel


# кандидаты: признаки ПКБ (другие кредиторы и правила банка) с IV >= IV_SELECT, надёжные (reliable)
cand = iv_A[iv_A['group'].isin(pkb_groups) & (iv_A['iv'] >= IV_SELECT) & iv_A['reliable']]
# остаются только стабильные признаки (признак, не попавший в проверку стабильности, считается стабильным)
cand = cand[[stab_A['stable'].get(f, True) for f in cand.index]]
if len(cand):
    # корреляция и отбор — среди 60 кандидатов с наибольшей IV
    corr_A = spearman(X_A, list(cand.index[:60]))
    shortlist = greedy_select(cand['iv'].iloc[:60], corr_A)
    print(f'{fmt_int(len(cand))} — число кандидатов, {fmt_int(len(shortlist))} — размер короткого списка:')
    display(iv_A.loc[shortlist, ['group', 'family', 'window', 'iv', 'gini', 'share_nonzero', 'description']])
    # тепловая карта корреляций для первых 15 признаков короткого списка: строки — номер и короткое русское название
    # признака, столбцы — те же номера; в клетках — значение корреляции; под графиком — «Расшифровка обозначений на
    # графике» (номер = номер строки)
    sl = shortlist[:15]
    names_sl = code_names(sl)
    _v = corr_A.loc[sl, sl].values
    fig, ax = plt.subplots(figsize=(0.5 * len(sl) + 6.0, 0.42 * len(sl) + 2.4))
    im = ax.imshow(_v, cmap='RdBu_r', vmin=-1, vmax=1)
    for _i in range(len(sl)):
        for _j in range(len(sl)):
            ax.text(_j, _i, f'{_v[_i, _j]:.2f}', ha='center', va='center', fontsize=6,
                    color='white' if abs(_v[_i, _j]) > 0.6 else '#333333')
    ax.set_xticks(range(len(sl)))
    ax.set_xticklabels([str(i) for i in range(1, len(sl) + 1)], rotation=0, fontsize=8)
    ax.set_xlabel('номер признака (как в строках)')
    ax.set_yticks(range(len(sl)))
    ax.set_yticklabels([chart_label(f'{i}. {n}', 60) for i, n in enumerate(names_sl, 1)], fontsize=7)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.7, label='Корреляция Спирмена')
    ax.set_title(f'Корреляция признаков ПКБ из короткого списка (в отбор проходят пары с |ρ| < {CORR_MAX})')
    plt.tight_layout(); plt.show()
    show_chart_legend(sl, names_sl)
else:
    shortlist = []
    print('Ни один признак ПКБ не прошёл фильтры по IV / надёжности / стабильности')

## 15. Дополнительно: добавляют ли правила ПКБ точности к переменным договора
Это не рейтинг признаков и не сравнение ПКБ с переменными договора: ответ по каждому правилу ПКБ — в разделе 10.
Здесь — только проверка, становится ли модель на переменных договора точнее, если добавить к ней признаки ПКБ
(вне времени, out-of-time, OOT):
* разбиение по дате скоринга: эпизоды с более ранними датами (доля `TRAIN_SHARE`) — обучающая выборка (train),
  остальные — тестовая выборка (test), т. е. проверка вне времени;
* бины (интервалы), информационная ценность (IV) и отбор признаков заново подбираются **только на обучающей выборке**;
* **«Только переменные договора»**: логистическая регрессия (logistic regression) на WoE (вес свидетельства) переменных
  договора (раздел 6);
  **«Переменные договора + ПКБ»** = та же модель + отобранные признаки ПКБ (Первое кредитное бюро, PKB) групп «ПКБ: другие
  кредиторы» и «ПКБ: правила, настроенные банком»;
  **«Переменные договора + ПКБ + события по нашим договорам»** = предыдущая модель + события ПКБ по нашим договорам
  (показывает, что добавляет «сырой» поток событий; в основном он дублирует внутренние данные банка);
  (во внутренних кодах таблицы — `A: baseline`, `B: + PKB other lenders`, `C: + PKB own contracts`);
* ΔДжини (Gini) на тестовой выборке с бутстрэп-оценкой 95%-го доверительного интервала (ДИ); ресэмплинг выполняется по клиентам, чтобы
  несколько договоров одного клиента попадали в выборку вместе;
* контрольная проверка на градиентном бустинге (gradient boosting) по всем признакам (без отбора);
* если окна наблюдения тестовой выборки задевают массовый провал загрузки ПКБ (раздел 5), под результатами печатается
  предупреждение: прирост Джини посчитан по неполным данным.

In [ ]:
# Градиентный бустинг на гистограммах; в старых версиях sklearn нужен экспериментальный импорт.
# Если класс недоступен, контрольная проверка на бустинге пропускается.
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:
    try:
        from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
        from sklearn.ensemble import HistGradientBoostingClassifier
    except ImportError:
        HistGradientBoostingClassifier = None


# Джини (Gini) = 2 * AUC - 1 (с необязательными весами наблюдений)
def gini(y, s, w=None):
    return 2 * roc_auc_score(y, s, sample_weight=w) - 1


# Доля всех «плохих», попавших в верхнюю долю q наблюдений с наибольшим скором (capture rate)
def capture(y, s, q):
    o = np.argsort(-np.asarray(s), kind='mergesort')
    k = max(int(np.ceil(q * len(y))), 1)
    return float(np.asarray(y)[o][:k].sum() / max(np.sum(y), 1))


# Бутстрэп разницы Джини (модель s_b минус модель s_a): ресэмплинг кластеров (клиентов по ИИН), чтобы все договоры
# одного клиента попадали в выборку вместе. Возвращает 2.5%, 50% и 97.5% перцентили (границы 95% ДИ и медиану).
def boot_delta(y, s_a, s_b, clusters, n_boot=N_BOOT, seed=SEED):
    codes, uniq = pd.factorize(pd.Series(clusters))
    nc = len(uniq)
    r = np.random.default_rng(seed)
    out = []
    for _ in range(n_boot):
        # вес наблюдения = сколько раз его клиент попал в бутстрэп-выборку
        w = np.bincount(r.integers(0, nc, nc), minlength=nc)[codes].astype(float)
        mk = w > 0
        if len(np.unique(y[mk])) < 2:
            continue
        out.append(gini(y[mk], s_b[mk], w[mk]) - gini(y[mk], s_a[mk], w[mk]))
    return np.percentile(out, [2.5, 50, 97.5]) if out else np.array([NAN] * 3)


# Отбор признаков только на обучающей выборке: IV >= iv_min и надёжность, затем жадный фильтр по корреляции
# (только для числовых признаков) в порядке убывания IV; не более max_n признаков.
# Возвращает (список отобранных признаков, таблица IV на обучающей выборке).
def select_on_train(Xtr, ytr, feats, max_n=MAX_MODEL_FEATURES, iv_min=IV_SELECT):
    if not feats:
        return [], pd.DataFrame()
    t, _ = compute_iv(Xtr, ytr, feats)
    t = t[(t['iv'] >= iv_min) & t['reliable']]
    if t.empty:
        return [], t
    num = [f for f in t.index[:60] if pd.api.types.is_numeric_dtype(Xtr[f])]
    corr = spearman(Xtr, num) if num else None
    sel = []
    for f in t.index[:60]:
        if corr is not None and f in num and any(s in num and abs(corr.loc[f, s]) >= CORR_MAX for s in sel):
            continue
        sel.append(f)
        if len(sel) >= max_n:
            break
    return sel, t


# Бины и WoE, подобранные на обучающей выборке: {признак: (модель бинов, значения WoE по кодам бинов)}
def woe_fit(Xtr, ytr, feats):
    enc = {}
    for f in feats:
        m = fit_bins(Xtr[f], ytr)
        enc[f] = (m, bin_stats(apply_bins(Xtr[f], m), ytr, m['n_codes'])[3])
    return enc


# Перевод признаков в WoE по ранее подобранным бинам (строки — наблюдения, столбцы — признаки)
def woe_apply(X, enc, feats):
    return np.column_stack([enc[f][1][apply_bins(X[f], enc[f][0])] for f in feats]) if feats else np.zeros((len(X), 0))


# Оценка прироста качества: разбиение вне времени, модели A / B / C (логистическая регрессия на WoE),
# Джини и доля выявленных «плохих» в топ-10/20/30% на тестовой выборке, бутстрэп-ДИ для ΔДжини,
# контрольная проверка на градиентном бустинге и график кумулятивной доли выявленных «плохих» (gains).
# clusters — ИИН клиентов (для бутстрэпа по клиентам), label — подпись популяции в выводе и на графике.
def incremental_value(X, y, score_date, clusters, base_feats, pkb_feats, own_feats, label):
    y = np.asarray(y, float)
    # граница разбиения — квантиль TRAIN_SHARE по дате скоринга: раньше — обучение, позже — тест
    cut = pd.Series(score_date).quantile(TRAIN_SHARE)
    tr = (pd.Series(score_date) <= cut).values
    te = ~tr
    if y[te].sum() < 10 or y[tr].sum() < 10:
        print('Недостаточно «плохих» для проверки вне времени (out-of-time)'); return None
    Xtr, Xte, ytr, yte = X.loc[tr], X.loc[te], y[tr], y[te]
    print(f'{label}: обучающая выборка {fmt_int(tr.sum())} (доля «плохих» {ytr.mean():.2%}, до {pd.Timestamp(cut):%d.%m.%Y}), '
          f'тестовая {fmt_int(te.sum())} (доля «плохих» {yte.mean():.2%})')
    # отбор на обучающей выборке: переменные договора (мягкий порог IV, до 30), ПКБ по другим кредиторам и правилам банка,
    # ПКБ по нашим договорам (до 5)
    sel_base, _ = select_on_train(Xtr, ytr, base_feats, max_n=30, iv_min=0.005)
    sel_pkb, iv_tr = select_on_train(Xtr, ytr, pkb_feats)
    sel_own, _ = select_on_train(Xtr, ytr, own_feats, max_n=5)
    # наборы признаков моделей; названия моделей — ключи таблицы результатов (используются и в разделе 18), не переименовывать
    sets = {'A: baseline': sel_base, 'B: + PKB other lenders': sel_base + sel_pkb,
            'C: + PKB own contracts': sel_base + sel_pkb + sel_own}
    # бины и WoE подбираются один раз для всех признаков самой широкой модели C
    enc = woe_fit(Xtr, ytr, sorted(set(sets['C: + PKB own contracts'])))
    res, scores, coefs = [], {}, None
    # для каждой модели: логистическая регрессия на WoE, Джини на обучающей и тестовой выборке,
    # доля выявленных «плохих» в топ-10/20/30% самых рискованных эпизодов теста
    for name, fs in sets.items():
        if not fs:
            continue
        lr = LogisticRegression(C=1.0, max_iter=2000)
        lr.fit(woe_apply(Xtr, enc, fs), ytr)
        p_tr = lr.predict_proba(woe_apply(Xtr, enc, fs))[:, 1]
        p_te = lr.predict_proba(woe_apply(Xte, enc, fs))[:, 1]
        scores[name] = p_te
        res.append({'model': name, 'n_features': len(fs), 'gini_train': gini(ytr, p_tr), 'gini_test': gini(yte, p_te),
                    'capture_top10%': capture(yte, p_te, 0.1), 'capture_top20%': capture(yte, p_te, 0.2),
                    'capture_top30%': capture(yte, p_te, 0.3)})
        if name.startswith('B'):
            # коэффициенты модели B с группой признака и проверкой знака
            coefs = pd.DataFrame({'feature': fs, 'coef': lr.coef_[0]}).set_index('feature')
            coefs['group'] = ['baseline' if f.startswith('bl_') else catalog.loc[f, 'group'] for f in fs]
            coefs['sign_ok'] = coefs['coef'] < 0          # WoE > 0 означает меньший риск, поэтому коэффициенты должны быть отрицательными
    if not res:
        print(f'{label}: на обучающей выборке не отобрано ни одного признака')
        return None
    res = pd.DataFrame(res).set_index('model')
    # ΔДжини моделей B и C относительно модели A на тесте и её бутстрэп 95% ДИ (кластеры — клиенты тестовой выборки)
    cl_te = np.asarray(clusters)[te]
    for name in (res.index[1:] if 'A: baseline' in scores else []):
        lo, mid, hi = boot_delta(yte, scores['A: baseline'], scores[name], cl_te)
        res.loc[name, 'delta_gini_vs_A'] = res.loc[name, 'gini_test'] - res.loc['A: baseline', 'gini_test']
        res.loc[name, 'delta_CI95'] = f'[{lo:+.4f}, {hi:+.4f}]'
    # контрольная проверка на градиентном бустинге по всем признакам без отбора (модели A и B)
    if RUN_HGB and HistGradientBoostingClassifier is not None:
        # матрица для бустинга: числовые признаки как есть, категориальные — в WoE
        def hgb_matrix(Xs, fs):
            cat = [f for f in fs if not pd.api.types.is_numeric_dtype(X[f])]
            num = [f for f in fs if f not in cat]
            return np.column_stack([Xs[num].astype(float).values, woe_apply(Xs, enc_cat, cat)]) if cat else Xs[num].astype(float).values
        cat_feats = [f for f in base_feats if not pd.api.types.is_numeric_dtype(X[f])]
        enc_cat = woe_fit(Xtr, ytr, cat_feats)
        hg = {}
        for name, fs in (('A: baseline', base_feats), ('B: + PKB other lenders', base_feats + pkb_feats)):
            m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                               min_samples_leaf=max(50, int(0.002 * tr.sum())),
                                               l2_regularization=1.0, random_state=SEED)
            m.fit(hgb_matrix(Xtr, fs), ytr)
            hg[name] = m.predict_proba(hgb_matrix(Xte, fs))[:, 1]
            res.loc[name, 'gini_test_boosting'] = gini(yte, hg[name])
        lo, mid, hi = boot_delta(yte, hg['A: baseline'], hg['B: + PKB other lenders'], cl_te) if pkb_feats else (NAN,) * 3
        res.loc['B: + PKB other lenders', 'delta_boosting_CI95'] = f'[{lo:+.4f}, {hi:+.4f}]'
    print(f'{label}: модели на тестовой выборке вне времени (OOT)')
    display(res)
    if coefs is not None:
        print('Коэффициенты модели «Переменные договора + ПКБ» (в шкале WoE; ожидаемый знак — отрицательный):')
        display(coefs.join(iv_tr[['iv']], how='left') if 'iv' in iv_tr.columns else coefs)
    # кривая доли выявленных «плохих» (gains) моделей A и B на тестовой выборке; диагональ — случайный порядок
    if len(scores) >= 2:
        fig, ax = plt.subplots(figsize=(6.4, 4.6))
        for (name, s), col in zip(list(scores.items())[:2], (PAL['gray'], PAL['blue'])):
            o = np.argsort(-s, kind='mergesort')
            cum = np.cumsum(yte[o]) / yte.sum()
            ax.plot(np.arange(1, len(o) + 1) / len(o), cum, color=col,
                    label=chart_label(f'{RU_VALUES.get(name, name)}, Джини {gini(yte, s):.3f}', 44))
        ax.plot([0, 1], [0, 1], color='#c9c8c2', lw=0.8)
        # единица наблюдения: эпизоды просрочки (популяция A) или срезы договоров без просрочки (популяция B)
        ax.set_xlabel(('доля срезов договоров' if label.endswith('B') else 'доля эпизодов') + ', начиная с самых рискованных')
        ax.set_ylabel('доля выявленных «плохих»')
        ax.set_title(f'{label}: доля выявленных «плохих» на выборке вне времени (OOT)')
        ax.legend(loc='lower right', fontsize=7.5)
        plt.tight_layout(); plt.show()
    return res, sets


# Наборы признаков популяции A: переменные договора, ПКБ (другие кредиторы и правила банка), ПКБ по нашим договорам
base_feats_A = list(base_A.columns)
pkb_feats_A = [f for f in F_A.columns if catalog.loc[f, 'group'] in pkb_groups]
own_feats_A = [f for f in F_A.columns if catalog.loc[f, 'group'] == 'pkb_own']
# дата скоринга и ИИН клиента для каждого эпизода (ИИН — кластер для бутстрэпа)
sd_A = pop.set_index('ep_id').loc[X_A.index, 'score_date'].values
iin_A = pop.set_index('ep_id').loc[X_A.index, 'iin'].values
with Timer('Прирост качества, популяция A'):
    inc_A = incremental_value(X_A, y_A, sd_A, iin_A, base_feats_A, pkb_feats_A, own_feats_A, 'Популяция A')
# провал загрузки ПКБ (раздел 5) в окнах наблюдения тестовой выборки: прирост Джини выше посчитан по неполным данным
_oot_note_A = _dq_oot_note(sd_A, 'эпизодов', 'раздел 15') if inc_A is not None else ''
if _oot_note_A:
    print(_oot_note_A)

## 16. Дополнительные целевые переменные (таргеты)
Информационная ценность (IV) тех же признаков ПКБ относительно других определений исхода (бины подбираются заново для каждой
целевой переменной). Дополнительные целевые переменные задаются в `EXTRA_TARGETS` (имя -> порог дней просрочки (DPD)),
например `nocure_16` = 1, если эпизод дошёл до DPD ≥ 16 (не самоизлечился раньше).

In [ ]:
# IV признаков популяции A для каждой дополнительной целевой переменной из EXTRA_TARGETS.
# Учитываются только эпизоды, исход которых уже наблюдаем (непустое значение); нужно не меньше 20 «плохих».
sec = {}
for name in EXTRA_TARGETS:
    yy = pop.set_index('ep_id').loc[X_A.index, name].values
    mk = ~np.isnan(yy)
    if mk.sum() == 0 or np.nansum(yy) < 20:
        print(f'Дополнительный таргет DPD ≥ {EXTRA_TARGETS[name]}: пока недостаточно эпизодов с наблюдаемым исходом')
        continue
    t, _ = compute_iv(X_A.loc[mk], yy[mk], features_A)
    sec[f'iv_{name}'] = t['iv']
    print(f'Дополнительный таргет DPD ≥ {EXTRA_TARGETS[name]}: {fmt_int(mk.sum())} эпизодов, доля «плохих» {np.nanmean(yy[mk]):.2%}')
# сводная таблица: IV по основной целевой переменной (iv_roll<BAD_DPD>) и по дополнительным; показаны признаки ПКБ
if sec:
    iv_targets = iv_A[['group', 'iv']].rename(columns={'iv': f'iv_roll{BAD_DPD}'}).join(pd.DataFrame(sec))
    display(iv_targets[iv_targets['group'].isin(pkb_groups)].head(25))

## 17. Часть B: раннее предупреждение (early warning, EWS) для договоров без просрочки
Те же признаки ПКБ (Первое кредитное бюро, PKB) для договоров, которые на дату среза не находятся в просрочке.
Целевая переменная (таргет): выход на просрочку в течение `EWS_HORIZON` дней. Раздел показывает, могут ли события
мониторинга ПКБ служить поводом для контакта с клиентом ещё до возникновения просрочки. В таблицах и на графиках —
только признаки ПКБ, кроме дополнительной проверки прироста Джини (как в разделе 15: модель на переменных договора
с признаками ПКБ и без них; это не рейтинг признаков ПКБ). Вывод по каждому правилу для популяции B — в конце раздела.

In [ ]:
# Часть B выполняется, только если популяция B построена и в ней не меньше 20 «плохих».
iv_B, impact_B, inc_B, _oot_note_B = None, None, None, ''
if pop_B is not None and len(pop_B) and pop_B['target'].sum() >= 20:
    with Timer('Признаки ПКБ, популяция B'):
        F_B, catalog_B = build_pkb_features(pop_B, ev)
    X_B = F_B.join(base_B)
    y_B = pop_B.set_index('ep_id').loc[X_B.index, 'target'].values.astype(float)
    print(f'Популяция B: {fmt_int(len(y_B))} срезов, доля «плохих» {y_B.mean():.2%}')
    # влияние правил (лифт) и IV признаков — так же, как для популяции A
    impact_B = rule_impact(F_B, y_B, catalog_B)
    print(f'Популяция B: влияние правил ПКБ (флаг «было событие за последние {IMPACT_WINDOW} дн.»), топ-25:')
    display(impact_B.head(25))
    plot_rule_lift(impact_B, f'Популяция B: выход на просрочку DPD ≥ {EWS_MIN_DPD} в течение {EWS_HORIZON} дн., событие за последние {IMPACT_WINDOW} дн.')
    iv_B, models_B = compute_iv(X_B, y_B, list(X_B.columns), catalog_B)
    print('Популяция B: топ-30 признаков ПКБ по IV (переменные договора сюда не входят; на графике ниже — признаки ПКБ '
          'других кредиторов и правил, настроенных банком):')
    display(iv_B[iv_B['group'] != 'baseline'][cols_show].head(30))
    plot_iv(iv_B[iv_B['group'].isin(pkb_groups)], 'Популяция B: топ признаков ПКБ по IV')
    # каталог признаков временно дополняется признаками популяции B (он нужен внутри incremental_value),
    # после расчёта восстанавливается исходный каталог
    catalog_main = catalog
    catalog = pd.concat([catalog, catalog_B[~catalog_B.index.isin(catalog.index)]])
    print('Дополнительно (как в разделе 15, это не рейтинг признаков ПКБ): добавляют ли правила ПКБ точности к '
          'переменным договора, популяция B')
    with Timer('Прирост качества, популяция B'):
        inc_B = incremental_value(
            X_B, y_B, pop_B.set_index('ep_id').loc[X_B.index, 'score_date'].values,
            pop_B.set_index('ep_id').loc[X_B.index, 'iin'].values, list(base_B.columns),
            [f for f in F_B.columns if catalog_B.loc[f, 'group'] in pkb_groups],
            [f for f in F_B.columns if catalog_B.loc[f, 'group'] == 'pkb_own'], 'Популяция B')
    catalog = catalog_main
    # провал загрузки ПКБ (раздел 5) в окнах наблюдения тестовой выборки популяции B
    _oot_note_B = (_dq_oot_note(pop_B.set_index('ep_id').loc[X_B.index, 'score_date'].values, 'срезов', 'раздел 17')
                   if inc_B is not None else '')
    if _oot_note_B:
        print(_oot_note_B)
else:
    print('Часть B пропущена')

### Влияние правил ПКБ на риск: вывод по каждому правилу, популяция B
То же, что в разделе 10, для договоров без просрочки: «плохой» — договор вышел на просрочку в течение `EWS_HORIZON` дней
после даты среза. Те же события (кроме событий по нашим договорам), те же пороги вывода, то же разбиение на обучающий
и тестовый периоды по дате среза. Единица наблюдения — срез договора, поэтому пороги «мало данных» считаются в срезах,
а в таблице и на листе Excel вместо «эпизодов» — «срезов».
В Excel — лист «Итог по правилам ПКБ, B».

In [ ]:
# Вывод по каждому правилу ПКБ для популяции B (функции — в разделе 10); только если часть B выполнена выше
rules_B, rules_model_B, _rules_note_B = None, None, ''
if iv_B is not None:
    _y_rules_B = pop_B.set_index('ep_id').loc[F_B.index, 'target'].values.astype(float)
    with Timer('Влияние правил ПКБ на риск, популяция B'):
        rules_B, rules_model_B = rule_verdicts(pop_B, F_B, catalog_B, _y_rules_B, unit='срезов')
    _rules_note_B = _dq_oot_note(rules_model_B['score_dates'], 'срезов', 'раздел 17', tail=_rules_oot_tail('раздел 17'))
    show_rule_verdicts(rules_B, rules_model_B,
                       f'Популяция B: влияние правил ПКБ на выход на просрочку DPD ≥ {EWS_MIN_DPD} в течение '
                       f'{EWS_HORIZON} дн. (событие за {RULE_WINDOW} дн. до даты среза)\nкаждое правило отдельно; '
                       'события по нашим договорам не учитываются', 'срезов', _rules_note_B)
else:
    print('Часть B не выполнялась: вывод по правилам для популяции B не строится')

## 18. Выгрузка результатов и резюме
Результаты записываются в папку `OUTPUT_DIR` (в Excel, если установлен `openpyxl`, иначе — в CSV-файлы).
Резюме ниже формируется автоматически из рассчитанных выше чисел.

Листы выгрузки (в скобках — имя CSV-файла, если `openpyxl` не установлен):
* **«Итог по правилам ПКБ»** (`rule_summary_A`) — первый лист: вывод по каждому правилу ПКБ (раздел 10); над таблицей —
  что за популяция и итог простыми словами, под таблицей — пороги вывода и термины; колонки «Правило» и «Вывод»
  закреплены слева (без `openpyxl` итог и пороги — в файле `rule_summary_A_notes.txt`); «Итог по правилам ПКБ, B» (`rule_summary_B`) — то же для популяции B, если она построена;
* «IV признаков, популяция A» (`iv_population_A`) — информационная ценность (IV) признаков ПКБ, популяция A;
* «Влияние правил, популяция A» (`rule_impact_A`) — влияние (лифт) правил мониторинга ПКБ (Первое кредитное бюро, PKB);
* «Стабильность, популяция A» (`stability_A`) — понедельная стабильность признаков ПКБ;
* «Бины признаков, популяция A» (`bins_A`) — таблицы бинов (интервалов) признаков ПКБ из 80 признаков с наибольшей IV;
* «Когорты, популяция A» (`cohorts_A`) — недельные когорты популяции A;
* «Каталог признаков» (`feature_catalog`) — каталог признаков с описаниями;
* «Сверка с нашими договорами» (`own_contract_matching`) — сопоставление событий ПКБ с нашими договорами по каждому правилу;
* «Прирост качества, популяция A / B» (`incremental_A`, `incremental_B`) — прирост качества (инкрементальный эффект)
  моделей, если он рассчитан;
* «IV по таргетам, популяция A» (`iv_by_target_A`) — IV признаков ПКБ по дополнительным целевым переменным;
* «IV признаков, популяция B», «Влияние правил, популяция B» (`iv_population_B`, `rule_impact_B`) — то же для
  популяции B (часть B, раннее предупреждение, EWS);
* **«Переменные договора (справочно)»** (`contract_vars_<лист>`) — последний лист: строки переменных договора (раздел 6),
  убранные с листов IV, стабильности, бинов и IV по таргетам; числа те же, строки только перенесены (у каждого блока —
  заголовок с названием листа, откуда он взят).

In [ ]:
# Таблицы бинов нескольких признаков, сложенные одна под другой, с колонкой 'feature' (имя признака)
def bins_long(X, y, models, feats):
    out = []
    for f in feats:
        t = bin_table(X[f], y, models[f])
        t.insert(0, 'feature', f)
        out.append(t)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


# Листы выгрузки: внутреннее имя (оно же имя CSV-файла) -> таблица
sheets = {'iv_population_A': iv_A, 'rule_impact_A': impact_A, 'stability_A': stab_A,
          'bins_A': bins_long(X_A, y_A, models_A, list(iv_A.head(80).index)),
          'cohorts_A': cohorts, 'feature_catalog': catalog, 'own_contract_matching': own_diag}
if inc_A is not None:
    sheets['incremental_A'] = inc_A[0]
if sec:
    sheets['iv_by_target_A'] = iv_targets
if iv_B is not None:
    sheets.update({'iv_population_B': iv_B, 'rule_impact_B': impact_B})
    if inc_B is not None:
        sheets['incremental_B'] = inc_B[0]
# Переменные договора (раздел 6, группа 'baseline') — не признаки ПКБ: их строки убираются с листов IV, стабильности, бинов
# и IV по таргетам и без изменений переносятся на последний лист «Переменные договора (справочно)» (по блоку на лист)
_contract_codes = set(base_A.columns) | (set(base_B.columns) if base_B is not None else set())


def _contract_rows(df):
    # строки переменных договора: по группе признака, иначе по коду признака (колонка 'feature' или индекс)
    if 'group' in df.columns:
        return (df['group'] == 'baseline').values
    if 'feature' in df.columns:
        return df['feature'].isin(_contract_codes).values
    return pd.Index(df.index).isin(_contract_codes)


contract_parts = {}
for _nm in ('iv_population_A', 'stability_A', 'bins_A', 'iv_by_target_A', 'iv_population_B'):
    if isinstance(sheets.get(_nm), pd.DataFrame) and len(sheets[_nm]):
        _m = _contract_rows(sheets[_nm])
        if _m.any():
            contract_parts[_nm] = sheets[_nm][_m]
            sheets[_nm] = sheets[_nm][~_m]
# первые листы — вывод по каждому правилу ПКБ (раздел 10 и часть B): таблицы уже на русском; над таблицей — что за
# популяция и итог, под таблицей — пороги вывода и термины
_first, _notes = {}, {}
if globals().get('rules_A') is not None:
    _first['rule_summary_A'] = rule_table_excel(rules_A, 'эпизодов')
    _notes['rule_summary_A'] = rule_excel_notes(rules_A, rules_model_A, 'эпизодов',
                                                rule_population_text(rules_model_A, 'A'))
if globals().get('rules_B') is not None:
    _first['rule_summary_B'] = rule_table_excel(rules_B, 'срезов')
    _notes['rule_summary_B'] = rule_excel_notes(rules_B, rules_model_B, 'срезов',
                                                rule_population_text(rules_model_B, 'B'))
sheets = {**_first, **sheets}
# Русские названия листов Excel (не длиннее 31 символа, без символов []:*?/\); имена файлов не меняются
RU_SHEETS = {'iv_population_A': 'IV признаков, популяция A', 'rule_impact_A': 'Влияние правил, популяция A',
             'stability_A': 'Стабильность, популяция A', 'bins_A': 'Бины признаков, популяция A',
             'cohorts_A': 'Когорты, популяция A', 'feature_catalog': 'Каталог признаков',
             'own_contract_matching': 'Сверка с нашими договорами', 'incremental_A': 'Прирост качества, популяция A',
             'iv_by_target_A': 'IV по таргетам, популяция A', 'iv_population_B': 'IV признаков, популяция B',
             'rule_impact_B': 'Влияние правил, популяция B', 'incremental_B': 'Прирост качества, популяция B',
             'rule_summary_A': 'Итог по правилам ПКБ', 'rule_summary_B': 'Итог по правилам ПКБ, B',
             'contract_vars': 'Переменные договора (справочно)'}
_CONTRACT_INTRO = ('Справочно: переменные договора (раздел 6) — данные банка, а не признаки ПКБ. Их строки перенесены сюда '
                   'без изменений с листов, названных в заголовках блоков (в блоке бинов номер строки слева — как на '
                   'исходном листе); с признаками ПКБ они не сравниваются и используются только в разделе 15.')
try:
    import openpyxl  # noqa: F401
    path = os.path.join(OUTPUT_DIR, 'pkb_monitoring_iv_results.xlsx')
    with pd.ExcelWriter(path) as xw:
        for name, df in sheets.items():
            if name in _first:                                  # вывод по правилам: таблица уже на русском
                _top, _bottom = _notes[name]
                pd.DataFrame({'': _top}).to_excel(xw, sheet_name=RU_SHEETS[name], index=False, header=False)
                df.to_excel(xw, sheet_name=RU_SHEETS[name], index=False, startrow=len(_top) + 1)
                pd.DataFrame({'': _bottom}).to_excel(xw, sheet_name=RU_SHEETS[name], startrow=len(_top) + len(df) + 3,
                                                     index=False, header=False)
                format_rule_sheet(xw, RU_SHEETS[name], df, len(_top) + 2)
                continue
            d = df.copy()
            # колонки с датами записываются как текст
            for c in d.columns:
                if pd.api.types.is_datetime64_any_dtype(d[c]):
                    d[c] = d[c].astype(str)
            # русские подписи колонок и значений (ru_labels), русское имя листа (не длиннее 31 символа)
            ru_labels(d).to_excel(xw, sheet_name=RU_SHEETS.get(name, name)[:31])
        if contract_parts:                                      # последний лист: переменные договора, блок на каждый лист
            _sh, _row = RU_SHEETS['contract_vars'], 0
            pd.DataFrame({'': [_CONTRACT_INTRO]}).to_excel(xw, sheet_name=_sh, index=False, header=False)
            _row = 2
            for name, part in contract_parts.items():
                pd.DataFrame({'': [f'С листа «{RU_SHEETS[name]}»: переменные договора, {fmt_int(len(part))} '
                                   f'{_plural_ru(len(part), ("строка", "строки", "строк"))}']}).to_excel(
                    xw, sheet_name=_sh, startrow=_row, index=False, header=False)
                ru_labels(part).to_excel(xw, sheet_name=_sh, startrow=_row + 1)
                _row += len(part) + 5
    print('Сохранено:', path)
except ImportError:
    # без openpyxl каждая таблица сохраняется в отдельный CSV-файл (с русскими подписями, имя файла — внутреннее)
    for name, df in sheets.items():
        (df if name in _first else ru_labels(df)).to_csv(os.path.join(OUTPUT_DIR, f'{name}.csv'), encoding='utf-8-sig',
                                                         index=name not in _first)
        if name in _first:                                      # итог, пороги и термины — рядом, в текстовом файле
            open(os.path.join(OUTPUT_DIR, f'{name}_notes.txt'), 'w', encoding='utf-8').write(
                '\n'.join(_notes[name][0] + [''] + _notes[name][1]) + '\n')
    for name, part in contract_parts.items():
        ru_labels(part).to_csv(os.path.join(OUTPUT_DIR, f'contract_vars_{name}.csv'), encoding='utf-8-sig')
    print('CSV-файлы сохранены в папку', OUTPUT_DIR)

# Текстовое резюме: каждая строка формируется из чисел, рассчитанных выше
lines = [f'ДАННЫЕ: {"СИНТЕТИЧЕСКИЕ (демо) — цифры ничего не говорят о реальном портфеле" if DEMO_MODE else "реальные (production)"}',
         f'Использованный период мониторинга ПКБ: {HES_START_TS.date()} .. {HES_END_TS.date()}; данные о днях просрочки (DPD) до {DATA_END_TS.date()}.',
         f'Популяция A: {fmt_int(len(pop))} выходов в софт-коллекшн с датами скоринга {pop["score_date"].min().date()} .. '
         f'{pop["score_date"].max().date()}; доля перекатов (roll rate) в DPD {BAD_DPD}+ = {BR_A:.2%} («плохих»: {fmt_int(y_A.sum())}).',
         f'{(F_A["days_since_any_oth"].notna()).mean():.0%} эпизодов имели хотя бы одно событие ПКБ у других кредиторов до даты скоринга.']
# главный результат: вывод по каждому правилу ПКБ (раздел 10)
if globals().get('rules_A') is not None:
    lines.append(f'ГЛАВНОЕ — влияние правил ПКБ на риск (раздел 10; событие за {RULE_WINDOW} дн. до даты скоринга, кроме '
                 'событий по нашим договорам):' + ''.join('\n    ' + s for s in rule_summary_lines(rules_A, rules_model_A)))
# только признаки ПКБ (другие кредиторы и правила банка)
pk = iv_A[iv_A['group'].isin(pkb_groups)]
lines.append(f'Признаков ПКБ: {len(pk)}; IV >= 0.1: {(pk["iv"] >= 0.1).sum()}, 0.02-0.1: {((pk["iv"] >= 0.02) & (pk["iv"] < 0.1)).sum()}, '
             f'ниже 0.02: {(pk["iv"] < 0.02).sum()}.')
# у каждого признака и правила: короткое русское название (code_name), полное описание (feature_desc, rkey_desc),
# в конце — код в данных
lines.append('Самые сильные признаки ПКБ:' + ''.join(
    f'\n    {code_name(f)} — {feature_desc(f)} (IV {r.iv:.3f}; по общей шкале IV признаков — {RU_VALUES.get(r.strength, r.strength)}; код в данных: {f})'
    for f, r in pk.head(5).iterrows()))
if inc_A is not None:
    r = inc_A[0]
    b = r.loc['B: + PKB other lenders'] if 'B: + PKB other lenders' in r.index else None
    if b is not None:
        lines.append(f'Дополнительно (раздел 15, не рейтинг): Джини вне времени (OOT) модели только на переменных договора '
                     f'{r.loc["A: baseline", "gini_test"]:.3f} -> с признаками ПКБ {b["gini_test"]:.3f} '
                     f'(прирост {b["delta_gini_vs_A"]:+.3f}, 95% ДИ {b["delta_CI95"]}).')
if globals().get('_rules_note_A'):
    lines.append('Популяция A, вывод по правилам. ' + _rules_note_A)
# предупреждения проверок потока событий (раздел 5): провал загрузки в окнах теста OOT, резкие изменения потока
if globals().get('_oot_note_A'):
    lines.append('Популяция A. ' + _oot_note_A)
if globals().get('_oot_note_B'):
    lines.append('Популяция B. ' + _oot_note_B)
_reg = globals().get('hes_regime')
if isinstance(_reg, pd.DataFrame) and len(_reg) and _reg['flag'].any():
    lines.append('РЕЗКОЕ ИЗМЕНЕНИЕ потока событий после периода выборки (раздел 5): ' + '; '.join(
        f'«{code_name(r.rule)}» — {r.verdict}' for r in _reg[_reg['flag']].itertuples())
        + '. Бины и WoE этих правил подобраны на старом потоке.')
# подозрительно высокая IV (>= 0.5) — признак нужно проверить на утечку данных
susp = pk[pk['iv'] >= 0.5]
if len(susp):
    lines.append('Проверить на утечку данных (leakage), IV >= 0.5:' + ''.join(
        f'\n    {code_name(f)} — {feature_desc(f)} (код в данных: {f})' for f in susp.index[:10]))
if iv_B is not None:
    pb = iv_B[iv_B['group'].isin(pkb_groups)]
    lines.append(f'Часть B (раннее предупреждение, EWS): лучший признак ПКБ «{code_name(pb.index[0])}» — {feature_desc(pb.index[0])} '
                 f'(IV {pb["iv"].iloc[0]:.3f}; код в данных: {pb.index[0]}).')
if globals().get('rules_B') is not None:
    lines.append('Часть B: влияние правил ПКБ на выход на просрочку (раздел 17):'
                 + ''.join('\n    ' + s for s in rule_summary_lines(rules_B, rules_model_B, 'срезов')))
lines.append(f'Оговорка: история мониторинга короткая ({(HES_END_TS - HES_START_TS).days} дн.), поэтому у {max(WINDOWS)}-дневных '
             'признаков в первые недели окно наблюдения усечено, а стабильность проверяется лишь на нескольких недельных когортах.')
print('\n'.join('- ' + s for s in lines))